**Active M5E workflow — 31 October 2026:** the only prepared pilot is two fresh ConvNeXt-Base arms (control shuffle and capped disease tickets), each 2 warm-up +15 fine-tuning epochs. Review the appended M5E section and set only RUN_M5E_PILOT=True when the owner is ready. Keep historical training flags False. The older backbone-comparison configuration below remains an engineering reference; its unrestricted sequence launch is blocked. No pilot was executed during preparation.

# WP2 high-performance preparation — MSI Katana 17
## 1. Objectives and manual ownership
Compare exactly two final candidates: **EfficientNet-B3** and **ConvNeXt-Base**, using verified ImageNet-1K V1 initialization. The older ResNet18/B0/Tiny modules remain engineering references, not planned full experiments.
Assignment 01 EfficientNet-B3/ConvNeXt-Base results motivate capacity feasibility only; they are not Assignment 02 accuracy targets and used a different framework/dataset.
All ML processing remains visible in this notebook. Codex preparation uses bounded smoke batches only. The owner alone enables full epochs/sequential experiments in VS Code.
Your executed pre-M5 notebook and B0 experiment have been preserved outside Git.


## 2. Environment, configuration and finite plan
Use Katana Ubuntu WSL and the unchanged as04_yolo_katana environment. Physical hardware is i7-13620H, RTX 4060 Laptop 8 GB, 32 GB RAM; preparation observed WSL limits of 12 CPUs and about 23.5 GiB RAM.
Every execution control defaults false. Default Run All reads training metadata/defines functions only. It does not train, decode images, acquire weights or tune the GPU.
The finite plan defaults to two head warm-up epochs plus **100 full fine-tuning epochs per model**, with primary-H@1 early stopping. Edit stage budgets explicitly; longer finite budgets are supported. CONFIG.epochs is a legacy display value; stage budgets control execution.
After review, set PREPARE_RUN, RUN_TRAINING, RUN_EXPENSIVE_EVALUATION and RUN_PLANNED_EXPERIMENTS true to run both experiments sequentially once. Save the notebook before starting; code identity is checked on recovery.

Measured presets: head warm-up microbatch 64; initial full-stage microbatch 32; final-resolution microbatch 16; target effective batch 64; BF16/channels-last; four workers, prefetch factor 2, pinned memory and persistent workers. Final measured compute+transfer throughput was about 95.0 images/s (B3/380) and 53.7 images/s (Base/320); short warmed end-to-end tests with input workers measured about 70.7 and 47.5 images/s. These are bounded measurements, not full-epoch or overnight guarantees.


In [ ]:
import copy
import ast
import hashlib
import json
import math
import os
import platform
import re
import random
import gc
import itertools
from contextlib import nullcontext
import tempfile
from collections import Counter
from dataclasses import dataclass
from numbers import Integral, Real
from typing import Any
from urllib.parse import urlparse
import subprocess
import sys
import time
import uuid
from collections.abc import Mapping
from datetime import datetime, timezone
from pathlib import Path

def find_project_root(start, override=None):
    markers = ("DPD_Small_train.csv", "notebooks/wp2_backbone_comparison.ipynb")
    if override:
        candidates = [Path(override).expanduser().resolve()]
    else:
        start = Path(start).resolve()
        candidates = [start, *start.parents]
    for candidate in candidates:
        if all((candidate / name).is_file() for name in markers):
            return candidate
    raise FileNotFoundError("Open the WSL repository/notebooks folder or set COS30082_PROJECT_ROOT.")

ROOT = find_project_root(Path.cwd(), os.environ.get("COS30082_PROJECT_ROOT"))
if not sys.platform.startswith("linux") or "microsoft" not in platform.release().lower():
    raise RuntimeError("Use Ubuntu WSL on Katana for this notebook.")

import torch
import torchvision
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np
from PIL import Image
from torch import Tensor, nn
from torch.nn import functional as F
from torch.utils.data import Dataset, DataLoader
from torch.utils.checkpoint import checkpoint as activation_checkpoint
from torchvision.transforms import v2
from torchvision.models import (
    efficientnet_b0, convnext_tiny, EfficientNet_B0_Weights, ConvNeXt_Tiny_Weights,
    efficientnet_b3, convnext_base, EfficientNet_B3_Weights, ConvNeXt_Base_Weights,
)

EXPECTED_VERSIONS = {"python": "3.10.20", "torch": "2.5.1+cu121", "torchvision": "0.20.1+cu121"}
ENVIRONMENT = {
    "python": platform.python_version(), "torch": str(torch.__version__),
    "torchvision": str(torchvision.__version__), "cuda_build": torch.version.cuda,
    "python_executable": sys.executable, "platform": platform.platform(),
}
for name, expected in EXPECTED_VERSIONS.items():
    if ENVIRONMENT[name] != expected:
        raise RuntimeError(f"{name}: expected {expected}, found {ENVIRONMENT[name]}; do not change packages.")
if Path(sys.prefix).name != "as04_yolo_katana":
    raise RuntimeError("Select the existing as04_yolo_katana VS Code kernel.")
print("Project:", ROOT)
print(json.dumps(ENVIRONMENT, indent=2))

ENVIRONMENT["cuda_available"] = torch.cuda.is_available()
if torch.cuda.is_available():
    properties = torch.cuda.get_device_properties(0)
    ENVIRONMENT["gpu"] = properties.name
    ENVIRONMENT["vram_gib"] = properties.total_memory / 2**30
print("Working directory:", Path.cwd())
print(json.dumps(ENVIRONMENT, indent=2))


In [ ]:
PREPARE_RUN = False
RUN_MODEL_DIAGNOSTICS = False
RUN_TRAINING = False
RUN_EXPENSIVE_EVALUATION = False
SAVE_FINAL_SNAPSHOT = False
VERIFY_SAVED_CHECKPOINT = False
SHOW_DATA_IMAGES = False
SHOW_AUGMENTATIONS = False
RUN_IMAGE_AUDIT = False
RUN_BATCH_INSPECTION = False
RUN_PRETRAINED_ACQUISITION = False
RUN_METRIC_EXAMPLES = False
READ_RECORDED_RESULTS = False
RUN_PLANNED_EXPERIMENTS = False
RUN_GPU_TUNING = False

CONFIG = {
    "architecture": "multitask_efficientnet_b3",  # or multitask_convnext_base
    "initialization": "local_pretrained",  # explicitly choose random if intended
    "local_weight_path": "/home/user1/COS30082_pretrained_weights/efficientnet_b3_rwightman-b3899882.pth",
    "expected_weight_sha256": "b3899882250c22946d0229d266049fcd133c169233530b36b9ffa7983988362f",
    "seed": 42,
    "batch_size": 16,
    "epochs": 102,  # 2 head warm-up + 100 full fine-tuning; controlled by stage budgets
    "learning_rate": 0.0001,
    "optimizer": "AdamW",  # common optimizer family
    "weight_decay": 0.01,
    "crop_weight": 0.5,
    "disease_weight": 0.5,
    "device": "cuda:0",
    "augment_fit": True,
    "num_workers": 4,
    "log_every_batches": 10,
    "weight_source_note": "Official Torchvision HTTPS acquisition during M5 preparation; published filename hash prefix and full SHA-256 verified; weights_only tensor loading passed.",
    "preprocessing_policy": "common_rgb_imagenet_progressive_v1",
    "deterministic_algorithms": False,
    "output_directory": "/home/user1/COS30082_experiments/wp2",
    "run_id": None,
    "warmup_batch_size": 64, "start_batch_size": 32,
    "warmup_epochs": 2, "finetune_epochs": 100, "progressive_epochs": 5,
    "start_resolution": 300, "final_resolution": 380,
    "accumulation_steps": 4, "effective_batch_size": 64, "precision": "bf16", "channels_last": True,
    "gradient_checkpointing": False, "pin_memory": True,
    "prefetch_factor": 2, "persistent_workers": True,
    "warmup_head_lr": 0.001, "backbone_lr": 0.00001, "head_lr": 0.0001,
    "early_stopping_patience": 20, "min_finetune_epochs": 15,
    "clip_grad_norm": 1.0, "resume_from": None,  # None generates a UTC timestamp + random suffix when prepared
}
APPROVED_ARCHITECTURES = ("multitask_efficientnet_b3", "multitask_convnext_base")
PRETRAINED_IDS = {
    "multitask_efficientnet_b0": "EfficientNet_B0_Weights.IMAGENET1K_V1",
    "multitask_efficientnet_b3": "EfficientNet_B3_Weights.IMAGENET1K_V1",
    "multitask_convnext_base": "ConvNeXt_Base_Weights.IMAGENET1K_V1",
    "multitask_convnext_tiny": "ConvNeXt_Tiny_Weights.IMAGENET1K_V1",
}
SELECTION_POLICY = {
    "primary": "maximize_development_top1_harmonic_mean",
    "secondary": "minimize_sample_weighted_combined_validation_total_loss",
    "tertiary": "earliest_epoch",
    "comparison": "full_precision_exact_no_tolerance",
    "zero_zero_rule": "WP5 must return H@1=0 when both Top-1 accuracies are zero",
}

def validate_config(config):
    if config["architecture"] not in APPROVED_ARCHITECTURES:
        raise ValueError("Select EfficientNet-B3 or ConvNeXt-Base for final experiments.")
    if config["initialization"] not in ("local_pretrained", "random"):
        raise ValueError("Explicitly select local_pretrained or random initialization.")
    for key in ("batch_size", "epochs", "log_every_batches"):
        if type(config[key]) is not int or config[key] < 1:
            raise ValueError(f"{key} must be a positive integer.")
    for key in ("seed", "num_workers"):
        if type(config[key]) is not int or config[key] < 0:
            raise ValueError(f"{key} must be a nonnegative integer.")
    if config["seed"] >= 2**32:
        raise ValueError("seed must be below 2**32.")
    for key in ("learning_rate", "weight_decay", "crop_weight", "disease_weight"):
        value = config[key]
        if isinstance(value, bool) or not isinstance(value, (int, float)) or not math.isfinite(value) or value < 0:
            raise ValueError(f"{key} must be finite and nonnegative.")
    if config["learning_rate"] == 0 or config["crop_weight"] + config["disease_weight"] == 0:
        raise ValueError("Learning rate and at least one loss weight must be positive.")
    if config["optimizer"] != "AdamW":
        raise ValueError("Use the common AdamW optimizer family for this comparison.")
    for key in ("augment_fit", "deterministic_algorithms"):
        if type(config[key]) is not bool:
            raise TypeError(f"{key} must be a bool.")
    if config["preprocessing_policy"] not in ("common_wp1_square224_v1", "common_rgb_imagenet_progressive_v1"):
        raise ValueError("Only the preserved common WP1 pipeline is supported; review alternatives separately.")
    if config["run_id"] is not None and (
        not isinstance(config["run_id"], str)
        or not re.fullmatch(r"[A-Za-z0-9][A-Za-z0-9_-]{0,95}", config["run_id"])
    ):
        raise ValueError("run_id must be a simple unique name, without path separators.")
    if not isinstance(config["output_directory"], str) or not Path(config["output_directory"]).is_absolute():
        raise ValueError("output_directory must be an absolute WSL path.")
    device = torch.device(config["device"])
    if device.type not in ("cpu", "cuda"):
        raise ValueError("Use an explicit CPU or CUDA device.")
    for key in ("warmup_batch_size", "start_batch_size", "warmup_epochs", "finetune_epochs", "progressive_epochs", "start_resolution",
                "final_resolution", "accumulation_steps", "effective_batch_size", "prefetch_factor", "early_stopping_patience",
                "min_finetune_epochs"):
        if type(config[key]) is not int or config[key] < 1:
            raise ValueError(f"{key} must be a positive integer.")
    if config["precision"] not in ("fp32", "fp16", "bf16"):
        raise ValueError("precision must be fp32, fp16 or bf16")
    if config["gradient_checkpointing"] and "convnext" not in config["architecture"]:
        raise ValueError("Activation checkpointing is supported for ConvNeXt only, avoiding double BatchNorm updates.")
    for key in ("warmup_head_lr", "backbone_lr", "head_lr", "clip_grad_norm"):
        if not isinstance(config[key], (int, float)) or not math.isfinite(config[key]) or config[key] <= 0:
            raise ValueError(f"{key} must be finite and positive")
    if config.get("fitting_sampler", "uniform_shuffle") not in ("uniform_shuffle", "capped_disease_tickets"):
        raise ValueError("Unknown fitting sampler.")
    cap = config.get("sampler_weight_cap", 3)
    if type(cap) is not int or not 1 <= cap <= 4:
        raise ValueError("Sampler ticket cap must be an integer from1 to4.")
    if config.get("early_stopping_reference", "global_best_h1") != "global_best_h1":
        raise ValueError("New runs use global-best H@1 stopping.")
    return copy.deepcopy(config)

def config_fingerprint(config):
    return hashlib.sha256(json.dumps(config, sort_keys=True, allow_nan=False).encode()).hexdigest()

validate_config(CONFIG)
print("Defaults are inert. Configure weights and explicit execution flags before preparing a run.")

print(json.dumps(CONFIG, indent=2))
RECORDED_RUN_SUMMARIES = []  # explicit run_summary.json paths for section 15

EXPERIMENT_PLAN = [
    copy.deepcopy(CONFIG),
    {**copy.deepcopy(CONFIG),
     "architecture": "multitask_convnext_base",
     "local_weight_path": "/home/user1/COS30082_pretrained_weights/convnext_base-6075fbad.pth",
     "expected_weight_sha256": "6075fbad31c688f9ed013922a59daf0fc2f4dc12e6e43bc01fa840c7bd1ca2cd",
     "start_resolution": 224, "final_resolution": 320,
     "batch_size": 16, "accumulation_steps": 4},
]
# Update measured microbatch/precision settings below only after bounded tuning.
# Two warm-up + 100 fine-tune epochs are a finite configurable budget, not an automatic launch.


In [ ]:
def seed_everything(seed=42):
    """Seed Python, NumPy and Torch; CUDA kernels may still be nondeterministic."""
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)

def seed_worker(worker_id):
    worker_seed = torch.initial_seed() % (2**32)
    random.seed(worker_seed)
    np.random.seed(worker_seed)

if PREPARE_RUN:
    seed_everything(CONFIG["seed"])
    torch.backends.cudnn.benchmark = False
    torch.backends.cudnn.deterministic = True
    torch.use_deterministic_algorithms(CONFIG["deterministic_algorithms"])
else:
    print("Seed/backend setup skipped: PREPARE_RUN=False.")


## 3. Dataset exploration and completeness
Read only DPD_Small_train.csv. Validate all seven columns, filename safety, unique rows, component/name consistency and composition/component correspondence. Metadata completeness is checked without decoding images.
Image existence/decoding checks and labelled image examples are separately guarded and cover training metadata only. They are not a claim that a fresh image audit has already run.


In [ ]:
COLUMN_NAMES = ["image_name", "crop_id", "disease_id", "composition_id",
                "crop_name", "disease_name", "composition_name"]
DEFAULT_CSVS = ("DPD_Small_train.csv",)
SPLIT_VERSION = "pytorch-v1"

def load_metadata(path):
    frame = pd.read_csv(path, header=None)
    if frame.shape[1] != len(COLUMN_NAMES) or frame.empty:
        raise ValueError(f"Expected non-empty seven-column metadata: {path}")
    frame.columns = COLUMN_NAMES
    if frame.isna().any().any():
        raise ValueError(f"Missing metadata values: {path}")
    for col in ("crop_id", "disease_id", "composition_id"):
        numeric = pd.to_numeric(frame[col], errors="raise")
        if ((numeric < 0) | (numeric % 1 != 0)).any():
            raise ValueError(f"Invalid {col}: {path}")
        frame[col] = numeric.astype("int64")
    for name in frame.image_name:
        if not isinstance(name, str) or name in (".", "..") or any(c in name for c in ("/", "\\", ":")):
            raise ValueError(f"Expected flat image filename, got {name!r}")
    return frame

def validate_metadata(train, seen, unseen):
    frames = [train, seen, unseen]
    for frame in frames:
        if frame.image_name.duplicated().any():
            raise ValueError("Duplicate image names within a split")
    for i in range(3):
        for j in range(i + 1, 3):
            if set(frames[i].image_name) & set(frames[j].image_name):
                raise ValueError("Image filename overlap across official splits")
    all_rows = pd.concat(frames)
    for kind in ("crop", "disease", "composition"):
        pairs = all_rows[[f"{kind}_id", f"{kind}_name"]].drop_duplicates()
        if pairs[f"{kind}_id"].duplicated().any() or pairs[f"{kind}_name"].duplicated().any():
            raise ValueError(f"Inconsistent {kind} ID/name mapping")
    pairs = all_rows[["composition_id", "crop_id", "disease_id"]].drop_duplicates()
    if pairs.composition_id.duplicated().any() or pairs.duplicated(["crop_id", "disease_id"]).any():
        raise ValueError("Inconsistent composition/component mapping")
    if set(train.composition_id) & set(unseen.composition_id):
        raise ValueError("Official unseen compositions appear in training")
    if not set(seen.composition_id) <= set(train.composition_id):
        raise ValueError("Seen-test contains compositions absent from training")
    for col in ("crop_id", "disease_id"):
        if set(train[col]) != set(range(train[col].nunique())):
            raise ValueError(f"Training {col} must be contiguous and start at zero")
        if not set(all_rows[col]) <= set(train[col]):
            raise ValueError(f"Test {col} absent from training")
    return {"split_sizes": [len(f) for f in frames],
            "num_crops": int(train.crop_id.nunique()),
            "num_diseases": int(train.disease_id.nunique()),
            "num_seen_compositions": int(train.composition_id.nunique()),
            "num_unseen_compositions": int(unseen.composition_id.nunique()),
            "singleton_compositions": int((train.composition_id.value_counts() == 1).sum()),
            "filename_overlap": False, "seen_unseen_composition_overlap": False}

source_frame = load_metadata(ROOT / "DPD_Small_train.csv")
validate_metadata(source_frame, source_frame.iloc[:0], source_frame.iloc[:0])
if (len(source_frame), source_frame.crop_id.nunique(), source_frame.disease_id.nunique(),
        source_frame.composition_id.nunique()) != (8000, 55, 175, 323):
    raise RuntimeError("Unexpected training metadata; investigate before preparing an experiment.")
print("Metadata shape:", source_frame.shape)
print(source_frame.head().to_string(index=False))
print(source_frame[["crop_id", "disease_id", "composition_id"]].nunique().to_string())
print("Largest crop classes:\n", source_frame.crop_name.value_counts().head(10).to_string())
print("Largest disease classes:\n", source_frame.disease_name.value_counts().head(10).to_string())
print("Composition image counts:\n", source_frame.composition_id.value_counts().describe().to_string())
IMAGE_DIRECTORY = ROOT / "DPD_Small_dataset"


In [ ]:
if RUN_IMAGE_AUDIT:
    failures = []
    for index, name in enumerate(source_frame.image_name, 1):
        path = IMAGE_DIRECTORY / name
        try:
            with Image.open(path) as image:
                image.load()
        except (OSError, ValueError) as error:
            failures.append({"image": name, "error": str(error)})
        if index % 1000 == 0:
            print(f"Checked {index}/{len(source_frame)} training images")
    print("Training-only image audit:", len(source_frame), "images;", len(failures), "failures")
    if failures:
        raise RuntimeError(f"Training image audit failed: {failures[:5]}")
else:
    print("Training image audit not run.")
if SHOW_DATA_IMAGES:
    example_rows = source_frame.sample(n=4, random_state=CONFIG["seed"])
    fig, axes = plt.subplots(1, 4, figsize=(14, 4))
    for axis, row in zip(axes, example_rows.itertuples()):
        with Image.open(IMAGE_DIRECTORY / row.image_name) as image:
            axis.imshow(image.convert("RGB"))
        axis.set_title(f"{row.crop_name}\n{row.disease_name}\nglobal ID {row.composition_id}")
        axis.axis("off")
    plt.tight_layout()
    plt.show()
else:
    print("Labelled image examples disabled.")


## 4. Common policy with explicit progressive resolution
Both models use the same RGB, square-resize, fitting-only flips/affine augmentation and ImageNet normalization. The local functions are visible and WP1 files are untouched.
B3 starts at 300 and progresses to 380; Base warms up/verifies at 224 and prioritizes 320. These architecture-specific resolutions are deliberate and recorded, not a claim of identical computational cost.
Validation uses the current epoch resolution, without augmentation. Each checkpoint retains the full stage/resolution policy and its selected epoch resolution.
Native ImageNet inference presets are B3 resize 320/bicubic → crop 300, Base resize 232/bilinear → crop 224. The supervised square policy is a documented deviation.


In [ ]:
IMAGENET_MEAN = (0.485, 0.456, 0.406)
IMAGENET_STD = (0.229, 0.224, 0.225)

def get_transforms(image_size=224, augment=False, normalize_imagenet=True):
    if image_size <= 0:
        raise ValueError("image_size must be positive")
    steps = [v2.ToImage(), v2.Resize((image_size, image_size), antialias=True)]
    if augment:
        steps += [
            v2.RandomHorizontalFlip(), v2.RandomVerticalFlip(),
            v2.RandomAffine(degrees=18, translate=(0.05, 0.05), scale=(0.95, 1.05)),
        ]
    steps += [v2.ToDtype(torch.float32, scale=True)]
    if normalize_imagenet:
        steps += [v2.Normalize(IMAGENET_MEAN, IMAGENET_STD)]
    return v2.Compose(steps)

def preprocessing_record(config):
    size = config.get("active_resolution", config.get("final_resolution", 224))
    return {
        "policy": config["preprocessing_policy"], "image_size": [size, size],
        "resize": {"method": "square", "interpolation": "bilinear", "antialias": True},
        "dtype": "float32", "rgb": True, "scale_uint8_to_unit_float": True,
        "mean": [0.485, 0.456, 0.406], "std": [0.229, 0.224, 0.225],
        "augment_fit": config["augment_fit"],
        "augmentation": {
            "horizontal_flip_p": 0.5, "vertical_flip_p": 0.5,
            "affine_degrees": [-18, 18], "translate": [0.05, 0.05],
            "scale": [0.95, 1.05], "affine_interpolation": "nearest", "fill": 0,
        } if config["augment_fit"] else None,
        "fit_transform": repr(get_transforms(size, augment=config["augment_fit"], normalize_imagenet=True)),
        "validation_transform": repr(get_transforms(size, augment=False, normalize_imagenet=True)),
    }

preprocessing = preprocessing_record(CONFIG)
print(json.dumps(preprocessing, indent=2))


In [ ]:
if SHOW_AUGMENTATIONS:
    with torch.random.fork_rng(devices=[]):
        torch.manual_seed(CONFIG["seed"])
        row = source_frame.iloc[0]
        with Image.open(IMAGE_DIRECTORY / row.image_name) as opened:
            image = opened.convert("RGB")
            examples = [get_transforms(224, augment=True)(image) for _ in range(4)]
        fig, axes = plt.subplots(1, 4, figsize=(12, 3))
        for axis, tensor in zip(axes, examples):
            restored = tensor * torch.tensor(IMAGENET_STD)[:, None, None]
            restored += torch.tensor(IMAGENET_MEAN)[:, None, None]
            axis.imshow(restored.clamp(0, 1).permute(1, 2, 0).numpy())
            axis.axis("off")
        plt.suptitle("Four augmentation draws from one training image")
        plt.show()
else:
    print("Augmented image visualization disabled.")


## 5. Three-way development split
The visible algorithm reproduces the accepted seed-42 procedure: shuffle sorted composition groups, hold out 20% of compositions while leaving every crop/disease component in fitting, then apply the accepted classwise 20% image split to the retained portion.
Singletons remain in fitting. Classwise rounding and source-row order are preserved. A fresh random.Random(42) is used for each of the two split stages, as in the accepted algorithm.
Verify exact coverage, labels and component sets. No manifest is rewritten. All global composition IDs remain unchanged; both folds use one ordered 323-candidate universe.


In [ ]:
def make_validation_split(frame, val_split=0.2, seed=42, mode="stratified"):
    """Classwise split keeps singletons in train; heldout removes whole classes.

    Classwise rounding makes the realized image fraction approximate.
    heldout mode interprets val_split as a fraction of composition classes.
    """
    if not 0 < val_split < 1:
        raise ValueError("val_split must be between 0 and 1")
    frame = frame.reset_index(drop=True)
    rng = random.Random(seed)
    groups = {int(key): list(group.index) for key, group in frame.groupby("composition_id", sort=True)}
    val_indices = []
    if mode == "stratified":
        for indices in groups.values():
            rng.shuffle(indices)
            n = min(len(indices) - 1, max(1, round(len(indices) * val_split))) if len(indices) > 1 else 0
            val_indices.extend(indices[:n])
    elif mode == "heldout_composition":
        candidates = list(groups)
        rng.shuffle(candidates)
        remaining_crop = Counter(frame.crop_id)
        remaining_disease = Counter(frame.disease_id)
        target = max(1, round(len(groups) * val_split))
        heldout = []
        for comp_id in candidates:
            group = frame.loc[groups[comp_id]]
            crops, diseases = Counter(group.crop_id), Counter(group.disease_id)
            if all(remaining_crop[k] > n for k, n in crops.items()) and all(remaining_disease[k] > n for k, n in diseases.items()):
                heldout.append(comp_id)
                val_indices.extend(groups[comp_id])
                remaining_crop.subtract(crops)
                remaining_disease.subtract(diseases)
                if len(heldout) == target:
                    break
        if len(heldout) < target:
            raise ValueError(f"Only {len(heldout)} valid heldout classes; requested {target}. Reduce val_split.")
    else:
        raise ValueError(f"Unknown split mode: {mode}")
    validation = frame.loc[sorted(val_indices)].reset_index(drop=True)
    training = frame.drop(index=val_indices).reset_index(drop=True)
    if training.empty or validation.empty:
        raise ValueError("Split produced an empty training or validation set")
    for col in ("crop_id", "disease_id"):
        if set(frame[col]) != set(training[col]):
            raise ValueError(f"Split removed a training component: {col}")
    return training, validation

def _check_partitions(
    source: pd.DataFrame, fit: pd.DataFrame, seen: pd.DataFrame, unseen: pd.DataFrame,
) -> None:
    combined = pd.concat([fit, seen, unseen], ignore_index=True)
    if (combined.image_name.duplicated().any()
            or set(combined.image_name) != set(source.image_name)):
        raise ValueError("Development partitions must be disjoint and cover training filenames")
    if not source.sort_values("image_name").reset_index(drop=True).equals(
        combined.sort_values("image_name").reset_index(drop=True)
    ):
        raise ValueError("Development partitions changed original metadata rows or label types")
    fit_ids, seen_ids, unseen_ids = (set(frame.composition_id) for frame in (fit, seen, unseen))
    if not seen_ids <= fit_ids or unseen_ids & (fit_ids | seen_ids):
        raise ValueError("Seen coverage or held-out composition isolation failed")
    for column in ("crop_id", "disease_id"):
        if set(fit[column]) != set(source[column]):
            raise ValueError(f"Fitting partition lost training components: {column}")


In [ ]:
@dataclass(frozen=True)
class DevelopmentSplit:
    """Original labels and one candidate universe for both validation folds.

    Treat the contained DataFrames/mappings as read-only. Frozen field bindings
    do not make pandas objects immutable; DPDataset copies each frame.
    """

    fit: pd.DataFrame
    val_seen: pd.DataFrame
    val_unseen: pd.DataFrame
    mappings: dict[str, Any]
    candidate_ids: tuple[int, ...]
    candidate_pairs: tuple[tuple[int, int], ...]
    source_csv: str
    source_sha256: str
    candidate_sha256: str
    seed: int
    heldout_fraction: float
    seen_fraction: float
    split_version: str = f"{SPLIT_VERSION}/three-way-v1"

def _fraction(value: float, name: str) -> float:
    if isinstance(value, bool) or not isinstance(value, Real) or not 0 < value < 1:
        raise ValueError(f"{name} must be a finite fraction between zero and one")
    return float(value)

def _training_mappings(frame: pd.DataFrame) -> dict[str, Any]:
    """Build only training-derived maps; the combined train/test wrapper is unsuitable."""
    mappings = {}
    for kind in ("crop", "disease"):
        pairs = frame[[f"{kind}_id", f"{kind}_name"]].drop_duplicates()
        pairs = pairs.sort_values(f"{kind}_id")
        id_to_name = {int(key): str(name) for key, name in pairs.itertuples(index=False, name=None)}
        mappings[f"{kind}_id_to_name"] = id_to_name
        mappings[f"{kind}_name_to_id"] = {name: key for key, name in id_to_name.items()}
        mappings[f"num_{kind}s"] = len(id_to_name)
    pairs = frame[["composition_id", "crop_id", "disease_id", "composition_name"]]
    pairs = pairs.drop_duplicates().sort_values("composition_id")
    mappings["composition_id_to_pair"] = {
        int(row.composition_id): (int(row.crop_id), int(row.disease_id))
        for row in pairs.itertuples()
    }
    mappings["composition_id_to_name"] = {
        int(row.composition_id): str(row.composition_name) for row in pairs.itertuples()
    }
    ids = tuple(mappings["composition_id_to_pair"])
    mappings["composition_id_to_index"] = {comp: index for index, comp in enumerate(ids)}
    mappings["composition_index_to_id"] = dict(enumerate(ids))
    mappings["num_compositions"] = len(ids)
    return mappings

def build_development_split(
    train_csv: str | Path = DEFAULT_CSVS[0],
    *,
    seed: int = 42,
    heldout_fraction: float = 0.2,
    seen_fraction: float = 0.2,
) -> DevelopmentSplit:
    """Compose existing train-only splits without reading images or writing files.

    The default DPD-Small source yields 4868 fit / 1238 seen / 1894 unseen
    images. heldout_fraction counts compositions; seen_fraction is classwise
    image sampling within the retained portion. Synthetic training CSVs are
    supported for tests. Known official test CSV paths are rejected.
    """
    if isinstance(seed, bool) or not isinstance(seed, Integral) or not 0 <= seed < 2**32:
        raise ValueError("seed must be an integer in [0, 2**32)")
    seed = int(seed)
    heldout_fraction = _fraction(heldout_fraction, "heldout_fraction")
    seen_fraction = _fraction(seen_fraction, "seen_fraction")
    path = Path(train_csv)
    if path.name.startswith("DPD_Small_test_") or path.resolve().name.startswith("DPD_Small_test_"):
        raise ValueError("Official test metadata cannot be used as development training data")
    before_hash = hashlib.sha256(path.read_bytes()).hexdigest()
    source = load_metadata(path)
    if hashlib.sha256(path.read_bytes()).hexdigest() != before_hash:
        raise ValueError("Training CSV changed while loading")
    # Empty control frames reuse WP1 validation without loading either test CSV.
    empty = source.iloc[:0]
    validate_metadata(source, empty, empty)
    retained, unseen = make_validation_split(
        source, val_split=heldout_fraction, seed=seed, mode="heldout_composition"
    )
    fit, seen = make_validation_split(
        retained, val_split=seen_fraction, seed=seed, mode="stratified"
    )
    _check_partitions(source, fit, seen, unseen)
    mappings = _training_mappings(source)
    ids = tuple(mappings["composition_id_to_pair"])
    pairs = tuple(mappings["composition_id_to_pair"][comp] for comp in ids)
    records = [[comp, *pair] for comp, pair in zip(ids, pairs)]
    candidate_hash = hashlib.sha256(
        json.dumps(records, separators=(",", ":")).encode("utf-8")
    ).hexdigest()
    return DevelopmentSplit(
        fit=fit, val_seen=seen, val_unseen=unseen, mappings=mappings,
        candidate_ids=ids, candidate_pairs=pairs, source_csv=str(path.resolve()),
        source_sha256=before_hash, candidate_sha256=candidate_hash, seed=seed,
        heldout_fraction=heldout_fraction, seen_fraction=seen_fraction,
    )


In [ ]:
development = None
candidate_spec = None
class_mappings = None
development = build_development_split(ROOT / "DPD_Small_train.csv", seed=42,
                                     heldout_fraction=0.2, seen_fraction=0.2)
sizes = tuple(len(frame) for frame in (development.fit, development.val_seen, development.val_unseen))
if sizes != (4868, 1238, 1894) or len(development.candidate_ids) != 323:
    raise RuntimeError(f"Unexpected development contract: {sizes}; investigate before training.")
candidate_spec = {
    "ids": development.candidate_ids, "pairs": development.candidate_pairs,
    "sha256": development.candidate_sha256,
    "scoring_requirement": "sum_log_softmax_v1",
}
class_mappings = {key: development.mappings[key] for key in
                  ("crop_id_to_name", "disease_id_to_name")}
print("Images:", sizes, "| candidate count:", len(candidate_spec["ids"]))
print("Training CSV SHA-256:", development.source_sha256)
print("Candidate SHA-256:", development.candidate_sha256)
print("First candidate ID/pairs:", list(zip(candidate_spec["ids"], candidate_spec["pairs"]))[:5])
split_membership = {name: getattr(development, name).image_name.tolist()
                    for name in ("fit", "val_seen", "val_unseen")}
split_sha256 = hashlib.sha256(json.dumps(split_membership, sort_keys=True,
                                       separators=(",", ":")).encode()).hexdigest()
print("Partition membership SHA-256:", split_sha256)
print(pd.DataFrame([
    {"partition": name, "images": len(getattr(development, name)),
     "crops": getattr(development, name).crop_id.nunique(),
     "diseases": getattr(development, name).disease_id.nunique(),
     "compositions": getattr(development, name).composition_id.nunique()}
    for name in ("fit", "val_seen", "val_unseen")
]).to_string(index=False))
assert set(development.fit.crop_id) == set(range(55))
assert set(development.fit.disease_id) == set(range(175))


## 6. Visible Dataset and DataLoaders
DPDataset below constructs paths, opens images with PIL, converts RGB, applies the visible transforms and creates int64 crop/disease/global-composition targets.
Loader construction is lazy and guarded. The fitting loader shuffles and augments; validation loaders do neither. Worker seeds and generator seeds are explicit. Batch inspection is a separate manual image-reading action and restores RNG/generator state so it does not consume the fitting order.


In [ ]:
class DPDataset(Dataset):
    """Return float32 CHW RGB images and int64 crop/disease/composition targets."""

    def __init__(self, frame, img_dir, image_size=224, as_dict_targets=True,
                 augment=False, normalize_imagenet=True):
        self.frame = frame.reset_index(drop=True).copy()
        self.img_dir = Path(img_dir)
        self.as_dict_targets = as_dict_targets
        self.transform = get_transforms(image_size, augment, normalize_imagenet)

    def __len__(self):
        return len(self.frame)

    def __getitem__(self, index):
        row = self.frame.iloc[index]
        path = self.img_dir / row.image_name
        try:
            with Image.open(path) as image:
                image = self.transform(image.convert("RGB"))
        except (OSError, ValueError) as exc:
            raise RuntimeError(f"Cannot decode image: {path}") from exc
        values = tuple(torch.tensor(int(row[name]), dtype=torch.long)
                       for name in ("crop_id", "disease_id", "composition_id"))
        targets = dict(zip(("crop", "disease", "composition"), values))
        return image, targets if self.as_dict_targets else values

def get_development_dataloaders(
    split: DevelopmentSplit,
    img_dir: str | Path = "DPD_Small_dataset",
    *,
    batch_size: int = 16,
    image_size: int = 224,
    augment_fit: bool = True,
    as_dict_targets: bool = True,
    normalize_imagenet: bool = True,
    num_workers: int = 0,
    pin_memory: bool = False,
) -> tuple[DataLoader, DataLoader, DataLoader]:
    """Return fit/seen-validation/unseen-validation loaders, with no iteration.

    Reuses DPDataset and its transforms. Only fitting shuffles/augments; both
    validation loaders use the same split candidate_ids/candidate_pairs.
    Construct once per run: seeding follows the existing helper's global RNG
    behavior and does not promise deterministic CUDA kernels.
    """
    if not isinstance(split, DevelopmentSplit):
        raise TypeError("split must be a DevelopmentSplit")
    for name, value, minimum in (
        ("batch_size", batch_size, 1), ("image_size", image_size, 1),
        ("num_workers", num_workers, 0),
    ):
        if isinstance(value, bool) or not isinstance(value, Integral) or value < minimum:
            raise ValueError(f"{name} must be an integer >= {minimum}")
    image_dir = Path(img_dir)
    if not image_dir.is_dir():
        raise FileNotFoundError(f"Image directory not found: {image_dir}")
    seed_everything(split.seed)
    loaders = []
    for index, frame in enumerate((split.fit, split.val_seen, split.val_unseen)):
        dataset = DPDataset(
            frame, image_dir, image_size=int(image_size), as_dict_targets=as_dict_targets,
            augment=augment_fit and index == 0, normalize_imagenet=normalize_imagenet,
        )
        loaders.append(DataLoader(
            dataset, batch_size=int(batch_size), shuffle=index == 0,
            num_workers=int(num_workers), pin_memory=pin_memory,
            worker_init_fn=seed_worker,
            generator=torch.Generator().manual_seed(split.seed + index),
        ))
    return tuple(loaders)


In [ ]:
preview_loaders = None
if PREPARE_RUN:
    preview_loaders = get_development_dataloaders(
        development, IMAGE_DIRECTORY, batch_size=CONFIG["batch_size"], image_size=CONFIG["start_resolution"],
        augment_fit=CONFIG["augment_fit"], num_workers=CONFIG["num_workers"],
        pin_memory=CONFIG["device"].startswith("cuda"))
    print(pd.DataFrame([
        {"fold": name, "images": len(loader.dataset), "batches": len(loader),
         "batch_size": loader.batch_size}
        for name, loader in zip(("fit", "seen", "unseen"), preview_loaders)
    ]).to_string(index=False))
else:
    print("Lazy DataLoader preparation skipped.")
if RUN_BATCH_INSPECTION:
    if preview_loaders is None or CONFIG["num_workers"] != 0:
        raise RuntimeError("Prepare loaders and use num_workers=0 for reproducible batch inspection.")
    loader = preview_loaders[0]
    generator_state = loader.generator.get_state()
    python_state, numpy_state = random.getstate(), np.random.get_state()
    try:
        with torch.random.fork_rng(devices=[]):
            images, labels = next(iter(loader))
        print("Images:", images.shape, images.dtype)
        print("Targets:", {key: (value.shape, str(value.dtype)) for key, value in labels.items()})
        print("Example global targets:", {key: value[:4].tolist() for key, value in labels.items()})
    finally:
        loader.generator.set_state(generator_state)
        random.setstate(python_state)
        np.random.set_state(numpy_state)
else:
    print("Image batch inspection disabled.")


## 7. Final architectures and engineering references
EfficientNet-B3 exports 1,536 pooled features and independent 55/175 heads. ConvNeXt-Base exports 1,024 features while retaining final LayerNorm/flattening. No model softmax.
All constructors use weights=None; verified stock ImageNet state is loaded strictly before classifier removal. No CUB-200 fine-tuned heads are used.
B0/Tiny construction remains available only to preserve regression/reference compatibility. CONFIG/EXPERIMENT_PLAN permit only B3 and Base for final training.
Optional non-reentrant activation checkpointing is implemented for ConvNeXt; BatchNorm EfficientNet checkpointing is rejected to avoid duplicated running-stat updates.


In [ ]:
MODEL_FEATURE_DIMS = {"multitask_efficientnet_b0": 1280, "multitask_convnext_tiny": 768,
                      "multitask_efficientnet_b3": 1536, "multitask_convnext_base": 1024}

class MultiTaskBackbone(nn.Module):
    """One shared feature extractor and two independent raw-logit heads."""

    def __init__(
        self,
        architecture: str,
        *,
        backbone_state_dict: Mapping[str, Tensor] | None = None,
    ) -> None:
        super().__init__()
        if architecture == "multitask_efficientnet_b0":
            backbone = efficientnet_b0(weights=None)
            feature_dim = backbone.classifier[-1].in_features
        elif architecture == "multitask_efficientnet_b3":
            backbone = efficientnet_b3(weights=None)
            feature_dim = backbone.classifier[-1].in_features
        elif architecture == "multitask_convnext_base":
            backbone = convnext_base(weights=None)
            feature_dim = backbone.classifier[-1].in_features
        elif architecture == "multitask_convnext_tiny":
            backbone = convnext_tiny(weights=None)
            feature_dim = backbone.classifier[-1].in_features
        else:
            raise ValueError("Unsupported notebook backbone architecture")
        if backbone_state_dict is not None:
            backbone.load_state_dict(backbone_state_dict, strict=True)
        if architecture.startswith("multitask_efficientnet"):
            # Torchvision already pools and flattens before this classifier.
            # Remove both classifier dropout and the ImageNet linear layer.
            backbone.classifier = nn.Identity()
        else:
            # Retain final LayerNorm2d and flatten after global pooling.
            backbone.classifier[-1] = nn.Identity()
        self.architecture = architecture
        self.feature_dim = feature_dim
        self.backbone = backbone
        self.crop_head = nn.Linear(feature_dim, 55)
        self.disease_head = nn.Linear(feature_dim, 175)
        self.gradient_checkpointing = False

    def forward(self, images: Tensor) -> dict[str, Tensor]:
        """Compute features once; export them before either prediction head."""
        if self.gradient_checkpointing and self.training and torch.is_grad_enabled():
            if "convnext" not in self.architecture:
                raise ValueError("Checkpointing BatchNorm backbones would duplicate running-stat updates.")
            x = images
            for stage in self.backbone.features:
                x = activation_checkpoint(stage, x, use_reentrant=False, preserve_rng_state=True)
            features = self.backbone.classifier(self.backbone.avgpool(x))
        else:
            features = self.backbone(images)
        return {
            "crop_logits": self.crop_head(features),
            "disease_logits": self.disease_head(features),
            "features": features,
        }

def create_multitask_model(architecture, *, backbone_state_dict=None):
    if not isinstance(architecture, str) or architecture not in MODEL_FEATURE_DIMS:
        raise ValueError(f"Unsupported notebook architecture: {architecture!r}")
    return MultiTaskBackbone(architecture, backbone_state_dict=backbone_state_dict)


In [ ]:
if RUN_MODEL_DIAGNOSTICS:
    validate_config(CONFIG)
    with torch.random.fork_rng(devices=[]):
        torch.manual_seed(CONFIG["seed"])
        diagnostic_model = create_multitask_model(CONFIG["architecture"]).cpu().eval()
        with torch.inference_mode():
            diagnostic = diagnostic_model(torch.zeros(1, 3, 64, 64))
        print(diagnostic_model)
        print("Parameters:", sum(p.numel() for p in diagnostic_model.parameters()))
        print("Trainable:", sum(p.numel() for p in diagnostic_model.parameters() if p.requires_grad))
        print({key: tuple(value.shape) for key, value in diagnostic.items()})
        if not all(torch.isfinite(value).all().item() for value in diagnostic.values()):
            raise RuntimeError("Synthetic diagnostic produced non-finite output.")
        del diagnostic_model, diagnostic
else:
    print("Synthetic model diagnostic skipped: RUN_MODEL_DIAGNOSTICS=False.")


## 8. Verified ImageNet initialization
Preparation acquired official B3/Base V1 artifacts outside Git, verified Torchvision's published filename hash prefixes, computed full SHA-256 and safely loaded tensor dictionaries. Full hashes/paths are configured below; acquisition provenance is in the handoff.
Local loading always verifies the configured full hash. There is no random fallback. Final experiments reject random mode; explicit random mode remains for controlled engineering checks.
RUN_PRETRAINED_ACQUISITION is independently false. Acquisition uses official HTTPS URLs and cannot happen on import/open/default Run All.


In [ ]:
def stream_sha256(stream):
    stream.seek(0)
    digest = hashlib.sha256()
    for block in iter(lambda: stream.read(1024 * 1024), b""):
        digest.update(block)
    return digest.hexdigest()

def file_sha256(path):
    with Path(path).open("rb") as stream:
        return stream_sha256(stream)

def verified_backbone_state(config):
    if config["initialization"] == "random":
        return None, {"mode": "random", "source": "Torchvision weights=None; random task heads"}
    if config["initialization"] != "local_pretrained":
        raise ValueError("Unknown initialization mode; no fallback.")
    path_value = config["local_weight_path"]
    expected = config["expected_weight_sha256"]
    if not path_value or not isinstance(expected, str) or not re.fullmatch(r"[0-9a-fA-F]{64}", expected):
        raise ValueError("Local pretrained mode requires a file path and explicit 64-character SHA-256.")
    path = Path(path_value).expanduser().resolve()
    if not path.is_file():
        raise FileNotFoundError(f"Approved local weights missing: {path}")
    expected = expected.lower()
    # Hash and load the same open file; verify again before accepting its state.
    with path.open("rb") as stream:
        actual = stream_sha256(stream)
        if actual != expected:
            raise ValueError(f"Pretrained SHA-256 mismatch for {path}; no fallback.")
        stream.seek(0)
        state = torch.load(stream, map_location="cpu", weights_only=True)
        stream.seek(0)
        if stream_sha256(stream) != expected:
            raise ValueError("Local weight file changed while loading.")
    if not isinstance(state, Mapping) or not state or not all(
        isinstance(key, str) and isinstance(value, torch.Tensor) for key, value in state.items()
    ):
        raise ValueError("Expected a nonempty stock Torchvision tensor state dictionary.")
    return state, {
        "mode": "local_pretrained", "identifier": PRETRAINED_IDS[config["architecture"]],
        "path": str(path), "sha256": expected, "bytes": path.stat().st_size,
        "task_heads": "random",
        "official_source_url": WEIGHT_ENUMS[config["architecture"]].url,
        "source_note": config.get("weight_source_note", ""),
    }


WEIGHT_ENUMS = {
    "multitask_efficientnet_b0": EfficientNet_B0_Weights.IMAGENET1K_V1,
    "multitask_convnext_tiny": ConvNeXt_Tiny_Weights.IMAGENET1K_V1,
    "multitask_efficientnet_b3": EfficientNet_B3_Weights.IMAGENET1K_V1,
    "multitask_convnext_base": ConvNeXt_Base_Weights.IMAGENET1K_V1,
}

def acquire_official_weights(config):
    if config["initialization"] != "local_pretrained":
        raise ValueError("Acquisition applies only to explicit local_pretrained mode.")
    expected = config["expected_weight_sha256"]
    if not config["local_weight_path"] or not isinstance(expected, str) or not re.fullmatch(r"[0-9a-fA-F]{64}", expected):
        raise ValueError("Provide destination and independently recorded full SHA-256.")
    destination = Path(config["local_weight_path"]).expanduser().resolve()
    if destination.exists():
        raise FileExistsError(destination)
    weights = WEIGHT_ENUMS[config["architecture"]]
    if urlparse(weights.url).scheme != "https" or urlparse(weights.url).hostname != "download.pytorch.org":
        raise ValueError("Expected an official HTTPS Torchvision weight URL.")
    destination.parent.mkdir(parents=True, exist_ok=True)
    temporary = None
    try:
        with tempfile.NamedTemporaryFile(dir=destination.parent, delete=False) as stream:
            temporary = Path(stream.name)
        torch.hub.download_url_to_file(weights.url, str(temporary),
                                       hash_prefix=expected.lower(), progress=True)
        actual = file_sha256(temporary)
        if actual != expected.lower():
            raise ValueError("Downloaded file checksum mismatch.")
        os.link(temporary, destination)  # atomic no-overwrite publication on WSL
    finally:
        if temporary is not None:
            temporary.unlink(missing_ok=True)
    return {"url": weights.url, "identifier": PRETRAINED_IDS[config["architecture"]],
            "sha256": actual, "path": str(destination)}


In [ ]:
acquisition_info = []
if RUN_PRETRAINED_ACQUISITION:
    for planned in EXPERIMENT_PLAN:
        acquisition_info.append(acquire_official_weights(validate_config(planned)))
else:
    print("Official acquisition disabled; prepared artifacts are already outside Git.")
local_state, initialization_info, initialization_fingerprint = None, None, None
PRETRAINED_READINESS = []
if PREPARE_RUN:
    # Validate both artifacts before an overnight sequence, without retaining duplicate CPU state.
    for planned in EXPERIMENT_PLAN:
        state, info = verified_backbone_state(validate_config(planned))
        del state
        PRETRAINED_READINESS.append(info)
    print(json.dumps(PRETRAINED_READINESS, indent=2))
    print("Both files verified. Each run rechecks and strictly loads its own artifact before construction.")
else:
    print("Pretrained preflight disabled; no file loading or download on default Run All.")


## 9. Loss, staged AdamW and scheduling
Use equal weighted crop/disease raw-logit cross-entropy. Stage A freezes the pretrained backbone and keeps it in eval mode while training heads.
Stage B unfreezes all features. Defaults use backbone LR 1e-5 and head LR 1e-4, AdamW decay 0.01, with a shared cosine multiplier ending at 1% of initial LR. Warm-up heads use LR 1e-3.
These are reviewed starting references, not tuned accuracy guarantees. The actual group LRs, completed stages and schedule states are recorded.


In [ ]:
def _validate_weight(value: float, name: str) -> float:
    if isinstance(value, bool) or not isinstance(value, Real):
        raise TypeError(f"{name} must be a real number")
    weight = float(value)
    if not math.isfinite(weight) or weight < 0:
        raise ValueError(f"{name} must be finite and nonnegative")
    return weight

def _classification_loss(logits: Tensor, labels: Tensor, classes: int, name: str) -> Tensor:
    if not isinstance(logits, Tensor) or not logits.is_floating_point():
        raise TypeError(f"{name} logits must be floating-point tensors")
    if logits.ndim != 2 or logits.shape[0] == 0 or logits.shape[1] != classes:
        raise ValueError(f"{name} logits must have shape [B, {classes}] with B > 0")
    if not isinstance(labels, Tensor) or labels.dtype != torch.int64:
        raise TypeError(f"{name} labels must be int64 tensors")
    if labels.ndim != 1 or labels.shape[0] != logits.shape[0]:
        raise ValueError(f"{name} labels must have shape [B] matching logits")
    if labels.device != logits.device:
        raise ValueError(f"{name} labels and logits must be on the same device")
    if ((labels < 0) | (labels >= classes)).any().item():
        raise ValueError(f"{name} labels must be in [0, {classes})")
    if not torch.isfinite(logits).all().item():
        raise ValueError(f"{name} logits must be finite")
    # Keep CE/log-sum-exp in float32 under AMP; preserve float64 engineering inputs.
    stable_logits = logits.float() if logits.dtype in (torch.float16, torch.bfloat16) else logits
    with torch.autocast(device_type=logits.device.type, enabled=False):
        return F.cross_entropy(stable_logits, labels, reduction="mean")

def compute_multitask_loss(
    outputs: Mapping[str, Tensor],
    targets: Mapping[str, Tensor],
    *,
    crop_weight: float = 0.5,
    disease_weight: float = 0.5,
) -> dict[str, Tensor]:
    """Return total/crop/disease losses from raw logits and int64 component IDs.

    Weights need not sum to one, but must be finite, nonnegative and not both
    zero. Extra output/target fields, including composition IDs, are ignored.
    """
    crop_weight = _validate_weight(crop_weight, "crop_weight")
    disease_weight = _validate_weight(disease_weight, "disease_weight")
    if crop_weight == disease_weight == 0:
        raise ValueError("At least one loss weight must be positive")
    if not isinstance(outputs, Mapping) or not isinstance(targets, Mapping):
        raise TypeError("outputs and targets must be mappings")
    losses = {}
    for name, classes in (("crop", 55), ("disease", 175)):
        if f"{name}_logits" not in outputs or name not in targets:
            raise ValueError(f"Missing {name} logits or labels")
        losses[name] = _classification_loss(
            outputs[f"{name}_logits"], targets[name], classes, name
        )
    crop_logits, disease_logits = outputs["crop_logits"], outputs["disease_logits"]
    if (crop_logits.shape[0] != disease_logits.shape[0]
            or crop_logits.device != disease_logits.device):
        raise ValueError("Crop and disease logits require matching batches and devices")
    total = crop_weight * losses["crop"] + disease_weight * losses["disease"]
    if not all(torch.isfinite(value).item() for value in (*losses.values(), total)):
        raise ValueError("Loss calculation produced a non-finite value")
    return {"total": total, **losses}


In [ ]:
def make_optimizer(model, config):
    validate_config(config)
    optimizer = torch.optim.AdamW(model.parameters(), lr=config["learning_rate"],
                                  weight_decay=config["weight_decay"])
    return optimizer

print("Common optimizer: AdamW")
print("Learning rate:", CONFIG["learning_rate"], "| weight decay:", CONFIG["weight_decay"])
print("Task weights:", CONFIG["crop_weight"], CONFIG["disease_weight"])


## 10. AMP, exact accumulation and visible training
The visible loop uses PyTorch autocast and FP16 GradScaler where appropriate; BF16 uses no scaling. Gradients remain scaled for the entire accumulation window, are unscaled once, checked/clipped, then stepped.
Each microbatch contributes its sample fraction of the actual window. Short final batches/windows are not underweighted. BatchNorm still sees physical microbatches, so accumulation does not make its statistics equivalent to a large physical batch.
Head warm-up keeps backbone BN/statistics/stochastic layers in eval mode. Full fine-tuning restores train behavior. Transfers use pinned memory/nonblocking copies; channels-last is configurable.
No loop executes on definition. Section 13 requires all explicit owner controls. Failures do not commit a partial epoch.

The effective batch target is sample-exact, even when a physical batch does not divide it: leftover samples are carried into the next window. FP16/BF16 logits are explicitly promoted for float32 cross-entropy outside autocast; FP64 engineering inputs remain FP64. Early stopping counts only strict primary H@1 improvements during full fine-tuning, not loss-only tie-break improvements.


In [ ]:
def _epoch_setup(
    model: torch.nn.Module,
    loader: DataLoader,
    device: str | torch.device,
    crop_weight: float,
    disease_weight: float,
) -> tuple[torch.device, float, float]:
    if not isinstance(model, torch.nn.Module):
        raise TypeError("model must be a torch.nn.Module")
    if not isinstance(loader, DataLoader):
        raise TypeError("loader must be a DataLoader")
    crop_weight = _validate_weight(crop_weight, "crop_weight")
    disease_weight = _validate_weight(disease_weight, "disease_weight")
    if crop_weight == disease_weight == 0:
        raise ValueError("At least one loss weight must be positive")
    device = torch.device(device)
    if device.type == "cpu":
        device = torch.device("cpu")
    elif device.type == "cuda" and device.index is None:
        device = torch.device("cuda", torch.cuda.current_device())
    for value in (*model.parameters(), *model.buffers()):
        if value.device != device:
            raise ValueError("Move the model to the requested device before calling the epoch function")
    return device, crop_weight, disease_weight

def _epoch_batch(
    batch: Any, device: torch.device, *, require_composition: bool,
) -> tuple[Tensor, dict[str, Tensor]]:
    if not isinstance(batch, (tuple, list)) or len(batch) != 2:
        raise ValueError("Each batch must contain (inputs, targets)")
    inputs, raw_targets = batch
    if (not isinstance(inputs, Tensor) or not inputs.is_floating_point()
            or inputs.ndim < 2 or inputs.shape[0] == 0):
        raise ValueError("Inputs must be a nonempty floating-point batched tensor")
    if not torch.isfinite(inputs).all().item():
        raise ValueError("Inputs must be finite")
    if isinstance(raw_targets, Mapping):
        targets = {key: raw_targets[key] for key in ("crop", "disease", "composition")
                   if key in raw_targets}
    elif isinstance(raw_targets, (tuple, list)) and len(raw_targets) in (2, 3):
        targets = dict(zip(("crop", "disease", "composition"), raw_targets))
    else:
        raise TypeError("Targets must be a mapping or a crop/disease[/composition] sequence")
    required = ("crop", "disease", "composition") if require_composition else ("crop", "disease")
    if any(key not in targets for key in required):
        raise ValueError(f"Targets require {', '.join(required)}")
    for name, value in targets.items():
        if not isinstance(value, Tensor) or value.dtype != torch.int64:
            raise TypeError(f"{name} targets must be int64 tensors")
        if value.ndim != 1 or value.shape[0] != inputs.shape[0]:
            raise ValueError(f"{name} targets must have shape [B] matching inputs")
        if name == "composition" and (value < 0).any().item():
            raise ValueError("Global composition IDs must be nonnegative")
    return inputs.to(device, non_blocking=device.type == "cuda"), {key: value.to(device, non_blocking=device.type == "cuda") for key, value in targets.items()}

def _epoch_summary(
    totals: dict[str, float], sample_count: int, batch_count: int,
) -> dict[str, Any]:
    if sample_count == 0:
        raise ValueError("Loader produced no samples; check empty data and drop_last")
    means = {key: value / sample_count for key, value in totals.items()}
    if not all(math.isfinite(value) for value in means.values()):
        raise ValueError("Epoch aggregation produced non-finite losses")
    return {
        "sample_count": sample_count,
        "batch_count": batch_count,
        "losses": means,
    }


In [ ]:

def split_cpu_batch(batch, start, stop):
    inputs, targets=batch
    if isinstance(targets,Mapping):
        labels={key:value[start:stop] for key,value in targets.items()}
    else:
        labels=tuple(value[start:stop] for value in targets)
    return inputs[start:stop],labels

def accumulation_windows(loader, accumulation_steps, effective_batch_size=None):
    iterator=iter(loader)
    if effective_batch_size is None:
        while True:
            window=list(itertools.islice(iterator,accumulation_steps))
            if not window:return
            yield window
    else:
        if type(effective_batch_size) is not int or effective_batch_size<1:
            raise ValueError("effective_batch_size must be positive")
        pending=None
        while True:
            window=[];remaining=effective_batch_size
            while remaining:
                if pending is None:
                    try:batch=next(iterator)
                    except StopIteration:break
                else:batch=pending;pending=None
                count=batch[0].shape[0]
                take=min(count,remaining)
                window.append(split_cpu_batch(batch,0,take))
                if take<count:pending=split_cpu_batch(batch,take,count)
                remaining-=take
            if not window:return
            yield window

def measure_loader_pipeline(config, split, image_directory, workers):
    cfg={**config,"num_workers":workers}
    resolution=cfg["final_resolution"]
    # At most six batches from a fitting-only subset; no model or optimizer.
    frame=split.fit.iloc[:min(256,6*cfg["batch_size"])]
    subset=DevelopmentSplit(frame,split.val_seen.iloc[:1],split.val_unseen.iloc[:1],split.mappings,
                            split.candidate_ids,split.candidate_pairs,split.source_csv,split.source_sha256,
                            split.candidate_sha256,split.seed,split.heldout_fraction,split.seen_fraction)
    loaders=create_stage_loaders(subset,image_directory,cfg,resolution)
    pipeline_started=time.perf_counter()
    times=[];samples=[];iterator=iter(loaders[0])
    try:
        for index in range(6):
            started=time.perf_counter()
            try:inputs,labels=next(iterator)
            except StopIteration:break
            times.append(time.perf_counter()-started);samples.append(len(inputs))
        if len(times)<2:raise ValueError("Insufficient pipeline batches")
        return {"architecture":cfg["architecture"],"resolution":resolution,"workers":workers,
                "microbatch":cfg["batch_size"],"pin_memory":cfg["pin_memory"],
                "persistent_workers":workers>0 and cfg["persistent_workers"],
                "prefetch_factor":cfg["prefetch_factor"] if workers>0 else None,
                "first_batch_seconds":times[0],"post_first_queue_wait_seconds":sum(times[1:])/len(times[1:]),
                "overall_delivery_images_per_second":sum(samples)/(time.perf_counter()-pipeline_started),"batches":len(times),
                "scope":"DataLoader fetch only, not end-to-end GPU throughput"}
    finally:shutdown_loaders(loaders)

def amp_context(device, precision):
    device = torch.device(device)
    if precision == "fp32":
        return nullcontext()
    if precision not in ("fp16", "bf16"):
        raise ValueError("Unknown precision")
    if precision == "fp16" and device.type != "cuda":
        raise ValueError("FP16 training requires CUDA")
    if precision == "bf16" and device.type == "cuda" and not torch.cuda.is_bf16_supported():
        raise ValueError("CUDA BF16 is unsupported")
    return torch.autocast(device_type=device.type,
                          dtype=torch.float16 if precision == "fp16" else torch.bfloat16)

def make_scaler(device, precision):
    return torch.amp.GradScaler("cuda", enabled=torch.device(device).type == "cuda" and precision == "fp16",
                               init_scale=1024.0)

def optimizer_update(model, optimizer, scaler, clip_grad_norm=None):
    scaler.unscale_(optimizer)  # once, after the complete accumulation window
    parameters = [p for p in model.parameters() if p.grad is not None]
    if not parameters:
        raise FloatingPointError("No gradients were produced")
    try:
        norm = torch.nn.utils.clip_grad_norm_(parameters, clip_grad_norm or float("inf"),
                                              error_if_nonfinite=True, foreach=True)
    except RuntimeError as error:
        optimizer.zero_grad(set_to_none=True)
        raise FloatingPointError("Non-finite unscaled gradient norm; no optimizer step") from error
    scaler.step(optimizer)
    scaler.update()
    return float(norm)

def create_stage_loaders(split, image_directory, config, resolution, generator_states=None):
    loaders=[]
    plan=fitting_sampling_plan(split,config)
    for index,frame in enumerate((split.fit,split.val_seen,split.val_unseen)):
        dataset=DPDataset(frame,image_directory,image_size=resolution,
                          augment=config["augment_fit"] and index==0)
        generator=torch.Generator().manual_seed(config["seed"]+index)
        if generator_states is not None:generator.set_state(generator_states[index])
        sampler=(CappedDiseaseTicketSampler(plan,generator=generator)
                 if index==0 and plan["mode"]=="capped_disease_tickets" else None)
        arguments=dict(batch_size=config["batch_size"],shuffle=index==0 and sampler is None,
                       sampler=sampler,num_workers=config["num_workers"],pin_memory=config["pin_memory"],
                       worker_init_fn=seed_worker,generator=generator)
        if config["num_workers"]>0:
            arguments.update(prefetch_factor=config["prefetch_factor"],
                             persistent_workers=config["persistent_workers"])
        loader=DataLoader(dataset,**arguments)
        if index==0:loader.fitting_sampling_plan=plan
        loaders.append(loader)
    return tuple(loaders)

def stage_for_epoch(config, epoch):
    if epoch < 1 or epoch > config["warmup_epochs"] + config["finetune_epochs"]:
        raise ValueError("Epoch is outside this finite experiment budget")
    if epoch <= config["warmup_epochs"]:
        return "warmup", epoch, config["start_resolution"]
    fine_epoch = epoch - config["warmup_epochs"]
    resolution = (config["start_resolution"] if fine_epoch <= config["progressive_epochs"]
                  else config["final_resolution"])
    return "finetune", fine_epoch, resolution

def configure_training_stage(model, config, stage):
    if stage not in ("warmup", "finetune"):
        raise ValueError("Unknown stage")
    for parameter in model.backbone.parameters():
        parameter.requires_grad_(stage == "finetune")
    for head in (model.crop_head, model.disease_head):
        for parameter in head.parameters():
            parameter.requires_grad_(True)
    model.gradient_checkpointing = bool(config["gradient_checkpointing"] and stage == "finetune")
    heads = list(model.crop_head.parameters()) + list(model.disease_head.parameters())
    if stage == "warmup":
        groups = [{"params": heads, "lr": config["warmup_head_lr"], "name": "heads"}]
    else:
        groups = [{"params": model.backbone.parameters(), "lr": config["backbone_lr"], "name": "backbone"},
                  {"params": heads, "lr": config["head_lr"], "name": "heads"}]
    optimizer = torch.optim.AdamW(groups, weight_decay=config["weight_decay"])
    scheduler = None
    if stage == "finetune":
        # Same cosine multiplier for both parameter groups, preserving the LR ratio.
        budget = config["finetune_epochs"]
        scheduler = torch.optim.lr_scheduler.LambdaLR(
            optimizer, lr_lambda=lambda completed: 0.01 + 0.99 * (1 + math.cos(math.pi * min(completed, budget) / budget)) / 2)
    return optimizer, scheduler

def update_early_stopping(state, harmonic, config, *, stage="finetune"):
    """Strict global H@1 reference; only fine-tuning consumes patience/grace.

    Warm-up sets the global reference and never consumes bad epochs.
    An exact H@1 tie (including a loss-only checkpoint improvement) is bad
    during fine-tuning. Only a strictly greater H@1 resets patience.
    """
    if isinstance(harmonic,bool) or not isinstance(harmonic,Real) or not math.isfinite(harmonic) or not 0<=harmonic<=1:
        raise ValueError("H@1 must be a finite fraction.")
    if stage not in ("warmup","finetune"):
        raise ValueError("Unknown early-stopping stage.")
    result=dict(state)
    if result.get("reference","global_best_h1")!="global_best_h1":
        raise ValueError("Do not silently reinterpret a legacy fine-only stopping state.")
    result["reference"]="global_best_h1"
    improved=harmonic>result.get("best_h1",-1.)
    result["best_h1"]=float(harmonic) if improved else result["best_h1"]
    result.setdefault("bad_epochs",0);result.setdefault("fine_epochs",0)
    if stage=="warmup":
        result["bad_epochs"]=0
        return result,False
    result["bad_epochs"]=0 if improved else result["bad_epochs"]+1
    result["fine_epochs"]+=1
    stop=(result["fine_epochs"]>=config["min_finetune_epochs"]
          and result["bad_epochs"]>=config["early_stopping_patience"])
    return result,stop


In [ ]:
def train_one_epoch(model, fitting_loader, optimizer, device, *, crop_weight=0.5,
                    disease_weight=0.5, log_every=10, precision="fp32",
                    accumulation_steps=1, scaler=None, channels_last=False,
                    clip_grad_norm=None, frozen_backbone=False, effective_batch_size=None):
    device, crop_weight, disease_weight = _epoch_setup(
        model, fitting_loader, device, crop_weight, disease_weight)
    if torch.is_inference_mode_enabled():
        raise ValueError("Training cannot run inside inference_mode")
    if type(accumulation_steps) is not int or accumulation_steps < 1:
        raise ValueError("accumulation_steps must be positive")
    if not isinstance(optimizer, torch.optim.Optimizer):
        raise TypeError("optimizer must be supplied explicitly")
    scaler = make_scaler(device, precision) if scaler is None else scaler
    model.train()
    if frozen_backbone:
        model.backbone.eval()  # preserve pretrained BN buffers/stochastic behavior during head warm-up
    totals = {key: 0.0 for key in ("total", "crop", "disease")}
    sample_count = batch_count = updates = 0
    started = time.perf_counter()
    with torch.enable_grad():
        for window in accumulation_windows(fitting_loader, accumulation_steps, effective_batch_size):
            window_samples = sum(batch[0].shape[0] for batch in window)
            model.zero_grad(set_to_none=True)
            for batch in window:
                inputs, targets = _epoch_batch(batch, device, require_composition=False)
                if channels_last and inputs.ndim == 4:
                    inputs = inputs.contiguous(memory_format=torch.channels_last)
                with amp_context(device, precision):
                    losses = compute_multitask_loss(model(inputs), targets,
                                                   crop_weight=crop_weight, disease_weight=disease_weight)
                    # Exact sample weighting handles short microbatches and the final partial window.
                    scaled_objective = losses["total"] * (inputs.shape[0] / window_samples)
                scaler.scale(scaled_objective).backward()
                for key, value in losses.items():
                    totals[key] += float(value.detach()) * inputs.shape[0]
                sample_count += inputs.shape[0]
                batch_count += 1
            optimizer_update(model, optimizer, scaler, clip_grad_norm)
            updates += 1
            if updates == 1 or updates % log_every == 0 or sample_count == len(fitting_loader.dataset):
                memory = torch.cuda.memory_allocated(device) / 2**20 if device.type == "cuda" else 0
                print(f"images {sample_count}/{len(fitting_loader.dataset)} microbatches {batch_count} update {updates} "
                      f"total={totals['total']/sample_count:.5f} crop={totals['crop']/sample_count:.5f} "
                      f"disease={totals['disease']/sample_count:.5f} "
                      f"elapsed={time.perf_counter()-started:.1f}s CUDA={memory:.0f} MiB", flush=True)
    summary = _epoch_summary(totals, sample_count, batch_count)
    summary.update(optimizer_steps=updates, seconds=time.perf_counter()-started)
    summary["images_per_second"] = sample_count / summary["seconds"]
    return summary


## 11. Seen/unseen development validation
Both folds use eval/inference mode, the epoch's explicit resolution, AMP policy and the same 323 ordered global candidate IDs. Return actual sample-weighted losses, detached CPU logits, global truths and counts.
No official test loader is created. Bounded preparation checks a small number of real images for finite behavior, not full validation or architecture selection.


In [ ]:
def validate_one_epoch(
    model: torch.nn.Module,
    validation_loader: DataLoader,
    device: str | torch.device,
    *,
    crop_weight: float = 0.5,
    disease_weight: float = 0.5,
    precision: str = "fp32",
    channels_last: bool = False,
) -> dict[str, Any]:
    """Evaluate once, returning sample-weighted losses and CPU predictions.

    Sets eval mode and uses inference_mode without clearing gradients or
    stepping an optimizer. CPU logits and int64 composition_ids preserve loader
    iteration order. Future WP5 callers supply the shared candidate IDs/pairs;
    no composition scoring, Top-k or harmonic mean is implemented here.
    """
    device, crop_weight, disease_weight = _epoch_setup(
        model, validation_loader, device, crop_weight, disease_weight
    )
    model.eval()
    totals = {key: 0.0 for key in ("total", "crop", "disease")}
    sample_count = batch_count = 0
    crop_parts, disease_parts, composition_parts = [], [], []
    with torch.inference_mode():
        for batch in validation_loader:
            inputs, targets = _epoch_batch(batch, device, require_composition=True)
            if channels_last and inputs.ndim == 4:
                inputs = inputs.contiguous(memory_format=torch.channels_last)
            with amp_context(device, precision):
                outputs = model(inputs)
                losses = compute_multitask_loss(
                    outputs, targets, crop_weight=crop_weight, disease_weight=disease_weight
                )
            size = inputs.shape[0]
            for key, value in losses.items():
                totals[key] += value.item() * size
            sample_count += size
            batch_count += 1
            # Snapshot each batch even when a model/loader reuses CPU storage.
            crop_parts.append(outputs["crop_logits"].detach().to("cpu", copy=True))
            disease_parts.append(outputs["disease_logits"].detach().to("cpu", copy=True))
            composition_parts.append(targets["composition"].detach().to("cpu", copy=True))
    result = _epoch_summary(totals, sample_count, batch_count)
    result.update(
        crop_logits=torch.cat(crop_parts),
        disease_logits=torch.cat(disease_parts),
        composition_ids=torch.cat(composition_parts),
    )
    return result


## 12. Composition scoring, Top-k and harmonic mean
For candidate column j with pair (c_j,d_j), score_j = log_softmax(crop_logits)[c_j] + log_softmax(disease_logits)[d_j].
The visible implementation uses float64 log-softmax for stable scoring of extreme finite float32 logits. It validates candidate hashes, unique sorted global IDs, pair correspondence and component ranges. Unrepresentable/non-finite scores fail clearly.
Stable descending sorting breaks exact score ties by the smaller global composition ID, because candidate columns are in ascending global-ID order. No epsilon is added.
Top-k compares ranked **global IDs** with original truths. H@k = 2*S@k*U@k/(S@k+U@k), or zero for zero/zero. The equivalent ratio form below avoids product underflow.
These local, independently tested functions represent the agreed WP5 metric contract. They do not modify or replace Jonathan's PR; future integration must verify equivalence. No unavailable adapter blocks this notebook.


In [ ]:
def validate_candidate_spec(spec):
    ids, pairs = spec["ids"], spec["pairs"]
    if not ids or len(ids) != len(pairs):
        raise ValueError("Candidate IDs and component pairs must be nonempty and aligned.")
    if any(isinstance(value, bool) or not isinstance(value, Integral) or value < 0 for value in ids):
        raise ValueError("Global candidate IDs must be nonnegative integers.")
    ids = tuple(int(value) for value in ids)
    if ids != tuple(sorted(set(ids))):
        raise ValueError("Candidates must have unique ascending global IDs for deterministic ties.")
    normalized = []
    for pair in pairs:
        if len(pair) != 2 or any(isinstance(v, bool) or not isinstance(v, Integral) for v in pair):
            raise ValueError("Each candidate requires an integer crop/disease pair.")
        crop, disease = map(int, pair)
        if not 0 <= crop < 55 or not 0 <= disease < 175:
            raise ValueError("Candidate components exceed the 55/175 label spaces.")
        normalized.append((crop, disease))
    if len(set(normalized)) != len(normalized):
        raise ValueError("Distinct global IDs cannot identify the same component pair.")
    records = [[identifier, *pair] for identifier, pair in zip(ids, normalized)]
    checksum = hashlib.sha256(json.dumps(records, separators=(",", ":")).encode()).hexdigest()
    if spec.get("sha256") is not None and spec["sha256"] != checksum:
        raise ValueError("Candidate mapping hash does not match its IDs/pairs.")
    return ids, tuple(normalized), checksum

def composition_scores(crop_logits, disease_logits, candidate_spec):
    ids, pairs, _ = validate_candidate_spec(candidate_spec)
    for name, logits, width in (("crop", crop_logits, 55), ("disease", disease_logits, 175)):
        if (not isinstance(logits, Tensor) or not logits.is_floating_point()
                or logits.ndim != 2 or logits.shape[0] == 0 or logits.shape[1] != width):
            raise ValueError(f"{name} logits must be nonempty floating [N,{width}].")
        if not torch.isfinite(logits).all().item():
            raise ValueError("Component logits must be finite.")
    if crop_logits.shape[0] != disease_logits.shape[0] or crop_logits.device != disease_logits.device:
        raise ValueError("Component batches/devices must match.")
    pair_tensor = torch.tensor(pairs, dtype=torch.int64, device=crop_logits.device)
    crop_log_prob = F.log_softmax(crop_logits.to(torch.float64), dim=1)
    disease_log_prob = F.log_softmax(disease_logits.to(torch.float64), dim=1)
    scores = crop_log_prob[:, pair_tensor[:, 0]] + disease_log_prob[:, pair_tensor[:, 1]]
    if not torch.isfinite(scores).all().item():
        raise ValueError("Log scores exceed representable precision.")
    return scores

def composition_topk(scores, truths, candidate_spec, k):
    ids, _, _ = validate_candidate_spec(candidate_spec)
    if (not isinstance(scores, Tensor) or not scores.is_floating_point() or scores.ndim != 2
            or scores.shape[0] == 0 or scores.shape[1] != len(ids)
            or not torch.isfinite(scores).all().item()):
        raise ValueError("Scores must be finite [N,candidates] with N>0.")
    if type(k) is not int or not 1 <= k <= len(ids):
        raise ValueError("k must be in 1..candidate_count.")
    if (not isinstance(truths, Tensor) or truths.dtype != torch.int64
            or truths.ndim != 1 or truths.shape[0] != scores.shape[0]):
        raise ValueError("Composition truths must be int64 [N].")
    global_ids = torch.tensor(ids, dtype=torch.int64, device=scores.device)
    truths = truths.to(scores.device)
    if not truths[:, None].eq(global_ids[None, :]).any(dim=1).all().item():
        raise ValueError("A global composition truth is outside the candidate universe.")
    order = torch.argsort(scores, dim=1, descending=True, stable=True)
    ranked_ids = global_ids[order[:, :k]]
    return ranked_ids.eq(truths[:, None]).any(dim=1).double().mean().item()

def harmonic_mean(seen_accuracy, unseen_accuracy):
    for value in (seen_accuracy, unseen_accuracy):
        if isinstance(value, bool) or not isinstance(value, Real) or not math.isfinite(value) or not 0 <= value <= 1:
            raise ValueError("Accuracies must be finite fractions in [0,1].")
    low, high = sorted((float(seen_accuracy), float(unseen_accuracy)))
    return 0.0 if high == 0 else 2.0 * low / (1.0 + low / high)

def evaluate_development(seen, unseen, candidate_spec):
    metrics = {}
    for fold, result in (("seen", seen), ("unseen", unseen)):
        if result["sample_count"] != len(result["composition_ids"]):
            raise ValueError("Validation sample count and truths disagree.")
        scores = composition_scores(result["crop_logits"], result["disease_logits"], candidate_spec)
        for k in (1, 5):
            metrics[f"{fold}_top{k}"] = composition_topk(
                scores, result["composition_ids"], candidate_spec, k)
    for k in (1, 5):
        metrics[f"top{k}_harmonic_mean"] = harmonic_mean(
            metrics[f"seen_top{k}"], metrics[f"unseen_top{k}"])
    return metrics


In [ ]:
if RUN_METRIC_EXAMPLES:
    synthetic_candidates = {"ids": (10, 75, 100, 150, 200, 332),
                            "pairs": tuple((i, i) for i in range(6))}
    crop = torch.zeros(2, 55)
    disease = torch.zeros(2, 175)
    scores = composition_scores(crop, disease, synthetic_candidates)
    # Uniform scores: exact ties rank the smaller global ID first.
    truths = torch.tensor([10, 332], dtype=torch.int64)
    top1 = composition_topk(scores, truths, synthetic_candidates, 1)
    top5 = composition_topk(scores, truths, synthetic_candidates, 5)
    assert top1 == 0.5 and top5 == 0.5
    assert harmonic_mean(0.0, 0.0) == 0.0
    assert math.isclose(harmonic_mean(0.5, 0.25), 1.0 / 3.0)
    print("SYNTHETIC DEMONSTRATION ONLY:", {"Top-1": top1, "Top-5": top5,
                                            "H(0.5,0.25)": harmonic_mean(0.5, 0.25)})
else:
    print("Synthetic scoring demonstration disabled; no experimental metric is claimed.")


## 13. Selection, recovery and sequential orchestration
Best-model selection remains exact: highest H@1, lowest sample-weighted combined validation loss, earliest epoch. Portable best checkpoints retain the accepted format-version-1 contract.
A separate completed-epoch recovery envelope stores model/optimizer/scheduler/scaler, Python/NumPy/Torch/CUDA and loader-generator state, history, selection and early-stopping state.
last_completed.pt is the one owned atomic replacement slot, written only after validation, scheduler/selection/logging finish. Immutable best filenames include epoch and attempt suffix, protecting against orphan files from interrupted publication.
Explicit resume requires matching substantive config, source/candidate hashes and executable-code identity, and resumes the next epoch. With workers>0, worker-local augmentation RNG is not captured; exact bitwise continuation is not promised. Use workers=0 for the strongest epoch-boundary RNG reproducibility.
Create STOP_AFTER_EPOCH in the run directory for a safe boundary stop; remove it manually before resuming. Keyboard interruption resumes the last published complete epoch, not partial in-memory state.
Both experiments run sequentially only after owner enablement; an interruption stops the sequence. No infinite budget or automatic full experiment occurs in preparation.

Executable identity excludes saved outputs, Markdown and editable run/configuration controls while retaining processing code and policy constants. This permits autosave and setting resume_from without invalidating recovery; substantive configuration is compared separately and must match. Changing batch size, precision, stage budget or optimizer on resume is rejected rather than silently changing the experiment.


In [ ]:
def combined_validation_loss(seen, unseen):
    counts, losses = [], []
    for summary in (seen, unseen):
        count, loss = summary["sample_count"], summary["losses"]["total"]
        if type(count) is not int or count <= 0 or not math.isfinite(loss):
            raise ValueError("Selection requires actual positive sample counts and finite total losses.")
        counts.append(count)
        losses.append(loss)
    combined = (counts[0] * losses[0] + counts[1] * losses[1]) / sum(counts)
    if not math.isfinite(combined):
        raise ValueError("Combined validation loss must be finite.")
    return combined

def checkpoint_selection_key(metrics, seen, unseen, epoch):
    if metrics is None:
        raise RuntimeError("Missing development metrics: checkpoint selection is disabled.")
    if type(epoch) is not int or epoch < 1:
        raise ValueError("epoch must be a positive integer.")
    harmonic = metrics["top1_harmonic_mean"]
    if not math.isfinite(harmonic) or not 0 <= harmonic <= 1:
        raise ValueError("H@1 must be a finite fraction.")
    # Exact full-precision comparisons: maximize H@1, minimize loss, earliest.
    return (-harmonic, combined_validation_loss(seen, unseen), epoch)


METRIC_IMPLEMENTATION = "notebook_wp5_spec_v1"
SCORING_ID = "sum_log_softmax_v1"
RANKING_TIE_POLICY = "stable_descending_then_ascending_global_ID"

def checkpoint_metadata(run_context, epoch, selection):
    return {
        **run_context["base_metadata"], "epoch": epoch, "selection": selection,
        "metrics": {"implementation": METRIC_IMPLEMENTATION, "scoring_id": SCORING_ID,
                    "ranking_tie_policy": RANKING_TIE_POLICY},
    }


In [ ]:
_FLOAT_DTYPES = {str(dtype): dtype for dtype in
                 (torch.float16, torch.bfloat16, torch.float32, torch.float64)}

def _class_mappings(
    mappings: Mapping[str, Any] | None,
) -> dict[str, dict[int, str]] | None:
    """Normalize component ID/name maps from the existing data utilities."""
    if mappings is None:
        return None
    if not isinstance(mappings, Mapping):
        raise TypeError("class_mappings must be a mapping")
    result = {}
    for name, count in (("crop", 55), ("disease", 175)):
        key = f"{name}_id_to_name"
        values = mappings.get(key)
        if not isinstance(values, Mapping):
            raise ValueError(f"class_mappings requires {key}")
        if (any(isinstance(i, bool) or not isinstance(i, Integral) for i in values)
                or set(values) != set(range(count))):
            raise ValueError(f"{key} requires exactly the IDs 0..{count - 1}")
        names = [values[i] for i in range(count)]
        if (any(not isinstance(value, str) or not value.strip() for value in names)
                or len(set(names)) != count):
            raise ValueError(f"{key} requires unique, nonempty string names")
        result[key] = {i: str(value) for i, value in enumerate(names)}
    return result

def _plain_metadata(value: Mapping[str, Any] | None, name: str) -> dict[str, Any] | None:
    if value is None:
        return None
    if not isinstance(value, Mapping) or any(not isinstance(key, str) for key in value):
        raise TypeError(f"{name} must be a mapping with string keys")
    try:
        return json.loads(json.dumps(dict(value), allow_nan=False))
    except (TypeError, ValueError) as exc:
        raise ValueError(f"{name} must contain finite JSON-compatible values") from exc

def _checkpoint_config(architecture: str) -> dict[str, Any]:
    if not isinstance(architecture, str) or architecture not in MODEL_FEATURE_DIMS:
        raise ValueError(f"Unsupported checkpoint architecture: {architecture!r}")
    return {
        "format_version": 1,
        "architecture": architecture,
        "num_crops": 55,
        "num_diseases": 175,
        "feature_dim": MODEL_FEATURE_DIMS[architecture],
    }

def save_checkpoint(
    model: MultiTaskBackbone,
    path: str | Path,
    *,
    class_mappings: Mapping[str, Any] | None = None,
    preprocessing: Mapping[str, Any] | None = None,
    metadata: Mapping[str, Any] | None = None,
    overwrite: bool = False,
) -> Path:
    """Atomically save CPU state and plain metadata to an existing directory.

    Existing paths are protected unless overwrite=True. Component maps use
    crop_id_to_name and disease_id_to_name; other mapping fields are ignored.
    Optimizer state and exact training resumption are outside this format.
    """
    if not isinstance(model, MultiTaskBackbone):
        raise TypeError("Expected the visible notebook MultiTaskBackbone")
    architecture = model.architecture
    config = _checkpoint_config(architecture)
    dim = config["feature_dim"]
    if type(overwrite) is not bool:
        raise TypeError("overwrite must be a bool")
    if (model.crop_head.in_features, model.crop_head.out_features,
            model.disease_head.in_features, model.disease_head.out_features) != (dim, 55, dim, 175):
        raise ValueError("Model heads are incompatible with the checkpoint architecture")
    payload = {
        **config,
        "class_mappings": _class_mappings(class_mappings),
        "preprocessing": _plain_metadata(preprocessing, "preprocessing"),
        "metadata": _plain_metadata(metadata, "metadata") or {},
        "runtime": {
            "python": platform.python_version(),
            "torch": str(torch.__version__),
            "torchvision": str(torchvision.__version__),
        },
    }
    if payload["preprocessing"] and payload["preprocessing"].get("metadata_version") == 2:
        resolve_checkpoint_preprocessing(payload)
    destination = Path(path)
    if destination.exists() and not overwrite:
        raise FileExistsError(destination)
    state = model.state_dict()
    dtypes = {str(value.dtype) for value in state.values() if value.is_floating_point()}
    if len(dtypes) != 1 or not dtypes <= _FLOAT_DTYPES.keys():
        raise ValueError("Checkpoint requires one supported floating-point state dtype")
    payload["model_dtype"] = next(iter(dtypes))
    payload["model_state_dict"] = {
        name: value.detach().cpu().clone() for name, value in state.items()
    }
    temporary = None
    try:
        with tempfile.NamedTemporaryFile(
            dir=destination.parent, prefix=f".{destination.name}.", delete=False
        ) as stream:
            temporary = Path(stream.name)
            torch.save(payload, stream)
            stream.flush()
            os.fsync(stream.fileno())
        if overwrite:
            os.replace(temporary, destination)
        else:
            # Atomic publication without a check-then-overwrite race on WSL.
            os.link(temporary, destination)
    finally:
        if temporary is not None:
            temporary.unlink(missing_ok=True)
    return destination

def load_checkpoint(
    path: str | Path,
    *,
    expected_class_mappings: Mapping[str, Any] | None = None,
    expected_preprocessing: Mapping[str, Any] | None = None,
    expected_architecture: str | None = None,
    device: str | torch.device = "cpu",
    legacy_policy: str | None = None,
    legacy_resolution: int | None = None,
) -> tuple[MultiTaskBackbone, dict[str, Any]]:
    """Load state strictly, returning an evaluation-mode model and metadata.

    Labelled checkpoints require expected_class_mappings so ID/name changes
    cannot be silently accepted. Preprocessing is compared when supplied.
    Loading is restricted to tensors/plain data; there is no unsafe fallback.
    Version-1 selected-backbone state keys match the preserved reference modules.
    All architectures reconstruct offline; expected_architecture optionally
    guards the caller against loading another supported model by mistake.
    """
    payload = torch.load(Path(path), map_location="cpu", weights_only=True)
    if not isinstance(payload, dict):
        raise ValueError("Checkpoint must contain a dictionary")
    architecture = payload.get("architecture")
    config = _checkpoint_config(architecture)
    if expected_architecture is not None:
        _checkpoint_config(expected_architecture)
        if architecture != expected_architecture:
            raise ValueError("Checkpoint architecture does not match expected architecture")
    for key, expected in config.items():
        value = payload.get(key)
        if type(value) is not type(expected) or value != expected:
            raise ValueError(f"Incompatible checkpoint {key}: {value!r}")
    required = {"model_state_dict", "model_dtype", "class_mappings",
                "preprocessing", "metadata", "runtime"}
    if not required <= payload.keys():
        raise ValueError("Checkpoint is missing required state or metadata")
    dtype = payload["model_dtype"]
    if not isinstance(dtype, str) or dtype not in _FLOAT_DTYPES:
        raise ValueError("Unsupported checkpoint model_dtype")
    stored_mappings = _class_mappings(payload["class_mappings"])
    expected_mappings = _class_mappings(expected_class_mappings)
    if stored_mappings is not None and expected_mappings is None:
        raise ValueError("Labelled checkpoints require expected_class_mappings")
    if stored_mappings != expected_mappings:
        raise ValueError("Checkpoint class mappings do not match expected class IDs/names")
    preprocessing = _plain_metadata(payload["preprocessing"], "preprocessing")
    if (expected_preprocessing is not None
            and preprocessing != _plain_metadata(expected_preprocessing, "expected_preprocessing")):
        raise ValueError("Checkpoint preprocessing does not match expected preprocessing")
    info = {
        **payload,
        "class_mappings": stored_mappings,
        "preprocessing": preprocessing,
        "metadata": _plain_metadata(payload["metadata"], "metadata"),
        "runtime": _plain_metadata(payload["runtime"], "runtime"),
    }
    if preprocessing and preprocessing.get("metadata_version") == 2:
        info["inference_preprocessing"] = resolve_checkpoint_preprocessing(info)
        info["preprocessing_resolution_status"] = "epoch_specific_v2"
    elif legacy_policy is not None:
        info["inference_preprocessing"] = resolve_checkpoint_preprocessing(info, legacy_policy=legacy_policy, legacy_resolution=legacy_resolution)
        info["preprocessing_resolution_status"] = "legacy_explicit"
    else:
        info["preprocessing_resolution_status"] = "legacy_unresolved"
    state = info.pop("model_state_dict")
    if (not isinstance(state, Mapping)
            or any(not isinstance(value, Tensor) for value in state.values())):
        raise ValueError("Checkpoint model_state_dict must map names to tensors")
    if any(value.is_floating_point() and str(value.dtype) != dtype for value in state.values()):
        raise ValueError("Checkpoint state dtype does not match model_dtype")
    model = create_multitask_model(architecture).to(dtype=_FLOAT_DTYPES[dtype])
    model.load_state_dict(state, strict=True)
    model.to(device)
    model.eval()
    return model, info


In [ ]:

def notebook_code_identity(path):
    notebook=json.loads(Path(path).read_text(encoding="utf-8"))
    records=[]
    for cell in notebook["cells"]:
        if cell["cell_type"]!="code":continue
        tree=ast.parse("".join(cell["source"]))
        if cell["id"]=="configuration":
            # Controls/run plan are validated separately against saved configuration.
            # Retain validation functions and policy constants in executable identity.
            keep={"APPROVED_ARCHITECTURES","PRETRAINED_IDS","SELECTION_POLICY"}
            tree.body=[node for node in tree.body if isinstance(node,(ast.FunctionDef,ast.ClassDef))
                       or (isinstance(node,ast.Assign) and any(isinstance(target,ast.Name)
                           and target.id in keep for target in node.targets))]
        records.append({"id":cell["id"],"ast":ast.dump(tree,include_attributes=False)})
    return hashlib.sha256(json.dumps(records,sort_keys=True,separators=(",",":")).encode()).hexdigest()

def compatible_training_config(config):
    # Location/resume controls do not change optimization; every substantive setting must match.
    return {key: value for key, value in config.items()
            if key not in ("resume_from", "run_id", "output_directory")}

def capture_rng_state(loaders=None):
    numpy_state = np.random.get_state()
    return {
        "python": random.getstate(), "torch_cpu": torch.get_rng_state(),
        "numpy": {"algorithm": numpy_state[0], "keys": numpy_state[1].tolist(),
                  "position": numpy_state[2], "has_gauss": numpy_state[3],
                  "cached_gaussian": numpy_state[4]},
        "cuda": torch.cuda.get_rng_state_all() if torch.cuda.is_available() else [],
        "loaders": [loader.generator.get_state() for loader in loaders] if loaders else [],
    }

def restore_rng_state(state, loaders=None):
    random.setstate(state["python"])
    numpy_state = state["numpy"]
    np.random.set_state((numpy_state["algorithm"], np.array(numpy_state["keys"], dtype=np.uint32),
                         numpy_state["position"], numpy_state["has_gauss"], numpy_state["cached_gaussian"]))
    torch.set_rng_state(state["torch_cpu"])
    if state["cuda"] and torch.cuda.is_available():
        torch.cuda.set_rng_state_all(state["cuda"])
    if loaders:
        for loader, generator_state in zip(loaders, state["loaders"]):
            loader.generator.set_state(generator_state)

def atomic_torch_save(payload, path, *, overwrite=False):
    destination = Path(path)
    if destination.exists() and not overwrite:
        raise FileExistsError(destination)
    temporary = None
    try:
        with tempfile.NamedTemporaryFile(dir=destination.parent, prefix="."+destination.name+".",
                                         delete=False) as stream:
            temporary = Path(stream.name)
            torch.save(payload, stream)
            stream.flush()
            os.fsync(stream.fileno())
        if overwrite:
            os.replace(temporary, destination)
        else:
            os.link(temporary, destination)
    finally:
        if temporary is not None:
            temporary.unlink(missing_ok=True)
    return destination

def save_recovery_checkpoint(context, epoch, stage, resolution, scheduler, scaler, early_state):
    # Separate recovery envelope; portable best-model checkpoints retain the accepted v1 contract.
    payload = {
        "recovery_format_version": 1, "architecture": context["config"]["architecture"],
        "configuration": context["config"], "completed_epoch": epoch, "optimizer_stage": stage,
        "completed_resolution": resolution, "model_state_dict": context["model"].state_dict(),
        "preprocessing": {**model_preprocessing_metadata(context["config"], epoch=epoch),
            "actual_selected_checkpoint_resolution": (context["selected"].get("resolution") if context["selected"] else None)},
        "early_stopping_reference": "global_best_h1",
        "optimizer_state_dict": context["optimizer"].state_dict(),
        "scheduler_state_dict": None if scheduler is None else scheduler.state_dict(),
        "scaler_state_dict": scaler.state_dict(), "rng_state": capture_rng_state(context["loaders"]),
        "history": context["history"], "selected": context["selected"], "early_state": early_state,
        "base_metadata": context["base_metadata"], "run_directory": str(context["directory"]),
        "reproducibility": ("epoch-boundary RNG/order; worker augmentation state is not captured"
                            if context["config"]["num_workers"] > 0
                            else "epoch-boundary Python/NumPy/Torch/CUDA/loader RNG captured"),
    }
    # This one owned publication slot is explicitly replaced only after a complete epoch.
    return atomic_torch_save(payload, context["directory"]/"last_completed.pt", overwrite=True)

def read_recovery_checkpoint(path, config, split, notebook_sha256):
    payload = torch.load(Path(path), map_location="cpu", weights_only=True)
    if payload.get("recovery_format_version") != 1:
        raise ValueError("Unsupported recovery format")
    if compatible_training_config(payload["configuration"]) != compatible_training_config(config):
        raise ValueError("Recovery configuration differs; do not silently change budgets/precision/optimization.")
    provenance = payload["base_metadata"]
    if (provenance["split"]["source_sha256"] != split.source_sha256
            or provenance["split"]["candidate_sha256"] != split.candidate_sha256):
        raise ValueError("Recovery source/candidate mapping changed")
    if provenance["code"]["executable_code_sha256"] != notebook_sha256:
        raise ValueError("Saved notebook code differs; restore the reviewed notebook before exact recovery.")
    if payload["architecture"] != config["architecture"]:
        raise ValueError("Recovery architecture mismatch")
    if not 0 < payload["completed_epoch"] <= config["warmup_epochs"] + config["finetune_epochs"]:
        raise ValueError("Invalid completed epoch")
    if payload.get("preprocessing", {}).get("metadata_version") == 2:
        payload["inference_preprocessing"] = resolve_checkpoint_preprocessing(payload)
    return payload

def shutdown_loaders(loaders):
    # Release workers at resolution changes and between sequential experiments.
    for loader in loaders or ():
        iterator = getattr(loader, "_iterator", None)
        if iterator is not None:
            iterator._shutdown_workers()
            loader._iterator = None

def model_preprocessing_metadata(config, *, epoch=None, selected=False):
    """Separate the run plan from authoritative metadata for one saved epoch."""
    schedule={"warmup_resolution":config["start_resolution"],
              "progressive_finetune_resolution":config["start_resolution"],
              "final_planned_resolution":config["final_resolution"],
              "warmup_epochs":config["warmup_epochs"],
              "progressive_finetune_epochs":config["progressive_epochs"]}
    common={"metadata_version":2,"policy":config["preprocessing_policy"],
            **schedule,"normalization_shared":True}
    if epoch is None:
        return {**common,"scope":"run_plan","actual_resolution":None,
                "actual_selected_checkpoint_resolution":None}
    if type(epoch) is not int:
        raise ValueError("An actual checkpoint requires an integer epoch.")
    stage,stage_epoch,resolution=stage_for_epoch(config,epoch)
    phase=("warmup" if stage=="warmup" else
           "progressive_finetune" if stage_epoch<=config["progressive_epochs"] else "final_finetune")
    return {**preprocessing_record({**config,"active_resolution":resolution}),**common,
            "scope":"epoch_checkpoint","epoch":epoch,"stage":phase,"stage_epoch":stage_epoch,
            "actual_resolution":resolution,
            "actual_selected_checkpoint_resolution":resolution if selected else None}

def run_high_performance_experiment(config, split, image_directory, root):
    cfg = validate_config(config)
    if cfg["initialization"] != "local_pretrained":
        raise ValueError("Final B3/Base experiments require verified ImageNet pretrained initialization.")
    if cfg["device"].startswith("cuda") and not torch.cuda.is_available():
        raise RuntimeError("Requested CUDA device is unavailable")
    device = torch.device(cfg["device"])
    notebook_sha = notebook_code_identity(root/"notebooks/wp2_backbone_comparison.ipynb")
    seed_everything(cfg["seed"])
    torch.backends.cudnn.benchmark = not cfg["deterministic_algorithms"]
    torch.backends.cudnn.deterministic = cfg["deterministic_algorithms"]
    torch.use_deterministic_algorithms(cfg["deterministic_algorithms"])
    resume = None
    if cfg["resume_from"]:
        resume = read_recovery_checkpoint(cfg["resume_from"], cfg, split, notebook_sha)
        model = create_multitask_model(cfg["architecture"])  # full recovery state, no pretrained fallback
        model.load_state_dict(resume["model_state_dict"], strict=True)
        initialization = resume["base_metadata"]["initialization"]
    else:
        state, initialization = verified_backbone_state(cfg)
        initialization["source_note"] = cfg["weight_source_note"]
        model = create_multitask_model(cfg["architecture"], backbone_state_dict=state)
        del state
    initial_sha = (d_state_digest(model.state_dict()) if cfg.get("experiment_kind") == "m5e_pilot" else None)
    model.to(device)
    if cfg["channels_last"]:
        model.to(memory_format=torch.channels_last)
    scaler = make_scaler(device, cfg["precision"])
    start_epoch = 1 if resume is None else resume["completed_epoch"] + 1
    current_stage = None if resume is None else resume["optimizer_stage"]
    scheduler = None
    optimizer = None
    if resume is not None:
        optimizer, scheduler = configure_training_stage(model, cfg, current_stage)
        optimizer.load_state_dict(resume["optimizer_state_dict"])
        if scheduler is not None:
            scheduler.load_state_dict(resume["scheduler_state_dict"])
        scaler.load_state_dict(resume["scaler_state_dict"])
        directory = Path(resume["run_directory"])
        if Path(cfg["resume_from"]).resolve().parent != directory.resolve():
            raise ValueError("Resume from the original owned run directory; do not fork paths silently.")
        history, selected = resume["history"], resume["selected"]
        base = resume["base_metadata"]
        early_state = resume["early_state"]
    else:
        identifier = cfg["run_id"] or (
            datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%SZ")+"_"+cfg["architecture"]+"_"+uuid.uuid4().hex[:8])
        directory = reserve_run_directory(cfg["output_directory"], identifier)
        membership = {name:getattr(split,name).image_name.tolist() for name in ("fit","val_seen","val_unseen")}
        base = {
            "run_id":identifier,"architecture":cfg["architecture"],"configuration":cfg,
            "initialization":initialization,"preprocessing":model_preprocessing_metadata(cfg),
            "initial_model_sha256":initial_sha,"initialization_verified":True,
            "fitting_sampling":fitting_sampling_plan(split,cfg),
            "early_stopping":{"reference":"global_best_h1","patience":cfg["early_stopping_patience"],
                "minimum_finetune_grace":cfg["min_finetune_epochs"],"warmup_counts_toward_patience":False,
                "exact_h1_ties_reset":False,"loss_only_improvements_reset":False},
            "split":{"source_sha256":split.source_sha256,"candidate_sha256":split.candidate_sha256,
                     "split_sha256":hashlib.sha256(json.dumps(membership,sort_keys=True,separators=(",",":")).encode()).hexdigest(),
                     "candidate_ids":split.candidate_ids,"candidate_pairs":split.candidate_pairs},
            "selection_policy":SELECTION_POLICY,"environment":ENVIRONMENT,
            "code":{"executable_code_sha256":notebook_sha,
                    "notebook_file_sha256":file_sha256(root/"notebooks/wp2_backbone_comparison.ipynb")},"metric_implementation":METRIC_IMPLEMENTATION,
        }
        write_new_json(directory/"run_metadata.json",base)
        history, selected, early_state = [], None, {}
    context = {"config":cfg,"directory":directory,"model":model,"optimizer":optimizer,
               "history":history,"selected":selected,"base_metadata":base,"loaders":None,
               "started":True,"completed":False,"last_handoff":None}
    candidate_spec = {"ids":split.candidate_ids,"pairs":split.candidate_pairs,"sha256":split.candidate_sha256}
    mappings = {key:split.mappings[key] for key in ("crop_id_to_name","disease_id_to_name")}
    previous_resolution = None
    generator_states = None if resume is None else resume["rng_state"]["loaders"]
    stop_reason = "budget_completed"
    # If a process died after logging but before atomic publication, trim uncommitted log lines.
    with (directory/"history.jsonl").open("w",encoding="utf-8") as stream:
        for row in history:stream.write(json.dumps(row,allow_nan=False)+"\n")
    if resume is not None:
        restore_rng_state(resume["rng_state"])
    else:
        seed_everything(cfg["seed"])  # data RNG independent of model initialization
    try:
        if (resume is not None and early_state.get("fine_epochs",0) >= cfg["min_finetune_epochs"]
                and early_state.get("bad_epochs",0) >= cfg["early_stopping_patience"]):
            context["completed"] = True
            context["stop_reason"] = "already_early_stopped"
            return context
        for epoch in range(start_epoch, cfg["warmup_epochs"] + cfg["finetune_epochs"] + 1):
            stage, stage_epoch, resolution = stage_for_epoch(cfg, epoch)
            if stage != current_stage:
                optimizer, scheduler = configure_training_stage(model,cfg,stage)
                context["optimizer"] = optimizer
                current_stage = stage
            active_microbatch = (cfg["warmup_batch_size"] if stage=="warmup" else
                                 cfg["start_batch_size"] if resolution==cfg["start_resolution"] else cfg["batch_size"])
            loader_signature=(resolution,active_microbatch)
            if loader_signature != previous_resolution:
                if context["loaders"] is not None:
                    generator_states = [loader.generator.get_state() for loader in context["loaders"]]
                    shutdown_loaders(context["loaders"])
                context["loaders"] = create_stage_loaders(split,image_directory,{**cfg,"batch_size":active_microbatch},resolution,generator_states)
                previous_resolution = loader_signature
            loaders = context["loaders"]
            print(f"{cfg['architecture']} epoch {epoch} | {stage} {stage_epoch} | {resolution}px "
                  f"| microbatch={active_microbatch} target_effective={cfg['effective_batch_size']} "
                  f"| precision={cfg['precision']}",flush=True)
            if device.type=="cuda":
                torch.cuda.synchronize(device);torch.cuda.reset_peak_memory_stats(device)
            started = time.perf_counter()
            train = train_one_epoch(model,loaders[0],optimizer,device,
                crop_weight=cfg["crop_weight"],disease_weight=cfg["disease_weight"],
                precision=cfg["precision"],accumulation_steps=cfg["accumulation_steps"],
                scaler=scaler,channels_last=cfg["channels_last"],clip_grad_norm=cfg["clip_grad_norm"],
                effective_batch_size=cfg["effective_batch_size"],
                frozen_backbone=stage=="warmup",log_every=cfg["log_every_batches"])
            seen = validate_one_epoch(model,loaders[1],device,crop_weight=cfg["crop_weight"],
                disease_weight=cfg["disease_weight"],precision=cfg["precision"],channels_last=cfg["channels_last"])
            unseen = validate_one_epoch(model,loaders[2],device,crop_weight=cfg["crop_weight"],
                disease_weight=cfg["disease_weight"],precision=cfg["precision"],channels_last=cfg["channels_last"])
            if (seen["sample_count"],unseen["sample_count"]) != (len(split.val_seen),len(split.val_unseen)):
                raise RuntimeError("Validation sample counts changed")
            metrics = evaluate_development(seen,unseen,candidate_spec)
            key = checkpoint_selection_key(metrics,seen,unseen,epoch)
            pilot_diagnostics = None
            pilot_diagnostics_path = None
            if cfg.get("experiment_kind") == "m5e_pilot":
                pilot_diagnostics = pilot_epoch_diagnostics(seen,unseen,split,fitting_loader=loaders[0])
                pilot_diagnostics_path = directory/f"diagnostics_epoch_{epoch:04d}_{uuid.uuid4().hex[:8]}.json"
                write_new_json(pilot_diagnostics_path,pilot_diagnostics)
            new_checkpoint = None
            if context["selected"] is None or key < tuple(context["selected"]["key"]):
                new_checkpoint = directory/f"selected_epoch_{epoch:04d}_{uuid.uuid4().hex[:8]}.pt"
                selection={"epoch":epoch,"top1_harmonic_mean":metrics["top1_harmonic_mean"],
                           "seen_top1":metrics["seen_top1"],"unseen_top1":metrics["unseen_top1"],
                           "combined_validation_loss":key[1],"metrics":metrics,"policy":SELECTION_POLICY,
                           "stage":stage,"resolution":resolution,
                           "pilot_diagnostics_path":None if pilot_diagnostics_path is None else str(pilot_diagnostics_path)}
                save_checkpoint(model,new_checkpoint,class_mappings=mappings,
                                preprocessing=model_preprocessing_metadata(cfg,epoch=epoch,selected=True),
                                metadata={**base,"epoch":epoch,"selection":selection,
                                    "preprocessing":model_preprocessing_metadata(cfg,epoch=epoch,selected=True)})
                context["selected"]={**selection,"key":key,"path":str(new_checkpoint)}
            if scheduler is not None:
                scheduler.step()
            early_state, stop = update_early_stopping(early_state,metrics["top1_harmonic_mean"],cfg,stage=stage)
            if device.type=="cuda":torch.cuda.synchronize(device)
            row={"epoch":epoch,"stage":stage,"stage_epoch":stage_epoch,"resolution":resolution,
                 "physical_microbatch":active_microbatch,"effective_batch_target":cfg["effective_batch_size"],
                 "epoch_seconds":time.perf_counter()-started,"train_images_per_second":train["images_per_second"],
                 "peak_gpu_allocated_mib":torch.cuda.max_memory_allocated(device)/2**20 if device.type=="cuda" else None,
                 "peak_gpu_reserved_mib":torch.cuda.max_memory_reserved(device)/2**20 if device.type=="cuda" else None,
                 "train_samples":train["sample_count"],"optimizer_steps":train["optimizer_steps"],
                 "pilot_diagnostics_path":None if pilot_diagnostics_path is None else str(pilot_diagnostics_path),
                 "early_stopping":dict(early_state),"seen_samples":seen["sample_count"],
                 "unseen_samples":unseen["sample_count"],**metrics,
                 **{f"{fold}_{task}_loss":result["losses"][task] for fold,result in
                    (("train",train),("seen",seen),("unseen",unseen)) for task in ("total","crop","disease")},
                 "combined_validation_loss":key[1],"selected_epoch":context["selected"]["epoch"],
                 "new_selected_checkpoint":None if new_checkpoint is None else str(new_checkpoint),
                 "next_learning_rates":[group["lr"] for group in optimizer.param_groups],
                 "scaler_scale":scaler.get_scale()}
            context["history"].append(row)
            with (directory/"history.jsonl").open("a",encoding="utf-8") as stream:
                stream.write(json.dumps(row,allow_nan=False)+"\n");stream.flush();os.fsync(stream.fileno())
            save_recovery_checkpoint(context,epoch,stage,resolution,scheduler,scaler,early_state)
            print(json.dumps(row),flush=True)
            if stop or (directory/"STOP_AFTER_EPOCH").exists():
                stop_reason="early_stopping" if stop else "owner_stop_file"
                break
        context["completed"]=stop_reason != "owner_stop_file"
    except KeyboardInterrupt:
        stop_reason="interrupted_resume_last_completed"
        print("Interrupted. Resume only from last_completed.pt; current in-memory partial state is not committed.")
    finally:
        shutdown_loaders(context["loaders"])
    context["stop_reason"]=stop_reason
    return context

def summarize_high_performance_run(context):
    cfg=context["config"];history=context["history"]
    summary={"run_id":context["base_metadata"]["run_id"],"architecture":cfg["architecture"],
             "execution_status":"completed" if context["completed"] else "partial_or_interrupted",
             "stop_reason":context["stop_reason"],"configuration":cfg,
             "preprocessing":context["base_metadata"]["preprocessing"],"split":context["base_metadata"]["split"],
             "candidate_sha256":context["base_metadata"]["split"]["candidate_sha256"],
             "selection_policy":SELECTION_POLICY,"metrics":{"implementation":METRIC_IMPLEMENTATION,"scoring_id":SCORING_ID},
             "selected":context["selected"],"parameters":sum(p.numel() for p in context["model"].parameters()),
             "completed_epochs":len(history),"history":history,
             "initial_model_sha256":context["base_metadata"].get("initial_model_sha256"),
             "initialization_verified":context["base_metadata"].get("initialization_verified",False),
             "selected_preprocessing":(model_preprocessing_metadata(cfg,epoch=context["selected"]["epoch"],selected=True) if context["selected"] else None),
             "selected_pilot_diagnostics":(json.loads(Path(context["selected"]["pilot_diagnostics_path"]).read_text())
                if context["selected"] and context["selected"].get("pilot_diagnostics_path") else None),
             "duration_seconds":sum(row["epoch_seconds"] for row in history),
             "peak_gpu_allocated_mib":max((row["peak_gpu_allocated_mib"] for row in history
                                           if row["peak_gpu_allocated_mib"] is not None),default=None)}
    # Summary is an owned, derived publication, not a model checkpoint.
    destination=context["directory"]/"run_summary.json"
    temporary=destination.with_suffix(".json.tmp")
    temporary.write_text(json.dumps(summary,indent=2,allow_nan=False)+"\n")
    os.replace(temporary,destination)
    return summary

def run_experiment_sequence(plan, split, image_directory, root):
    # M5E blocks the historical unrestricted two-backbone launch.
    if len(plan)!=2 or any(c.get("experiment_kind")!="m5e_pilot" for c in plan):
        raise ValueError("Historical full comparison is disabled; use the explicit two-arm M5E pilot.")
    return run_m5e_pilot_sequence(plan,split,image_directory,root)


def run_m5e_pilot_sequence(plan, split, image_directory, root):
    """Owner-invoked finite two-arm run; never called by imports or setup."""
    plan=validate_m5e_pilot_plan(plan)
    if len(split.fit)!=4868 or len(split.val_seen)!=1238 or len(split.val_unseen)!=1894:
        raise ValueError("Pilot requires the accepted training-derived split.")
    # Verify one common local artifact before starting either experiment.
    state,provenance=verified_backbone_state(plan[0]);del state
    identifier="m5e_pair_"+datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%SZ")+"_"+uuid.uuid4().hex[:8]
    pair_directory=reserve_run_directory(plan[0]["output_directory"],identifier)
    prepared=[]
    for config in plan:
        prepared.append({**config,"output_directory":str(pair_directory)})
    write_new_json(pair_directory/"paired_configuration.json",
                   {"plan":prepared,"initialization":provenance,"comparison_budget":"17epochs;4868draws/epoch;77updates/epoch"})
    summaries=[]
    for config in prepared:
        context=run_high_performance_experiment(config,split,image_directory,root)
        summary=summarize_high_performance_run(context)
        summaries.append(summary)
        plot_run_history(context["history"],"M5E "+config["pilot_arm"])
        complete=context["completed"];del context;gc.collect()
        if torch.cuda.is_available():torch.cuda.empty_cache()
        if not complete:
            print("Pilot stopped after interruption; no paired effectiveness claim.")
            break
    if len(summaries)==2 and all(s["execution_status"]=="completed" for s in summaries):
        comparison=compare_m5e_pilot_results(summaries)
        write_new_json(pair_directory/"paired_comparison.json",comparison)
        pd.DataFrame(comparison["comparison"]).to_csv(pair_directory/"paired_comparison.csv",index=False)
        pd.DataFrame(comparison["per_pair"]).to_csv(pair_directory/"unseen_pair_comparison.csv",index=False)
        print(pd.DataFrame(comparison["comparison"]).to_string(index=False))
        print(comparison["assessment"])
    return summaries


In [ ]:
def measure_training_profile(model, images, targets, config, microbatch, *,
                             precision=None, channels_last=None, checkpointing=None,
                             warmup_steps=1, measured_steps=2):
    # Bounded disposable training batches only; never call this on an experiment model.
    if not (0 <= warmup_steps <= 2 and 1 <= measured_steps <= 3):
        raise ValueError("Probe budget is capped at two warm-ups and three measured batches")
    if images.shape[0] < microbatch or microbatch < 1:
        raise ValueError("Insufficient fitting samples for this microbatch")
    device=torch.device(config["device"])
    if device.type!="cuda":
        raise ValueError("GPU throughput probes require CUDA")
    precision=precision or config["precision"]
    channels_last=config["channels_last"] if channels_last is None else channels_last
    checkpointing=config["gradient_checkpointing"] if checkpointing is None else checkpointing
    model.gradient_checkpointing=checkpointing
    model.to(device)
    model.to(memory_format=torch.channels_last if channels_last else torch.contiguous_format)
    model.train()
    optimizer=torch.optim.AdamW(model.parameters(),lr=config["backbone_lr"],weight_decay=config["weight_decay"])
    scaler=make_scaler(device,precision)
    torch.cuda.synchronize(device);torch.cuda.reset_peak_memory_stats(device)
    times=[]
    result={"architecture":model.architecture,"resolution":images.shape[-1],"microbatch":microbatch,
            "precision":precision,"channels_last":channels_last,"activation_checkpointing":checkpointing,
            "warmup_steps":warmup_steps,"measured_steps":measured_steps}
    try:
        for index in range(warmup_steps+measured_steps):
            optimizer.zero_grad(set_to_none=True)
            started=time.perf_counter()
            x=images[:microbatch].to(device,non_blocking=True)
            if channels_last:x=x.contiguous(memory_format=torch.channels_last)
            y={key:value[:microbatch].to(device,non_blocking=True) for key,value in targets.items()}
            with amp_context(device,precision):
                output=model(x)
                losses=compute_multitask_loss(output,y)
            scaler.scale(losses["total"]).backward()
            norm=optimizer_update(model,optimizer,scaler,config["clip_grad_norm"])
            torch.cuda.synchronize(device)
            elapsed=time.perf_counter()-started
            if index>=warmup_steps:times.append(elapsed)
        result.update(status="passed",images_per_second=microbatch/(sum(times)/len(times)),
                      batch_seconds=sum(times)/len(times),loss=float(losses["total"].detach()),
                      gradient_norm=norm,peak_allocated_mib=torch.cuda.max_memory_allocated(device)/2**20,
                      peak_reserved_mib=torch.cuda.max_memory_reserved(device)/2**20)
    except FloatingPointError as error:
        result.update(status="nonfinite_gradient",error=str(error))
    except torch.cuda.OutOfMemoryError:
        result.update(status="oom",peak_allocated_mib=torch.cuda.max_memory_allocated(device)/2**20,
                      peak_reserved_mib=torch.cuda.max_memory_reserved(device)/2**20)
    finally:
        model.zero_grad(set_to_none=True)
        del optimizer,scaler
        gc.collect();torch.cuda.empty_cache()
    return result


In [ ]:
def bounded_gpu_tuning(config, split, image_directory, *, max_profiles=16, max_seconds=300, existing_profiles=None):
    if not 1 <= max_profiles <= 16 or not 1 <= max_seconds <= 600:
        raise ValueError("Bounded tuning permits at most 16 three-batch profiles per model")
    cfg=validate_config(config)
    seed_everything(cfg["seed"])
    state, provenance=verified_backbone_state(cfg)
    model=create_multitask_model(cfg["architecture"],backbone_state_dict=state).cuda()
    del state
    records=list(existing_profiles or []);started=time.perf_counter()
    dataset_cache={}
    def fitting_sample(resolution):
        if resolution not in dataset_cache:
            dataset=DPDataset(split.fit.iloc[:128],image_directory,image_size=resolution,augment=True)
            examples=[dataset[index] for index in range(128)]
            images=torch.stack([item[0] for item in examples])
            labels={key:torch.stack([item[1][key] for item in examples]) for key in ("crop","disease")}
            if cfg["pin_memory"]:
                images=images.pin_memory();labels={key:value.pin_memory() for key,value in labels.items()}
            dataset_cache[resolution]=(images,labels)
        return dataset_cache[resolution]
    try:
        for resolution in dict.fromkeys((cfg["start_resolution"],cfg["final_resolution"])):
            images,labels=fitting_sample(resolution)
            for batch in (2,4,8,16,32,64,128):
                if len(records)>=max_profiles or time.perf_counter()-started>max_seconds:
                    break
                prior=[row for row in records if row["status"]=="passed" and row["resolution"]==resolution
                       and row["precision"]=="bf16" and row["channels_last"]]
                if any(row["microbatch"]==batch for row in prior):
                    continue
                if prior:
                    largest=max(prior,key=lambda row:row["microbatch"])
                    smaller=[row for row in prior if row["microbatch"]<largest["microbatch"]]
                    base=min(prior,key=lambda row:row["microbatch"])["peak_reserved_mib"]
                    predicted=base+(largest["peak_reserved_mib"]-base)*batch/largest["microbatch"]
                    free,_=torch.cuda.mem_get_info()
                    practical=(free+torch.cuda.memory_allocated())/2**20-512
                    if predicted>practical:
                        skipped={"architecture":cfg["architecture"],"resolution":resolution,
                                 "microbatch":batch,"status":"not_attempted_predicted_vram_excess",
                                 "predicted_reserved_mib":predicted,"practical_budget_mib":practical}
                        records.append(skipped);print(json.dumps(skipped),flush=True)
                        middle=(largest["microbatch"]+batch)//2
                        estimate=base+(largest["peak_reserved_mib"]-base)*middle/largest["microbatch"]
                        while middle>largest["microbatch"]+1 and estimate>practical:
                            middle=(largest["microbatch"]+middle)//2
                            estimate=base+(largest["peak_reserved_mib"]-base)*middle/largest["microbatch"]
                        if middle>largest["microbatch"] and len(records)<max_profiles:
                            refined=measure_training_profile(model,images,labels,cfg,middle,precision="bf16",
                                                             channels_last=True,checkpointing=False)
                            records.append(refined);print(json.dumps(refined),flush=True)
                        break
                result=measure_training_profile(model,images,labels,cfg,batch,
                                                precision="bf16" if torch.cuda.is_bf16_supported() else "fp16",
                                                channels_last=True,checkpointing=False)
                records.append(result);print(json.dumps(result),flush=True)
                if result["status"]=="oom":
                    passed=[row["microbatch"] for row in records if row["status"]=="passed" and row["resolution"]==resolution]
                    if passed and len(records)<max_profiles and time.perf_counter()-started<max_seconds:
                        middle=(max(passed)+batch)//2
                        refined=measure_training_profile(model,images,labels,cfg,middle,precision="bf16",
                                                         channels_last=True,checkpointing=False)
                        records.append(refined);print(json.dumps(refined),flush=True)
                    break
        # Compare precision/layout at a known successful final-resolution microbatch.
        successful=[row for row in records if row["status"]=="passed" and row["resolution"]==cfg["final_resolution"]]
        if successful:
            best=max(successful,key=lambda row:row["images_per_second"])
            images,labels=fitting_sample(cfg["final_resolution"])
            for precision,layout,checkpointing in (
                    ("fp16",True,False),("bf16",False,False),
                    ("bf16",True,True) if "convnext" in cfg["architecture"] else ("fp32",True,False)):
                if len(records)>=max_profiles or time.perf_counter()-started>max_seconds:break
                result=measure_training_profile(model,images,labels,cfg,min(best["microbatch"],8),
                                                precision=precision,channels_last=layout,
                                                checkpointing=checkpointing)
                records.append(result);print(json.dumps(result),flush=True)
    finally:
        del model
        gc.collect();torch.cuda.empty_cache()
    return {"configuration":cfg,"weights":provenance,"profiles":records,
            "wall_seconds":time.perf_counter()-started,
            "scope":"128 fitting images reused; disposable model; at most 48 optimizer batches; no epochs or selection metrics"}

def compare_planned_runs(summaries):
    eligible=[summary for summary in summaries if summary.get("execution_status")=="completed"
              and summary.get("selected") is not None]
    if len(eligible)!=len(summaries):
        print("Paused/interrupted/unselected runs remain recorded but are excluded from winner selection.")
    table=pd.DataFrame([comparison_record(summary) for summary in eligible])
    print(table.to_string(index=False) if not table.empty else "No completed comparison rows.")
    summaries=eligible
    if len(summaries)!=2 or {s["architecture"] for s in summaries}!=set(APPROVED_ARCHITECTURES):
        print("Two completed final-candidate runs are required; no winner.")
        return table
    keys=("seed","warmup_epochs","finetune_epochs","progressive_epochs","optimizer","weight_decay",
          "warmup_head_lr","backbone_lr","head_lr","early_stopping_patience","min_finetune_epochs",
          "crop_weight","disease_weight","augment_fit","initialization")
    signatures=[]
    for summary in summaries:
        cfg=summary["configuration"]
        signatures.append(json.dumps({
            "configuration":{key:cfg[key] for key in keys},
            "effective_batch_size":cfg["effective_batch_size"],
            "source":summary["split"]["source_sha256"],"split":summary["split"]["split_sha256"],
            "candidates":summary["candidate_sha256"],"metric_specification":summary["metrics"],
            "selection_policy":summary["selection_policy"]},sort_keys=True))
    if len(set(signatures))!=1:
        print("Core budgets/protocol/effective batches differ; no automatic controlled winner.")
    else:
        leader=min(summaries,key=lambda summary:tuple(summary["selected"]["key"]))
        print("Development leader from actual completed records:",leader["architecture"])
        print("This is a development comparison, not an official-test claim; resolutions/resource settings are shown.")
    return table

GPU_TUNING_RESULTS = []
if RUN_GPU_TUNING:
    GPU_TUNING_RESULTS = [bounded_gpu_tuning(config,development,IMAGE_DIRECTORY) for config in EXPERIMENT_PLAN]
else:
    print("Bounded GPU tuning is disabled. Recorded preparation measurements are in the handoff.")


In [ ]:
def bounded_real_smoke(config, split, image_directory):
    cfg=validate_config(config);seed_everything(cfg["seed"])
    state,provenance=verified_backbone_state(cfg)
    model=create_multitask_model(cfg["architecture"],backbone_state_dict=state).cuda().to(memory_format=torch.channels_last)
    del state
    records={}
    try:
        optimizer,_=configure_training_stage(model,cfg,"warmup")
        warm_dataset=DPDataset(split.fit.iloc[:64],image_directory,image_size=cfg["start_resolution"],augment=True)
        warm_loader=DataLoader(warm_dataset,batch_size=64,num_workers=0,pin_memory=True)
        before={name:value.clone() for name,value in model.backbone.named_buffers()}
        torch.cuda.reset_peak_memory_stats()
        warm=train_one_epoch(model,warm_loader,optimizer,"cuda",precision=cfg["precision"],
                             effective_batch_size=64,channels_last=True,frozen_backbone=True,
                             scaler=make_scaler("cuda",cfg["precision"]),clip_grad_norm=cfg["clip_grad_norm"])
        if any(not torch.equal(value,before[name]) for name,value in model.backbone.named_buffers()):
            raise RuntimeError("Frozen warm-up changed backbone buffers")
        records["head_warmup"]={**warm,"physical_microbatch":64,"resolution":cfg["start_resolution"],
                               "peak_allocated_mib":torch.cuda.max_memory_allocated()/2**20,
                               "backbone_buffers_unchanged":True}
        del optimizer,before,warm_loader
        optimizer,_=configure_training_stage(model,cfg,"finetune")
        fine_dataset=DPDataset(split.fit.iloc[:64],image_directory,image_size=cfg["final_resolution"],augment=True)
        fine_loader=DataLoader(fine_dataset,batch_size=cfg["batch_size"],num_workers=0,pin_memory=True)
        torch.cuda.reset_peak_memory_stats()
        fine=train_one_epoch(model,fine_loader,optimizer,"cuda",precision=cfg["precision"],
                             effective_batch_size=64,channels_last=True,
                             scaler=make_scaler("cuda",cfg["precision"]),clip_grad_norm=cfg["clip_grad_norm"])
        records["one_full_finetune_update"]={**fine,"resolution":cfg["final_resolution"],
                                            "physical_microbatch":cfg["batch_size"],
                                            "peak_allocated_mib":torch.cuda.max_memory_allocated()/2**20}
        if fine["optimizer_steps"]!=1 or warm["optimizer_steps"]!=1:
            raise RuntimeError("Smoke step budget exceeded")
        for name,frame in (("seen",split.val_seen),("unseen",split.val_unseen)):
            loader=DataLoader(DPDataset(frame.iloc[:8],image_directory,image_size=cfg["final_resolution"],
                                       augment=False),batch_size=8,pin_memory=True)
            result=validate_one_epoch(model,loader,"cuda",precision=cfg["precision"],channels_last=True)
            records[name]={"samples":result["sample_count"],"losses":result["losses"],
                           "crop_shape":list(result["crop_logits"].shape),
                           "disease_shape":list(result["disease_logits"].shape),
                           "finite":bool(torch.isfinite(result["crop_logits"]).all()
                                         and torch.isfinite(result["disease_logits"]).all())}
        return {"configuration":cfg,"weights":provenance,"records":records,
                "scope":"64 fitting images reused for one warm-up update and one accumulated fine-tune update; 8 images per validation fold; no dataset epoch or selection metrics"}
    finally:
        del model
        gc.collect();torch.cuda.empty_cache()

def plot_run_history(history,title):
    frame=pd.DataFrame(history)
    if frame.empty:
        print("No recorded history; no plots.")
        return
    figure,axes=plt.subplots(1,3,figsize=(15,4))
    for axis,task in zip(axes,("total","crop","disease")):
        for fold in ("train","seen","unseen"):
            axis.plot(frame["epoch"],frame[f"{fold}_{task}_loss"],label=fold)
        axis.set(title=title+" "+task+" loss",xlabel="Epoch");axis.legend()
    plt.tight_layout();plt.show()
    frame.plot(x="epoch",y=["seen_top1","unseen_top1","seen_top5","unseen_top5",
                           "top1_harmonic_mean","top5_harmonic_mean"],ylim=(0,1),
               title=title+" development metrics");plt.show()
    frame.plot(x="epoch",y=["peak_gpu_allocated_mib","peak_gpu_reserved_mib"],
               title=title+" peak GPU MiB");plt.show()
    frame.plot(x="epoch",y=["epoch_seconds"],title=title+" epoch duration");plt.show()

def bounded_pipeline_training_comparison(config, split, image_directory):
    cfg=validate_config(config);seed_everything(cfg["seed"])
    state,provenance=verified_backbone_state(cfg)
    model=create_multitask_model(cfg["architecture"],backbone_state_dict=state).cuda().to(memory_format=torch.channels_last)
    del state
    records=[]
    frame=split.fit.iloc[:128]
    subset=DevelopmentSplit(frame,split.val_seen.iloc[:1],split.val_unseen.iloc[:1],split.mappings,
                            split.candidate_ids,split.candidate_pairs,split.source_csv,split.source_sha256,
                            split.candidate_sha256,split.seed,split.heldout_fraction,split.seen_fraction)
    try:
        for workers,pinned,persistent in ((4,True,True),(8,True,True),(4,False,True),(4,True,False)):
            setting={**cfg,"num_workers":workers,"pin_memory":pinned,"persistent_workers":persistent}
            loaders=create_stage_loaders(subset,image_directory,setting,cfg["final_resolution"])
            optimizer,_=configure_training_stage(model,cfg,"finetune")
            scaler=make_scaler("cuda",cfg["precision"])
            try:
                # Two bounded passes over the same 128-image subset: two updates warm, two timed.
                for measured in (False,True):
                    torch.cuda.synchronize();started=time.perf_counter()
                    result=train_one_epoch(model,loaders[0],optimizer,"cuda",precision=cfg["precision"],
                        effective_batch_size=64,channels_last=True,scaler=scaler,
                        clip_grad_norm=cfg["clip_grad_norm"],log_every=1000)
                    torch.cuda.synchronize();elapsed=time.perf_counter()-started
                    if result["optimizer_steps"]!=2:raise RuntimeError("Pipeline probe exceeded update budget")
                row={"architecture":cfg["architecture"],"resolution":cfg["final_resolution"],
                     "workers":workers,"pin_memory":pinned,"persistent_workers":persistent,
                     "prefetch_factor":cfg["prefetch_factor"],"physical_microbatch":cfg["batch_size"],
                     "effective_batch_size":64,"warmup_updates":2,"measured_updates":2,
                     "measured_images":128,"seconds":elapsed,"images_per_second":128/elapsed,
                     "scope":"end-to-end fitting-subset input+H2D+forward+backward+AdamW; no full dataset epoch"}
                records.append(row);print(json.dumps(row),flush=True)
            finally:
                shutdown_loaders(loaders)
                del optimizer,scaler
                gc.collect();torch.cuda.empty_cache()
    finally:
        del model
        gc.collect();torch.cuda.empty_cache()
    return {"configuration":cfg,"weights":provenance,"profiles":records,
            "budget":"128 fitting images reused; 16 optimizer updates maximum across four pipeline settings"}


In [ ]:
def reserve_run_directory(output_directory, run_id):
    if not re.fullmatch(r"[A-Za-z0-9][A-Za-z0-9_-]{0,95}",run_id):
        raise ValueError("Unsafe run ID")
    parent=Path(output_directory).expanduser().resolve();parent.mkdir(parents=True,exist_ok=True)
    directory=parent/run_id;directory.mkdir(exist_ok=False)
    return directory

def write_new_json(path, value):
    with Path(path).open("x",encoding="utf-8") as stream:
        json.dump(value,stream,indent=2,allow_nan=False);stream.write("\n")

context = None
if PREPARE_RUN:
    for planned_config in EXPERIMENT_PLAN:
        validate_config(planned_config)
    print("Plan validated. No model/optimizer or training starts here.")
    print(pd.DataFrame([{key:config[key] for key in
                        ("architecture","warmup_epochs","finetune_epochs","start_resolution",
                         "final_resolution","batch_size","accumulation_steps","precision")}
                        for config in EXPERIMENT_PLAN]).to_string(index=False))
else:
    print("Plan preparation disabled.")


In [ ]:
EXPERIMENT_SUMMARIES = []
if not (RUN_TRAINING and RUN_EXPENSIVE_EVALUATION):
    print("SKIPPED: full training and validation remain disabled.")
else:
    if not PREPARE_RUN or not RUN_PLANNED_EXPERIMENTS:
        raise RuntimeError("Owner must explicitly enable PREPARE_RUN and RUN_PLANNED_EXPERIMENTS.")
    EXPERIMENT_SUMMARIES = run_experiment_sequence(
        EXPERIMENT_PLAN, development, IMAGE_DIRECTORY, ROOT)


In [ ]:
print("Each completed epoch atomically publishes last_completed.pt in its owned run directory.")
print("For explicit recovery, set that model's resume_from to this path and restart a fresh kernel.")
print("Best model files remain separate portable version-1 checkpoints; recovery is a versioned training envelope.")
print("An interrupted partial epoch is never published; remove STOP_AFTER_EPOCH manually before resuming.")
if SAVE_FINAL_SNAPSHOT or VERIFY_SAVED_CHECKPOINT:
    print("Use the visible save_checkpoint/load_checkpoint functions for explicit portable model actions.")


## 14. Recorded visualizations
The owner-started sequence automatically plots each model's actual losses and development metrics, then records its summary. No experiment curves are fabricated during preparation.
Per-epoch JSONL includes stage/resolution, throughput, component/total losses, all metrics, duration, peak allocated/reserved VRAM, LRs and scaler scale. H@1-only early stopping is strict, with no tolerance, after a configurable minimum fine-tuning duration.


In [ ]:
history_frame = pd.DataFrame([])
if history_frame.empty:
    print("No current standalone epoch history; automatic per-model plots appear only after owner-run experiments.")
# The sequence runner plots recorded losses/metrics after each model. No invented curves.


## 15. Actual model comparison
Completed run summaries produce the full comparison table. The comparison checks common initialization mode, seeds, stage budgets, optimizer/loss settings, early stopping, effective batches and source/split/candidate/metric identities.
Physical batch, resolution, device and actual completed epochs/resource use are shown. Architecture-specific resolutions are declared; they are not hidden as equal-compute experiments.
A development leader is reported only when both final candidates completed under matching core settings. Never infer a winner from Assignment 01. Official test evaluation remains outside this workflow.


In [ ]:
if not EXPERIMENT_SUMMARIES:
    print("No full experiment executed. Bounded preparation evidence is separate.")
else:
    print("Recorded run states:", [(summary["architecture"],summary["execution_status"],
                                   summary["completed_epochs"]) for summary in EXPERIMENT_SUMMARIES])


In [ ]:
def comparison_record(summary):
    if summary.get("execution_status") != "completed" or summary.get("selected") is None:
        raise ValueError("Comparison requires a normally completed run with a selected checkpoint.")
    cfg, selected = summary["configuration"], summary["selected"]
    if summary["architecture"] not in APPROVED_ARCHITECTURES:
        raise ValueError("Unapproved comparison architecture.")
    epoch = selected["epoch"]
    row = next(item for item in summary["history"] if item["epoch"] == epoch)
    return {
        "backbone": summary["architecture"], "parameters": summary["parameters"],
        "best_epoch": epoch, "selected_stage": row.get("stage"),
        "selected_resolution": row.get("resolution"),
        "completed_epochs": summary["completed_epochs"], "seen_top1": row["seen_top1"], "seen_top5": row["seen_top5"],
        "unseen_top1": row["unseen_top1"], "unseen_top5": row["unseen_top5"],
        "development_H1": row["top1_harmonic_mean"], "development_H5": row["top5_harmonic_mean"], "duration_seconds": summary["duration_seconds"],
        "peak_gpu_MiB": summary["peak_gpu_allocated_mib"], "checkpoint": selected["path"],
        "batch_size": cfg["batch_size"], "device": cfg["device"], "seed": cfg["seed"],
        "budget_epochs": cfg["epochs"],
    }

def fairness_signature(summary):
    cfg = summary["configuration"]
    keys = ("seed", "epochs", "optimizer", "learning_rate", "weight_decay",
            "crop_weight", "disease_weight", "augment_fit", "initialization")
    record = {"configuration": {key: cfg[key] for key in keys},
              "preprocessing": summary["preprocessing"],
              "split_sha256": summary["split"]["split_sha256"],
              "candidate_sha256": summary["candidate_sha256"],
              "source_sha256": summary["split"]["source_sha256"],
              "metric_specification": summary["metrics"],
              "selection_policy": summary["selection_policy"],
              "deterministic_algorithms": cfg["deterministic_algorithms"]}
    return json.dumps(record, sort_keys=True, allow_nan=False)


comparison_summaries = list(EXPERIMENT_SUMMARIES)
if READ_RECORDED_RESULTS:
    for path in RECORDED_RUN_SUMMARIES:
        with Path(path).expanduser().open(encoding="utf-8") as stream:
            comparison_summaries.append(json.load(stream))
if comparison_summaries:
    compare_planned_runs(comparison_summaries)
else:
    print("No completed experiment results; no winner is claimed.")


## 16. Execution and recovery Q&A
- One shared feature extractor feeds independent crop/disease heads; both losses update it during full fine-tuning.
- Verified ImageNet initialization, head-only warm-up and proper BN handling differ fundamentally from the old one-epoch scratch B0 check.
- AMP reduces activation memory/compute; sample-exact accumulation increases effective batch while physical-batch normalization remains distinct.
- Completed-epoch recovery restores optimizer/scheduler/scaler and relevant RNG; persistent worker augmentation has an explicit reproducibility limit.
- Stop by epoch-boundary marker when practical. On failure/interruption, start a fresh kernel and explicitly set each pending model's resume_from to its original last_completed.pt.
- Preserve thermal protections, reliable power and adequate cooling. No power/thermal limits are changed here.
No substantive experiment or winner is claimed by engineering preparation. Review measured GPU settings, then the owner manually starts the finite sequence in VS Code.


### M5D — Diagnose weak unseen-composition accuracy (inference only)

This append-only diagnostic workflow preserves the original 16 experiment sections and their training,
metric and selection functions. Only training-derived development data is evaluated.
The exact executed notebook was backed up before clearing outputs and restoring default-off controls.

Read the preparation plan in the external evidence directory. The diagnostics independently verify
candidate mappings and score calculations, report separate crop/disease accuracies, all held-out
pairs, fitting support, fitting-composition preference and deterministic error examples. Saved states
are read with weights_only=True. No optimizer is constructed or stepped.

For each architecture evaluate selected/native, final/native-final and selected/final resolution.
The matched-resolution control separates resolution sensitivity from weight changes.
Logsumexp and raw-sum rankings are independent numerical audits; reported metrics retain the
accepted log_softmax specification, exact ranking and existing checkpoint-selection policy.
Historical selection and fine-only early stopping are replayed without modifying either.

Only enable RUN_M5D_DIAGNOSTICS for inference; leave all training controls off.
Outputs are written to a new diagnostic directory, never the original experiment directories.
Prediction NPZ files contain plain component logits/truths, not weights; load with allow_pickle=False.
Original checkpoint code identities remain unchanged: use the exact backed-up notebook for
any old-run recovery, because appending diagnostic code changes the current notebook identity.

Known audited defect: validation_transform text records224 while staged loaders use actual epoch resolution; selected preprocessing.image_size describes final target rather than selected native resolution. Use native selection/history fields for this diagnosis. Existing checkpoint metadata is preserved; corrections await review.


In [ ]:
# M5D is inference-only and independent of all training controls.
RUN_M5D_DIAGNOSTICS = False
SHOW_M5D_ERROR_IMAGES = False
M5D_RUN_DIRECTORIES = (
    "/home/user1/COS30082_experiments/wp2/20261030T100849Z_multitask_efficientnet_b3_f6ee99eb",
    "/home/user1/COS30082_experiments/wp2/20261030T103349Z_multitask_convnext_base_22e3ef2f",
)
M5D_OUTPUT_ROOT = "/home/user1/COS30082_review_handoffs/M5D_diagnostic_evidence_20261030"
RUN_M5D_RECORDED_BATCH_CHECK = False


In [ ]:
def d_candidate_records(candidate_spec):
    """Validate the diagnostic column contract independently of production helpers."""
    ids = tuple(candidate_spec["ids"])
    pairs = tuple(tuple(pair) for pair in candidate_spec["pairs"])
    if (not ids or len(ids) != len(pairs) or any(type(x) is not int or x < 0 for x in ids)
            or tuple(sorted(set(ids))) != ids or len(set(pairs)) != len(pairs)):
        raise ValueError("Candidates require unique ascending global IDs and unique pairs.")
    if any(len(pair) != 2 or any(type(x) is not int for x in pair)
           or not (0 <= pair[0] < 55 and 0 <= pair[1] < 175) for pair in pairs):
        raise ValueError("Candidate component IDs are out of range.")
    records = [[global_id, *pair] for global_id, pair in zip(ids, pairs)]
    digest = hashlib.sha256(json.dumps(records, separators=(",", ":")).encode()).hexdigest()
    if candidate_spec.get("sha256", digest) != digest:
        raise ValueError("Diagnostic candidate hash mismatch.")
    return ids, pairs, digest

def d_audit_contract(split):
    """Re-derive every candidate and split identity from training metadata only."""
    source = load_metadata(split.source_csv)
    by_id = {}
    for row in source.itertuples(index=False):
        record = (int(row.crop_id), int(row.disease_id))
        key = int(row.composition_id)
        if key in by_id and by_id[key] != record:
            raise ValueError("One global ID maps to multiple component pairs.")
        by_id[key] = record
    spec = {"ids": split.candidate_ids, "pairs": split.candidate_pairs,
            "sha256": split.candidate_sha256}
    ids, pairs, digest = d_candidate_records(spec)
    if tuple(sorted(by_id)) != ids or tuple(by_id[i] for i in ids) != pairs:
        raise ValueError("Training CSV and candidate columns disagree.")
    source_hash = file_sha256(split.source_csv)
    if source_hash != split.source_sha256:
        raise ValueError("Training source identity changed.")
    parts = {"fit": split.fit, "seen": split.val_seen, "unseen": split.val_unseen}
    memberships = {name: set(frame.image_name) for name, frame in parts.items()}
    if (any(memberships[a] & memberships[b] for a,b in
            (("fit","seen"),("fit","unseen"),("seen","unseen")))
            or set.union(*memberships.values()) != set(source.image_name)
            or sum(map(len, memberships.values())) != len(source)):
        raise ValueError("Split overlap or incomplete source coverage.")
    fitting_ids = set(split.fit.composition_id)
    heldout_ids = set(split.val_unseen.composition_id)
    if fitting_ids & heldout_ids or not set(split.val_seen.composition_id) <= fitting_ids:
        raise ValueError("Seen/unseen composition membership is invalid.")
    if (set(split.fit.crop_id) != set(source.crop_id)
            or set(split.fit.disease_id) != set(source.disease_id)):
        raise ValueError("Fitting component coverage changed.")
    membership = {name: getattr(split,name).image_name.tolist()
                  for name in ("fit","val_seen","val_unseen")}
    split_hash = hashlib.sha256(json.dumps(membership,sort_keys=True,separators=(",",":")).encode()).hexdigest()
    crops = source[["crop_id","crop_name"]].drop_duplicates().sort_values("crop_id")
    diseases = source[["disease_id","disease_name"]].drop_duplicates().sort_values("disease_id")
    for table, column in ((crops,"crop_id"),(diseases,"disease_id")):
        if table[column].duplicated().any():
            raise ValueError("Component ID/name correspondence is ambiguous.")
    fit_support = {
        task: {int(i):int(n) for i,n in split.fit.groupby(task+"_id").size().items()}
        for task in ("crop","disease","composition")}
    return {"candidate_count":len(ids), "candidate_sha256":digest,
            "source_sha256":source_hash, "split_sha256":split_hash,
            "image_counts":{k:len(v) for k,v in parts.items()},
            "fitting_crop_count":len(fit_support["crop"]),
            "fitting_disease_count":len(fit_support["disease"]),
            "fitting_composition_count":len(fitting_ids), "heldout_composition_count":len(heldout_ids),
            "ordered_candidates":[{"column":j,"global_id":i,"crop_id":p[0],"disease_id":p[1]}
                                  for j,(i,p) in enumerate(zip(ids,pairs))],
            "component_names":{"crop":crops.to_dict("records"),"disease":diseases.to_dict("records")},
            "fit_support":fit_support}

def d_independent_scores(crop_logits, disease_logits, candidate_spec):
    """Independent float64 logsumexp calculation; no new scoring policy."""
    ids,pairs,_ = d_candidate_records(candidate_spec)
    for logits,width in ((crop_logits,55),(disease_logits,175)):
        if (not isinstance(logits,Tensor) or not logits.is_floating_point()
                or logits.ndim != 2 or logits.shape[1] != width or logits.shape[0] < 1
                or not torch.isfinite(logits).all()):
            raise ValueError("Expected finite nonempty component logits with the agreed widths.")
    if crop_logits.shape[0] != disease_logits.shape[0] or crop_logits.device != disease_logits.device:
        raise ValueError("Component batches/devices differ.")
    c,d = crop_logits.double(),disease_logits.double()
    c = c-torch.logsumexp(c,dim=1,keepdim=True)
    d = d-torch.logsumexp(d,dim=1,keepdim=True)
    result = torch.stack([c[:,a]+d[:,b] for a,b in pairs],dim=1)
    if not torch.isfinite(result).all():
        raise ValueError("Non-finite independent composition scores.")
    return result

def d_audit_history(summary):
    """Replay recorded selection and both stopping references without changing either."""
    history = summary["history"]
    if not history or [r["epoch"] for r in history] != list(range(1,len(history)+1)):
        raise ValueError("History must contain all completed epochs in order.")
    for row in history:
        total = ((row["seen_samples"]*row["seen_total_loss"]+
                  row["unseen_samples"]*row["unseen_total_loss"])/
                 (row["seen_samples"]+row["unseen_samples"]))
        if not math.isfinite(total) or abs(total-row["combined_validation_loss"]) > 1e-12:
            raise ValueError("Recorded sample-weighted loss disagrees.")
    best = min(history,key=lambda r:(-r["top1_harmonic_mean"],r["combined_validation_loss"],r["epoch"]))
    key = [-best["top1_harmonic_mean"],best["combined_validation_loss"],best["epoch"]]
    if summary["selected"]["epoch"] != best["epoch"] or list(summary["selected"]["key"]) != key:
        raise ValueError("Historical selection does not follow the agreed exact policy.")
    fine = [r for r in history if r["stage"]=="finetune"]
    config = summary["configuration"]
    fine_best, fine_best_epoch, bad, fine_stop = -1.,None,0,None
    warm = [r["top1_harmonic_mean"] for r in history if r["stage"]=="warmup"]
    global_best, global_bad, global_stop = max(warm,default=-1.),0,None
    improvements=[]
    for count,row in enumerate(fine,1):
        h=row["top1_harmonic_mean"]
        if h > fine_best:
            fine_best,fine_best_epoch,bad = h,row["epoch"],0
            improvements.append({"epoch":row["epoch"],"h1":h,
                                 "seen_top1":row["seen_top1"],"unseen_top1":row["unseen_top1"]})
        else:
            bad+=1
        if (fine_stop is None and count >= config["min_finetune_epochs"]
                and bad >= config["early_stopping_patience"]):
            fine_stop=row["epoch"]
        if h > global_best:
            global_best,global_bad=h,0
        else:
            global_bad+=1
        if (global_stop is None and count >= config["min_finetune_epochs"]
                and global_bad >= config["early_stopping_patience"]):
            global_stop=row["epoch"]
    return {"global_selected_epoch":best["epoch"],"global_selection_key":key,
            "fine_best_epoch":fine_best_epoch,"fine_best_h1":fine_best,
            "fine_only_first_stop_epoch":fine_stop,"fine_improvements":improvements,
            "fine_state":{"best_h1":fine_best,"bad_epochs":bad,"fine_epochs":len(fine)},
            "globally_anchored_first_stop_epoch":global_stop,
            "counterfactual_only":"Global patience does not rewrite or reselect historical checkpoints.",
            "checkpoint_and_stop_reference_differ":bool(warm)}

def d_state_digest(state):
    """Hash exact parameter AND persistent-buffer bytes, including integer BN counters."""
    digest=hashlib.sha256()
    for name,value in sorted(state.items()):
        array=value.detach().cpu().contiguous()
        digest.update(name.encode());digest.update(str(array.dtype).encode())
        digest.update(str(tuple(array.shape)).encode())
        digest.update(array.reshape(-1).view(torch.uint8).numpy().tobytes())
    return digest.hexdigest()

def d_read_model_state(path, split, kind):
    """Safely inspect a portable selection or recovery envelope without optimizer recovery."""
    payload=torch.load(Path(path),map_location="cpu",weights_only=True,mmap=True)
    architecture=payload.get("architecture")
    if architecture not in ("multitask_efficientnet_b3","multitask_convnext_base"):
        raise ValueError("Diagnostic input must be an approved B3/Base state.")
    if kind not in ("selected","final"):
        raise ValueError("State kind must be selected or final.")
    if kind=="selected":
        for key,value in _checkpoint_config(architecture).items():
            if payload.get(key) != value:
                raise ValueError("Selected checkpoint architecture header mismatch.")
        expected={k:split.mappings[k] for k in ("crop_id_to_name","disease_id_to_name")}
        if _class_mappings(payload.get("class_mappings")) != _class_mappings(expected):
            raise ValueError("Selected checkpoint component mapping mismatch.")
        metadata=payload["metadata"]
        epoch=int(metadata["epoch"])
        resolution=int(metadata["selection"]["resolution"])
    else:
        if payload.get("recovery_format_version") != 1:
            raise ValueError("Unsupported recovery envelope.")
        metadata=payload["base_metadata"]
        epoch=int(payload["completed_epoch"]);resolution=int(payload["completed_resolution"])
    audit=d_audit_contract(split)
    provenance=metadata["split"]
    for key in ("source_sha256","candidate_sha256","split_sha256"):
        if provenance.get(key) != audit[key]:
            raise ValueError("Checkpoint "+key+" disagrees with training-derived development data.")
    if (tuple(provenance["candidate_ids"]) != tuple(split.candidate_ids)
            or tuple(map(tuple,provenance["candidate_pairs"])) != tuple(split.candidate_pairs)):
        raise ValueError("Stored ordered candidates are inconsistent.")
    config=metadata["configuration"]
    if metadata["architecture"] != architecture or config["architecture"] != architecture:
        raise ValueError("Nested architecture identity mismatch.")
    if metadata["preprocessing"]["policy"] != "common_rgb_imagenet_progressive_v1":
        raise ValueError("Unreviewed preprocessing policy; inspect before diagnostic reconstruction.")
    state=payload["model_state_dict"]
    nonfinite=[k for k,v in state.items() if v.is_floating_point() and not torch.isfinite(v).all()]
    if nonfinite:
        raise ValueError("Non-finite saved tensors: "+str(nonfinite))
    return state, {"architecture":architecture,"epoch":epoch,"native_resolution":resolution,
                   "configuration":config,"source_path":str(Path(path).resolve()),
                   "source_sha256":file_sha256(path),"code":metadata["code"],
                   "saved_early_state":payload.get("early_state"),"state_tensor_count":len(state),
                   "preprocessing":metadata["preprocessing"],"kind":kind}

def d_compare_saved_states(selected_path, final_path, split):
    """Measure saved weight/buffer changes; never load optimizer state into a model."""
    left,li=d_read_model_state(selected_path,split,"selected")
    right,ri=d_read_model_state(final_path,split,"final")
    if li["architecture"] != ri["architecture"] or set(left) != set(right):
        raise ValueError("Saved architectures/state keys differ.")
    groups={}
    for name,a in left.items():
        b=right[name]
        if a.shape != b.shape or a.dtype != b.dtype:
            raise ValueError("Saved tensor contract changed.")
        category=("crop_head" if name.startswith("crop_head.") else
                  "disease_head" if name.startswith("disease_head.") else
                  "backbone_buffers" if any(t in name for t in
                     ("running_mean","running_var","num_batches_tracked")) else "backbone")
        row=groups.setdefault(category,{"tensors":0,"changed_tensors":0,"squared_delta":0.,
                                        "selected_squared_norm":0.,"maximum_absolute_delta":0.})
        row["tensors"]+=1;row["changed_tensors"]+=int(not torch.equal(a,b))
        delta=b.double()-a.double()
        row["squared_delta"]+=float(delta.square().sum())
        row["selected_squared_norm"]+=float(a.double().square().sum())
        row["maximum_absolute_delta"]=max(row["maximum_absolute_delta"],float(delta.abs().max()))
    for row in groups.values():
        row["relative_l2_change"]=math.sqrt(row["squared_delta"])/max(math.sqrt(row["selected_squared_norm"]),1e-30)
    return {"selected":li,"final":ri,"groups":groups}


In [ ]:
def d_fold_report(crop_logits, disease_logits, frame, fitting, candidate_spec):
    """Component errors, every truth pair, fitting support and deterministic examples."""
    n=len(frame)
    independent=d_independent_scores(crop_logits,disease_logits,candidate_spec).cpu()
    scores=composition_scores(crop_logits,disease_logits,candidate_spec).cpu()
    if n < 1 or scores.shape[0] != n:
        raise ValueError("Nonempty rows and logits must match exactly.")
    ids,pairs,_=d_candidate_records(candidate_spec)
    id_to_pair=dict(zip(ids,pairs));id_to_col={v:i for i,v in enumerate(ids)}
    for row in frame.itertuples(index=False):
        if id_to_pair.get(int(row.composition_id)) != (int(row.crop_id),int(row.disease_id)):
            raise ValueError("Global truth ID and component labels disagree.")
    c,d=crop_logits.detach().double().cpu(),disease_logits.detach().double().cpu()
    clp=c-torch.logsumexp(c,1,keepdim=True);dlp=d-torch.logsumexp(d,1,keepdim=True)
    crank=torch.argsort(c,dim=1,descending=True,stable=True)
    drank=torch.argsort(d,dim=1,descending=True,stable=True)
    ranking=torch.argsort(scores,dim=1,descending=True,stable=True)
    id_tensor=torch.tensor(ids,dtype=torch.long)
    ranked_ids=id_tensor[ranking[:,:5]]
    truth_c=torch.tensor(frame.crop_id.to_numpy(),dtype=torch.long)
    truth_d=torch.tensor(frame.disease_id.to_numpy(),dtype=torch.long)
    truth_comp=torch.tensor(frame.composition_id.to_numpy(),dtype=torch.long)
    cols=torch.tensor([id_to_col[int(x)] for x in truth_comp],dtype=torch.long)
    comp1=ranked_ids[:,0]==truth_comp
    comp5=(ranked_ids==truth_comp[:,None]).any(1)
    crop1=crank[:,0]==truth_c;crop5=(crank[:,:5]==truth_c[:,None]).any(1)
    disease1=drank[:,0]==truth_d;disease5=(drank[:,:5]==truth_d[:,None]).any(1)
    fitting_ids=set(map(int,fitting.composition_id))
    predicted_fit=torch.tensor([int(x) in fitting_ids for x in ranked_ids[:,0]],dtype=torch.bool)
    top5_fit=sum(int(x) in fitting_ids for x in ranked_ids.flatten().tolist())
    support={task:{int(k):int(v) for k,v in fitting.groupby(task+"_id").size().items()}
             for task in ("crop","disease","composition")}
    crop_names={int(r.crop_id):str(r.crop_name) for r in fitting.itertuples()}
    disease_names={int(r.disease_id):str(r.disease_name) for r in fitting.itertuples()}
    def rate(values):
        count=int(values.sum())
        return {"hits":count,"sample_count":n,"accuracy":count/n}
    independent_rank=torch.argsort(independent,dim=1,descending=True,stable=True)
    raw=torch.stack([c[:,a]+d[:,b] for a,b in pairs],1)
    raw_rank=torch.argsort(raw,dim=1,descending=True,stable=True)
    summary={"sample_count":n,"crop_top1":rate(crop1),"crop_top5":rate(crop5),
             "disease_top1":rate(disease1),"disease_top5":rate(disease5),
             "composition_top1":rate(comp1),"composition_top5":rate(comp5),
             "joint_component_top1_hits":int((crop1 & disease1).sum()),
             "joint_component_top5_hits":int((crop5 & disease5).sum()),
             "correct_compositions_with_wrong_component_argmax":int((comp1 & ~(crop1 & disease1)).sum()),
             "predicted_fitting_composition_top1":rate(predicted_fit),
             "predicted_fitting_composition_top5_fraction":top5_fit/(n*min(5,len(ids))),
             "candidate_fitting_fraction":len(set(ids)&fitting_ids)/len(ids),
             "crop_only_wrong":int((~crop1 & disease1).sum()),
             "disease_only_wrong":int((crop1 & ~disease1).sum()),
             "both_components_wrong":int((~crop1 & ~disease1).sum()),
             "independent_score_max_absolute_difference":float((scores-independent).abs().max()),
             "full_ranking_matches_reference":bool(torch.equal(ranking,independent_rank)),
             "independent_top1_disagreements":int((ranking[:,0]!=independent_rank[:,0]).sum()),
             "independent_top5_set_disagreements":int((ranking[:,:5].sort(1).values!=independent_rank[:,:5].sort(1).values).any(1).sum()),
             "raw_sum_ranking_matches":bool(torch.equal(ranking,raw_rank)),
             "raw_sum_top1_disagreements":int((ranking[:,0]!=raw_rank[:,0]).sum()),
             "metric_authority":"Accepted log_softmax score formula; independent logsumexp/raw-sum are audits only."}
    arange=torch.arange(n)
    crop_ce=float(-clp[arange,truth_c].mean());disease_ce=float(-dlp[arange,truth_d].mean())
    summary["losses"]={"crop":crop_ce,"disease":disease_ce,"total":.5*(crop_ce+disease_ce)}
    summary["mean_true_crop_probability"]=float(clp[arange,truth_c].exp().mean())
    summary["mean_true_disease_probability"]=float(dlp[arange,truth_d].exp().mean())
    fitting_mask=torch.tensor([i in fitting_ids for i in ids],dtype=torch.bool)
    summary["mean_candidate_probability_mass_on_fitting_compositions"]=float(
        torch.softmax(scores,1)[:,fitting_mask].sum(1).mean())
    summary["mean_disease_entropy"]=float(-(dlp.exp()*dlp).sum(1).mean())
    summary["mean_crop_entropy"]=float(-(clp.exp()*clp).sum(1).mean())
    samples=[]
    for j,row in enumerate(frame.itertuples(index=False)):
        pred_ids=ranked_ids[j].tolist()
        true_column=id_to_col[int(row.composition_id)]
        true_rank=int((ranking[j]==true_column).nonzero()[0,0])+1
        error_type=("correct" if bool(comp1[j]) else
                    "crop_wrong" if not crop1[j] and disease1[j] else
                    "disease_wrong" if crop1[j] and not disease1[j] else "both_wrong")
        top_pairs=[{"global_id":i,"crop_id":id_to_pair[i][0],"disease_id":id_to_pair[i][1],
                    "crop":crop_names.get(id_to_pair[i][0],str(id_to_pair[i][0])),
                    "disease":disease_names.get(id_to_pair[i][1],str(id_to_pair[i][1])),
                    "fitting_composition":i in fitting_ids,
                    "score":float(scores[j,id_to_col[i]])} for i in pred_ids]
        samples.append({"image_name":str(row.image_name),"composition_id":int(row.composition_id),
            "crop_id":int(row.crop_id),"disease_id":int(row.disease_id),
            "crop_name":str(row.crop_name),"disease_name":str(row.disease_name),
            "crop_fit_support":support["crop"].get(int(row.crop_id),0),
            "disease_fit_support":support["disease"].get(int(row.disease_id),0),
            "composition_fit_support":support["composition"].get(int(row.composition_id),0),
            "crop_top1":bool(crop1[j]),"crop_top5":bool(crop5[j]),
            "disease_top1":bool(disease1[j]),"disease_top5":bool(disease5[j]),
            "composition_top1":bool(comp1[j]),"composition_top5":bool(comp5[j]),
            "predicted_global_id":pred_ids[0],"predicted_fitting_composition":bool(predicted_fit[j]),
            "crop_argmax":int(crank[j,0]),"disease_argmax":int(drank[j,0]),
            "true_pair_rank":true_rank,"top5_global_ids":pred_ids,"top5_pairs":top_pairs,
            "error_type":error_type,"true_pair_score":float(scores[j,true_column]),
            "top1_minus_true_score":float(scores[j,ranking[j,0]]-scores[j,true_column]),
            "true_disease_probability":float(dlp[j,truth_d[j]].exp()),
            "disease_argmax_probability":float(dlp[j,drank[j,0]].exp())})
    table=pd.DataFrame(samples)
    pair_rows=[]
    for global_id,group in table.groupby("composition_id",sort=True):
        first=group.iloc[0];crop_id=int(first.crop_id);disease_id=int(first.disease_id)
        disease_fit=fitting[fitting.disease_id==disease_id]
        count=len(group)
        pair_rows.append({"composition_id":int(global_id),"crop":first.crop_name,"disease":first.disease_name,
            "sample_count":count,"composition_top1_hits":int(group.composition_top1.sum()),
            "composition_top5_hits":int(group.composition_top5.sum()),
            "composition_top1":float(group.composition_top1.mean()),
            "composition_top5":float(group.composition_top5.mean()),
            "crop_top1":float(group.crop_top1.mean()),"disease_top1":float(group.disease_top1.mean()),
            "crop_fit_support":int(first.crop_fit_support),"disease_fit_support":int(first.disease_fit_support),
            "composition_fit_support":int(first.composition_fit_support),
            "disease_fitting_compositions":int(disease_fit.composition_id.nunique()),
            "disease_fitting_crops":int(disease_fit.crop_id.nunique()),
            "true_disease_fit_images_on_true_crop":int((disease_fit.crop_id==crop_id).sum()),
            "mean_true_pair_rank":float(group.true_pair_rank.mean()),
            "predicted_fitting_fraction":float(group.predicted_fitting_composition.mean())})
    component_rows=[]
    for task in ("crop","disease"):
        for label,group in table.groupby(task+"_id",sort=True):
            names=fitting.loc[fitting[task+"_id"]==label,task+"_name"]
            component_rows.append({"task":task,"class_id":int(label),"class_name":str(names.iloc[0]) if len(names) else str(label),
                "fit_support":support[task].get(int(label),0),"sample_count":len(group),
                "top1_hits":int(group[task+"_top1"].sum()),"top5_hits":int(group[task+"_top5"].sum()),
                "composition_top1_hits":int(group.composition_top1.sum())})
    bands=[]
    for task in ("crop","disease"):
        for low,high,label in ((0,5,"0-5"),(6,20,"6-20"),(21,100,"21-100"),(101,10**12,">100")):
            group=table[table[task+"_fit_support"].between(low,high)]
            bands.append({"task":task,"support_band":label,"sample_count":len(group),
                "component_top1_hits":int(group[task+"_top1"].sum()),
                "composition_top1_hits":int(group.composition_top1.sum())})
    # Deterministic representatives from each failure type, favoring different truth pairs.
    examples=[];used=set()
    wrong=sorted((x for x in samples if x["error_type"]!="correct"),
                 key=lambda x:(-x["top1_minus_true_score"],x["image_name"]))
    for category in ("disease_wrong","crop_wrong","both_wrong"):
        categories=[x for x in wrong if x["error_type"]==category]
        selected_pairs=set()
        for row in categories:
            if row["composition_id"] in selected_pairs:continue
            examples.append(row);used.add(row["image_name"]);selected_pairs.add(row["composition_id"])
            if len(selected_pairs)==2:break
    return {"summary":summary,"pairs":pair_rows,"components":component_rows,
            "support_bands":bands,"examples":examples,"samples":samples}

def d_collect_logits(model, loader, frame, device, *, precision="bf16",
                     channels_last=True, label="development"):
    """Only inference; verify returned target order, including final partial batches."""
    model.eval();device=torch.device(device)
    crops=[];diseases=[];offset=0
    with torch.inference_mode():
        for batch_number,(images,targets) in enumerate(loader,1):
            count=len(images);rows=frame.iloc[offset:offset+count]
            for key,column in (("crop","crop_id"),("disease","disease_id"),("composition","composition_id")):
                expected=torch.tensor(rows[column].to_numpy(),dtype=torch.long)
                if not torch.equal(targets[key].cpu(),expected):
                    raise ValueError("Inference loader reordered or changed truth labels.")
            images=images.to(device,non_blocking=device.type=="cuda")
            if channels_last and images.ndim==4:
                images=images.contiguous(memory_format=torch.channels_last)
            with amp_context(device,precision):
                output=model(images)
            crops.append(output["crop_logits"].detach().to(device="cpu", dtype=torch.float32, copy=True))
            diseases.append(output["disease_logits"].detach().to(device="cpu", dtype=torch.float32, copy=True))
            offset+=count
            if batch_number%20==0 or offset==len(frame):
                print(f"M5D {label}: {offset}/{len(frame)} inference images",flush=True)
    if offset!=len(frame) or not crops:
        raise ValueError("Incomplete or empty development inference.")
    c,d=torch.cat(crops),torch.cat(diseases)
    if c.shape!=(len(frame),55) or d.shape!=(len(frame),175) or not torch.isfinite(c).all() or not torch.isfinite(d).all():
        raise ValueError("Diagnostic logits violate the model contract.")
    return c,d


In [ ]:
def d_run_profile(path, kind, split, image_directory, destination, *, resolution=None,
                  device="cuda:0", batch_size=16, num_workers=2):
    """Evaluate one immutable saved state on exactly the two development folds."""
    state,info=d_read_model_state(path,split,kind)
    cfg=info["configuration"]
    selected_resolution=info["native_resolution"] if resolution is None else int(resolution)
    label=f"{info['architecture']}_{kind}_epoch{info['epoch']}_res{selected_resolution}"
    directory=Path(destination)/label;directory.mkdir(exist_ok=False)
    model=create_multitask_model(info["architecture"]) # weights=None; saved state only
    model.load_state_dict(state,strict=True)
    before=d_state_digest(model.state_dict());del state
    device=torch.device(device)
    model.to(device).eval().requires_grad_(False)
    if cfg["channels_last"]:model.to(memory_format=torch.channels_last)
    spec={"ids":split.candidate_ids,"pairs":split.candidate_pairs,"sha256":split.candidate_sha256}
    started=time.perf_counter();folds={}
    if device.type=="cuda":
        torch.cuda.reset_peak_memory_stats(device)
    for fold,frame in (("seen",split.val_seen),("unseen",split.val_unseen)):
        dataset=DPDataset(frame,image_directory,image_size=selected_resolution,
                          as_dict_targets=True,augment=False,normalize_imagenet=True)
        loader=DataLoader(dataset,batch_size=batch_size,shuffle=False,num_workers=num_workers,
                          pin_memory=device.type=="cuda",generator=torch.Generator().manual_seed(split.seed),
                          worker_init_fn=seed_worker)
        c,d=d_collect_logits(model,loader,frame,device,precision=cfg["precision"],
                             channels_last=cfg["channels_last"],label=label+"/"+fold)
        report=d_fold_report(c,d,frame,split.fit,spec)
        folds[fold]=report["summary"]
        # Plain NumPy prediction arrays (never pickle/weights) allow an independent reviewer
        # to recompute every metric without access to images or GPU.
        np.savez_compressed(directory/(fold+"_logits.npz"),crop_logits=c.numpy(),disease_logits=d.numpy(),
            crop_truth=frame.crop_id.to_numpy(dtype=np.int64),
            disease_truth=frame.disease_id.to_numpy(dtype=np.int64),
            composition_truth=frame.composition_id.to_numpy(dtype=np.int64))
        for table in ("pairs","components","support_bands"):
            pd.DataFrame(report[table]).to_csv(directory/(fold+"_"+table+".csv"),index=False)
        with (directory/(fold+"_examples.json")).open("x") as stream:
            json.dump(report["examples"],stream,indent=2,allow_nan=False)
        # Avoid storing repeated nested scores on all3132 rows; detailed top5 examples remain JSON.
        samples=pd.DataFrame(report["samples"])
        samples["top5_global_ids"]=samples.top5_global_ids.map(json.dumps)
        samples.drop(columns=["top5_pairs"]).to_csv(directory/(fold+"_samples.csv"),index=False)
        del loader,dataset,c,d,report
    if device.type=="cuda":torch.cuda.synchronize(device)
    after=d_state_digest(model.state_dict())
    if before!=after or any(p.grad is not None for p in model.parameters()):
        raise RuntimeError("Inference unexpectedly modified parameters/buffers/gradients.")
    metrics={f"{fold}_top{k}":folds[fold][f"composition_top{k}"]["accuracy"]
             for fold in ("seen","unseen") for k in (1,5)}
    for k in (1,5):
        a,b=metrics[f"seen_top{k}"],metrics[f"unseen_top{k}"]
        metrics[f"top{k}_harmonic_mean"]=0. if a+b==0 else 2*a*b/(a+b)
    result={"profile":label,"state":info,"evaluated_resolution":selected_resolution,
            "resolution_role":"native" if selected_resolution==info["native_resolution"] else "matched_final_control",
            "folds":folds,"metrics":metrics,"elapsed_seconds":time.perf_counter()-started,
            "batch_size":batch_size,"num_workers":num_workers,"device":str(device),
            "precision":cfg["precision"],"parameters_buffers_unchanged":before==after,
            "model_state_sha256_before":before,"model_state_sha256_after":after,
            "peak_allocated_mib":torch.cuda.max_memory_allocated(device)/2**20 if device.type=="cuda" else None,
            "checkpoint_decision_changed":False}
    with (directory/"profile_summary.json").open("x") as stream:
        json.dump(result,stream,indent=2,allow_nan=False)
    del model;gc.collect()
    if device.type=="cuda":torch.cuda.empty_cache()
    print("M5D PROFILE COMPLETE",json.dumps({"profile":label,"metrics":metrics}),flush=True)
    return result

def d_run_diagnostics(run_directories, split, image_directory, output_root, *, device="cuda:0"):
    """Six inference-only profiles and a read-only audit of the completed histories."""
    audit=d_audit_contract(split)
    if audit["image_counts"]!={"fit":4868,"seen":1238,"unseen":1894} or audit["candidate_count"]!=323:
        raise ValueError("M5D expects the accepted full training-derived split.")
    summaries=[json.loads((Path(d)/"run_summary.json").read_text()) for d in run_directories]
    if (len(summaries)!=2 or {s["architecture"] for s in summaries}!=
            {"multitask_efficientnet_b3","multitask_convnext_base"}):
        raise ValueError("Specify exactly one existing run per final architecture.")
    identifier=datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%SZ")+"_"+uuid.uuid4().hex[:8]
    directory=Path(output_root)/identifier;directory.mkdir(parents=True,exist_ok=False)
    with (directory/"contract_audit.json").open("x") as stream:
        json.dump(audit,stream,indent=2,allow_nan=False)
    history_audits=[];state_changes=[];profiles=[]
    for original,summary in zip(run_directories,summaries):
        history=d_audit_history(summary)
        selected_path=Path(summary["selected"]["path"])
        final_path=Path(original)/"last_completed.pt"
        changes=d_compare_saved_states(selected_path,final_path,split)
        if changes["final"]["saved_early_state"]!=history["fine_state"]:
            raise ValueError("Saved early-stopping state disagrees with historical replay.")
        if changes["final"]["epoch"]!=summary["completed_epochs"]:
            raise ValueError("Final recovery epoch disagrees with the run summary.")
        history_audits.append({"architecture":summary["architecture"],**history})
        state_changes.append(changes)
        # Preserve original records by reading only; write small derived copies to diagnostics.
        with (directory/(summary["architecture"]+"_original_summary.json")).open("x") as stream:
            json.dump(summary,stream,indent=2,allow_nan=False)
        for kind,path,resolution in (
            ("selected",selected_path,None),("final",final_path,None),
            ("selected",selected_path,summary["configuration"]["final_resolution"])):
            profiles.append(d_run_profile(path,kind,split,image_directory,directory,
                                          resolution=resolution,device=device))
    original_checks=[]
    for summary in summaries:
        for kind in ("selected","final"):
            profile=next(p for p in profiles if p["state"]["architecture"]==summary["architecture"]
                         and p["state"]["kind"]==kind and p["resolution_role"]=="native")
            recorded=(summary["selected"]["metrics"] if kind=="selected" else summary["history"][-1])
            original_checks.append({"architecture":summary["architecture"],"kind":kind,
                "metrics":[{"name":name,"recorded":recorded[name],"recomputed":profile["metrics"][name],
                            "difference":profile["metrics"][name]-recorded[name]} for name in profile["metrics"]]})
    result={"directory":str(directory),"contract":audit,"history_audits":history_audits,
            "state_comparisons":state_changes,"profiles":profiles,"historical_metric_reproduction":original_checks,
            "image_evaluations":sum(p["folds"]["seen"]["sample_count"]+p["folds"]["unseen"]["sample_count"] for p in profiles),
            "training_steps":0,"official_test_images":0,"selection_decisions_changed":False}
    with (directory/"diagnostic_results.json").open("x") as stream:
        json.dump(result,stream,indent=2,allow_nan=False)
    rows=[{"profile":p["profile"],"resolution":p["evaluated_resolution"],**p["metrics"],
           "unseen_crop_top1":p["folds"]["unseen"]["crop_top1"]["accuracy"],
           "unseen_disease_top1":p["folds"]["unseen"]["disease_top1"]["accuracy"],
           "unseen_predicted_fitting":p["folds"]["unseen"]["predicted_fitting_composition_top1"]["accuracy"]}
          for p in profiles]
    pd.DataFrame(rows).to_csv(directory/"comparison.csv",index=False)
    print(pd.DataFrame(rows).to_string(index=False),flush=True)
    return result

def d_show_error_images(results, image_directory, *, destination=None):
    """Optional small gallery of original development errors, never official images."""
    import matplotlib.pyplot as plt
    directory=Path(results["directory"])
    selected=next(p for p in results["profiles"] if p["state"]["architecture"]=="multitask_convnext_base"
                  and p["state"]["kind"]=="selected" and p["resolution_role"]=="native")
    examples=json.loads((directory/selected["profile"]/"unseen_examples.json").read_text())[:6]
    figure,axes=plt.subplots(2,3,figsize=(16,10))
    for axis in axes.flat:axis.axis("off")
    for axis,row in zip(axes.flat,examples):
        with Image.open(Path(image_directory)/row["image_name"]) as image:
            axis.imshow(image.convert("RGB"))
        best=row["top5_pairs"][0]
        axis.set_title(f"Truth: {row['crop_name']} / {row['disease_name']}\n"
                       f"Top1: {best['crop']} / {best['disease']}\n"
                       f"true pair rank {row['true_pair_rank']}",fontsize=9)
    figure.tight_layout()
    if destination is not None:figure.savefig(destination,dpi=110)
    plt.show()
    return figure


In [ ]:
def d_run_recorded_batch_checks(run_directories, split, image_directory, output_root, *, device="cuda:0"):
    """Repeat only selected/native inference with the original warm-up physical batch."""
    directory=Path(output_root)/("recorded_batch_"+datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%SZ")+"_"+uuid.uuid4().hex[:8])
    directory.mkdir(parents=True,exist_ok=False)
    checks=[]
    for run in run_directories:
        summary=json.loads((Path(run)/"run_summary.json").read_text())
        selected=summary["selected"]
        row=next(r for r in summary["history"] if r["epoch"]==selected["epoch"])
        profile=d_run_profile(selected["path"],"selected",split,image_directory,directory,
                              batch_size=row["physical_microbatch"],device=device)
        profile["historical_metrics"]=selected["metrics"]
        profile["metric_differences"]={k:profile["metrics"][k]-selected["metrics"][k]
                                       for k in profile["metrics"]}
        checks.append(profile)
    result={"directory":str(directory),"profiles":checks,"image_evaluations":
            sum(sum(p["folds"][f]["sample_count"] for f in ("seen","unseen")) for p in checks),
            "training_steps":0,"selection_decisions_changed":False}
    with (directory/"recorded_batch_checks.json").open("x") as stream:
        json.dump(result,stream,indent=2,allow_nan=False)
    return result

def d_support_structure(split):
    """Describe fitting support for held-out components without using images or tests."""
    rows=[]
    for global_id,group in split.val_unseen.groupby("composition_id",sort=True):
        row=group.iloc[0]
        disease=split.fit[split.fit.disease_id==row.disease_id]
        crop=split.fit[split.fit.crop_id==row.crop_id]
        rows.append({"composition_id":int(global_id),"crop":str(row.crop_name),"disease":str(row.disease_name),
                     "unseen_images":len(group),"crop_fit_support":len(crop),
                     "disease_fit_support":len(disease),"disease_fit_crops":int(disease.crop_id.nunique()),
                     "disease_fit_compositions":int(disease.composition_id.nunique()),
                     "same_crop_disease_fit_images":int((disease.crop_id==row.crop_id).sum())})
    table=pd.DataFrame(rows)
    result={"pairs":rows,"unseen_images":len(split.val_unseen),
            "unseen_pair_count":len(table),"fitting_composition_count":int(split.fit.composition_id.nunique()),
            "all_components_have_fitting_examples":bool((table.disease_fit_support>0).all() and (table.crop_fit_support>0).all()),
            "all_heldout_pairs_absent_from_fitting":bool((table.same_crop_disease_fit_images==0).all())}
    for task in ("crop","disease"):
        support=split.fit.groupby(task+"_id").size()
        result[task+"_fit_support"]={"minimum":int(support.min()),"median":float(support.median()),
                                    "maximum":int(support.max()),"classes":len(support)}
        result[task+"_heldout_image_support_bands"]={
            label:int(table.loc[table[task+"_fit_support"].between(low,high),"unseen_images"].sum())
            for low,high,label in ((0,5,"0-5"),(6,20,"6-20"),(21,100,"21-100"),(101,10**12,">100"))}
    result["unseen_images_whose_disease_fits_one_crop"]=int(table.loc[table.disease_fit_crops==1,"unseen_images"].sum())
    return result

def d_review_prediction_packet(profile_directory, split):
    """Recompute stored evidence using no images/GPU and allow_pickle=False."""
    directory=Path(profile_directory)
    stored=json.loads((directory/"profile_summary.json").read_text())
    spec={"ids":split.candidate_ids,"pairs":split.candidate_pairs,"sha256":split.candidate_sha256}
    checks={}
    for fold,frame in (("seen",split.val_seen),("unseen",split.val_unseen)):
        with np.load(directory/(fold+"_logits.npz"),allow_pickle=False) as arrays:
            for key,column in (("crop_truth","crop_id"),("disease_truth","disease_id"),("composition_truth","composition_id")):
                if not np.array_equal(arrays[key],frame[column].to_numpy()):
                    raise ValueError("Saved evidence truth order changed.")
            report=d_fold_report(torch.from_numpy(arrays["crop_logits"].copy()),
                                 torch.from_numpy(arrays["disease_logits"].copy()),frame,split.fit,spec)
        if report["summary"]!=stored["folds"][fold]:
            raise ValueError("Recomputed evidence differs from its profile summary.")
        checks[fold]={"sample_count":report["summary"]["sample_count"],"all_summary_fields_match":True}
    return {"profile":stored["profile"],"checks":checks}


In [ ]:
def d_evidence_synopsis(results):
    """Small descriptive tables from the already saved diagnostic reports."""
    directory=Path(results["directory"])
    rows=[]
    for profile in results["profiles"]:
        pairs=pd.read_csv(directory/profile["profile"]/"unseen_pairs.csv")
        unseen=profile["folds"]["unseen"]
        rows.append({"profile":profile["profile"],"heldout_pairs":len(pairs),
                     "pairs_with_zero_top1_hits":int((pairs.composition_top1_hits==0).sum()),
                     "pairs_with_any_top5_hit":int((pairs.composition_top5_hits>0).sum()),
                     "macro_pair_top1":float(pairs.composition_top1.mean()),
                     "micro_top1":unseen["composition_top1"]["accuracy"],
                     "joint_component_top1_hits":unseen["joint_component_top1_hits"],
                     "crop_only_wrong":unseen["crop_only_wrong"],
                     "disease_only_wrong":unseen["disease_only_wrong"],
                     "both_wrong":unseen["both_components_wrong"],
                     "correct_compositions_with_wrong_component_argmax":
                         unseen["correct_compositions_with_wrong_component_argmax"],
                     "best_pairs":pairs.sort_values(["composition_top1","sample_count"],
                         ascending=[False,False]).head(5).to_dict("records"),
                     "largest_failed_pairs":pairs[pairs.composition_top1_hits==0].sort_values(
                         ["sample_count","composition_id"],ascending=[False,True]).head(5).to_dict("records")})
    return {"profiles":rows}

def d_plot_diagnostic_histories(results):
    """Plots use only completed original histories; no new experiment is implied."""
    import matplotlib.pyplot as plt
    directory=Path(results["directory"])
    figure,axes=plt.subplots(2,2,figsize=(13,8))
    for column,architecture in enumerate(("multitask_efficientnet_b3","multitask_convnext_base")):
        summary=json.loads((directory/(architecture+"_original_summary.json")).read_text())
        history=pd.DataFrame(summary["history"])
        for fold in ("train","seen","unseen"):
            axes[0,column].plot(history.epoch,history[fold+"_disease_loss"],label=fold)
        axes[0,column].set_title(architecture+" disease cross-entropy")
        axes[0,column].set_ylabel("sample-weighted CE");axes[0,column].legend()
        for name in ("seen_top1","unseen_top1","top1_harmonic_mean"):
            axes[1,column].plot(history.epoch,history[name]*100,label=name)
        axes[1,column].axvline(summary["selected"]["epoch"],color="black",linestyle=":",label="selected epoch")
        axes[1,column].set_xlabel("completed epoch");axes[1,column].set_ylabel("accuracy / H@1 (%)")
        axes[1,column].legend()
    figure.tight_layout()
    figure.savefig(directory/"diagnostic_history.svg")
    plt.close(figure)
    return str(directory/"diagnostic_history.svg")


In [ ]:
def d_audit_numerical_ties(profile_directory, split):
    """Expose floating-point Top5 boundary sensitivity without changing metric rankings."""
    directory=Path(profile_directory)
    spec={"ids":split.candidate_ids,"pairs":split.candidate_pairs,"sha256":split.candidate_sha256}
    ids,pairs,_=d_candidate_records(spec)
    id_tensor=torch.tensor(ids)
    rows=[]
    for fold,frame in (("seen",split.val_seen),("unseen",split.val_unseen)):
        with np.load(directory/(fold+"_logits.npz"),allow_pickle=False) as data:
            c=torch.from_numpy(data["crop_logits"].copy()).double()
            d=torch.from_numpy(data["disease_logits"].copy()).double()
            truth=torch.from_numpy(data["composition_truth"].copy())
        accepted=composition_scores(c,d,spec)
        independent=d_independent_scores(c,d,spec)
        ar=torch.argsort(accepted,dim=1,descending=True,stable=True)
        ir=torch.argsort(independent,dim=1,descending=True,stable=True)
        different=(ar[:,:5].sort(1).values!=ir[:,:5].sort(1).values).any(1)
        raw=torch.stack([c[:,a]+d[:,b] for a,b in pairs],1)
        for i in different.nonzero().flatten().tolist():
            a_ids=id_tensor[ar[i,:5]].tolist();i_ids=id_tensor[ir[i,:5]].tolist()
            boundary_columns=set(ar[i,:6].tolist())|set(ir[i,:6].tolist())
            rows.append({"fold":fold,"image_name":str(frame.iloc[i].image_name),
                "truth":int(truth[i]),"accepted_top5":a_ids,"independent_top5":i_ids,
                "accepted_hit":int(truth[i]) in a_ids,"independent_hit":int(truth[i]) in i_ids,
                "accepted_boundary_gap":float(accepted[i,ar[i,4]]-accepted[i,ar[i,5]]),
                "raw_boundary_gap":float(raw[i,ar[i,4]]-raw[i,ar[i,5]]),
                "boundary_scores":[{"global_id":ids[j],"accepted":float(accepted[i,j]),
                                    "independent":float(independent[i,j]),"raw_sum":float(raw[i,j])}
                                   for j in sorted(boundary_columns)]})
    return {"profile":directory.name,"top5_candidate_set_disagreement_rows":len(rows),
            "top5_hit_disagreement_rows":sum(r["accepted_hit"]!=r["independent_hit"] for r in rows),
            "rows":rows,"policy_changed":False}


In [ ]:
def d_preprocessing_audit(info):
    """Compare recorded preprocessing descriptions with the actual saved epoch resolution."""
    resolution=info["native_resolution"]
    metadata=info["preprocessing"]
    actual_validation=repr(get_transforms(resolution,augment=False,normalize_imagenet=True))
    size_matches=metadata.get("image_size")==[resolution,resolution]
    transform_matches=metadata.get("validation_transform")==actual_validation
    return {"architecture":info["architecture"],"kind":info["kind"],"epoch":info["epoch"],
            "actual_epoch_resolution":resolution,"recorded_image_size":metadata.get("image_size"),
            "recorded_validation_transform":metadata.get("validation_transform"),
            "expected_validation_transform":actual_validation,
            "image_size_describes_actual_epoch":size_matches,
            "validation_description_matches_actual_loader":transform_matches,
            "metadata_consistent":size_matches and transform_matches,
            "original_record_modified":False,
            "interpretation":"Loader resolution is verified from epoch history/selection; metadata is not rewritten."}


In [ ]:
def d_prediction_support_summary(profile_directory, split):
    """Compare predicted component frequencies directly with fitting class support."""
    directory=Path(profile_directory);result={}
    for fold in ("seen","unseen"):
        samples=pd.read_csv(directory/(fold+"_samples.csv"))
        tasks={}
        for task in ("crop","disease"):
            fitting=split.fit.groupby(task+"_id").size()
            truth=samples[task+"_id"].value_counts()
            predicted=samples[task+"_argmax"].value_counts()
            correct=samples.loc[samples[task+"_argmax"]==samples[task+"_id"],task+"_id"].value_counts()
            names=split.fit[[task+"_id",task+"_name"]].drop_duplicates().set_index(task+"_id")[task+"_name"]
            rows=[{"class_id":int(i),"name":str(names.loc[i]),"fit_images":int(n),
                   "validation_images":int(truth.get(i,0)),"predicted_argmax_images":int(predicted.get(i,0)),
                   "correct_argmax_images":int(correct.get(i,0))} for i,n in fitting.items()]
            if sum(r["predicted_argmax_images"] for r in rows)!=len(samples):
                raise ValueError("Predicted class is absent from the fitting component universe.")
            mean_true=float(samples[task+"_id"].map(fitting).mean())
            mean_pred=float(samples[task+"_argmax"].map(fitting).mean())
            tasks[task]={"sample_count":len(samples),"mean_true_class_fitting_support":mean_true,
                         "mean_predicted_class_fitting_support":mean_pred,
                         "most_predicted_classes":sorted(rows,key=lambda r:(-r["predicted_argmax_images"],r["class_id"]))[:5],
                         "classes":rows}
        result[fold]=tasks
    return {"profile":directory.name,"folds":result}


In [ ]:
M5D_RESULTS = None
if RUN_M5D_DIAGNOSTICS:
    # Build only from the training CSV. No training, optimizer or acquisition is invoked.
    diagnostic_split = build_development_split(ROOT / "DPD_Small_train.csv", seed=42)
    M5D_RESULTS = d_run_diagnostics(M5D_RUN_DIRECTORIES, diagnostic_split,
                                   IMAGE_DIRECTORY, M5D_OUTPUT_ROOT, device=CONFIG["device"])
    write_new_json(Path(M5D_RESULTS["directory"])/"evidence_synopsis.json", d_evidence_synopsis(M5D_RESULTS))
    d_plot_diagnostic_histories(M5D_RESULTS)
    tie_audits = [d_audit_numerical_ties(Path(M5D_RESULTS["directory"])/p["profile"], diagnostic_split) for p in M5D_RESULTS["profiles"]]
    write_new_json(Path(M5D_RESULTS["directory"])/"numerical_tie_audits.json", tie_audits)
    write_new_json(Path(M5D_RESULTS["directory"])/"preprocessing_audits.json", [d_preprocessing_audit(p["state"]) for p in M5D_RESULTS["profiles"]])
    for profile in M5D_RESULTS["profiles"]:
        profile_path = Path(M5D_RESULTS["directory"])/profile["profile"]
        write_new_json(profile_path/"prediction_support.json", d_prediction_support_summary(profile_path, diagnostic_split))
    support_structure = d_support_structure(diagnostic_split)
    write_new_json(Path(M5D_RESULTS["directory"])/"support_structure.json", support_structure)
else:
    print("M5D inference diagnostics disabled. Set only RUN_M5D_DIAGNOSTICS=True to run.")
if SHOW_M5D_ERROR_IMAGES:
    if M5D_RESULTS is None:
        print("No diagnostic results in this kernel; load an existing diagnostic_results.json explicitly.")
    else:
        d_show_error_images(M5D_RESULTS, IMAGE_DIRECTORY)

if RUN_M5D_RECORDED_BATCH_CHECK:
    recorded_split = build_development_split(ROOT / "DPD_Small_train.csv", seed=42)
    M5D_RECORDED_BATCH_RESULTS = d_run_recorded_batch_checks(
        M5D_RUN_DIRECTORIES, recorded_split, IMAGE_DIRECTORY, M5D_OUTPUT_ROOT, device=CONFIG["device"])
    write_new_json(Path(M5D_RECORDED_BATCH_RESULTS["directory"])/"numerical_tie_audits.json",
                   [d_audit_numerical_ties(Path(M5D_RECORDED_BATCH_RESULTS["directory"])/p["profile"], recorded_split) for p in M5D_RECORDED_BATCH_RESULTS["profiles"]])
    write_new_json(Path(M5D_RECORDED_BATCH_RESULTS["directory"])/"preprocessing_audits.json", [d_preprocessing_audit(p["state"]) for p in M5D_RECORDED_BATCH_RESULTS["profiles"]])
    for profile in M5D_RECORDED_BATCH_RESULTS["profiles"]:
        profile_path = Path(M5D_RECORDED_BATCH_RESULTS["directory"])/profile["profile"]
        write_new_json(profile_path/"prediction_support.json", d_prediction_support_summary(profile_path, recorded_split))


### M5E — Correct provenance and isolate one sampler intervention

The active preparation is exactly two ConvNeXt-Base pilots: unchanged uniform fitting shuffle
and capped disease tickets. Both start fresh from the verified ImageNet V1 artifact with seed42
and identical initial heads. Both use2 head-warm-up +15 fine-tuning epochs, the same 224→320
schedule, AdamW/cosine LRs, augmentation, effective batch64 and unchanged equal task CE.
The old long two-backbone launch is blocked. Historical files are read only and unchanged.

For fitting disease count n_d and maximum fitting count M, ideal weight is
min(sqrt(M/n_d),cap); effective tickets are floor(weight), at least1. The default cap is3.
Uniformly choose N unique tickets from the finite pool each epoch. The expected disease
probability is n_d*t_d / sum_j(n_j*t_j), and any individual image appears at most3 times.
This is partial balancing, not equal disease probabilities. Each arm draws exactlyNfitting
samples per epoch; the balanced arm can omit some images and repeat others. Record both
expected and observed sampling, including unique coverage. Validation is sequential/unchanged.

New metadata version 2 separates the planned schedule from actual epoch resolution and transform.
Legacy version 1 state loading remains compatible; legacy inference requires explicit
legacy_policy="recorded_epoch_resolution" (or an explicit reviewed resolution if no epoch
resolution was recorded). Original metadata is returned unchanged and inconsistencies are
disclosed. No historical checkpoint or result is rewritten.

Early stopping now uses global best H@1 from warm-up onward. Warm-up never consumes patience.
Fine-tuning consumes one bad epoch for every exact H@1 tie or decrease. A lower validation loss
can win the checkpoint tie-break without resetting H@1 patience. Only a strictly higher H@1
resets it; no tolerance is used. Grace is a minimum number of fine-tuning epochs. The paired
pilot has 15 fine epochs and 15grace with patience20, so both complete the same 17-epoch budget
unless interrupted or invalid. Interrupted/unequal runs cannot produce a paired success claim.

Each completed epoch saves component Top1/5, canonical composition Top1/5/H@1/H@5, per-pair
unseen hits/accuracies, healthy frequency, fitting-composition preference, losses, actual sampling,
time, GPU memory and update counts. Comparison requires equal actual sample/update budgets
and identical initial-model hashes. Improved training loss alone is insufficient. A favorable
H@1/unseen-disease/unseen-composition signal still requires review and repeated evidence.

Owner execution: leave all historical training flags False; review M5E_PILOT_PLAN, then explicitly
set only RUN_M5E_PILOT=True and run sequentially in VS Code WSL. RUN_M5E_SMOKE is separate
and defaultFalse. No pilot experiment was run during preparation. New run directories are
outside the historical wp2directory. Do not compare pilot results with old runs as a sampler-only
effect: stopping, budget and corrected provenance are shared changes to both new arms.


In [ ]:
def resolve_checkpoint_preprocessing(info, *, legacy_policy=None, legacy_resolution=None):
    """Resolve input resolution without changing the raw checkpoint metadata.

    Legacy model-state loading stays available. Legacy inference requires an
    explicit policy acknowledging the old record, never a silent resolution guess.
    """
    raw=info.get("preprocessing") or {}
    metadata=info.get("metadata",info.get("base_metadata",{}))
    recorded=(info.get("completed_resolution") if "completed_resolution" in info else
              metadata.get("selection",{}).get("resolution"))
    if raw.get("metadata_version")==2:
        if raw.get("scope")!="epoch_checkpoint":
            raise ValueError("A run plan is not authoritative checkpoint preprocessing.")
        resolution=raw.get("actual_resolution")
        if type(resolution) is not int or resolution<1:
            raise ValueError("Missing actual epoch resolution.")
        if raw.get("image_size")!=[resolution,resolution]:
            raise ValueError("Actual resolution and image_size disagree.")
        expected=repr(get_transforms(resolution,augment=False,normalize_imagenet=True))
        if raw.get("validation_transform")!=expected:
            raise ValueError("Actual validation transform description is inconsistent.")
        if recorded is not None and recorded!=resolution:
            raise ValueError("Epoch selection/completion and preprocessing resolutions disagree.")
        cfg=metadata.get("configuration")
        if cfg is not None:
            if info.get("architecture") is not None and cfg["architecture"]!=info["architecture"]:
                raise ValueError("Checkpoint/configuration architecture mismatch.")
            epoch=raw.get("epoch")
            _,_,actual=stage_for_epoch(cfg,epoch)
            if actual!=resolution or metadata.get("epoch",epoch)!=epoch:
                raise ValueError("Epoch schedule disagrees with checkpoint resolution.")
        selected=raw.get("actual_selected_checkpoint_resolution")
        if selected is not None and "completed_resolution" not in info and selected!=resolution:
            raise ValueError("Portable selection resolution is inconsistent.")
        return {"actual_resolution":resolution,"source":"epoch_specific_v2","legacy":False,
                "validation_transform":expected}
    if legacy_policy=="recorded_epoch_resolution":
        resolution=recorded
        source="legacy_recorded_epoch_resolution"
    elif legacy_policy=="explicit_resolution":
        resolution=legacy_resolution
        source="legacy_explicit_resolution"
        if recorded is not None and resolution!=recorded:
            raise ValueError("Explicit resolution contradicts authoritative legacy epoch metadata.")
    else:
        raise ValueError("Legacy inference requires an explicit resolution policy; raw metadata is preserved.")
    if type(resolution) is not int or resolution<1:
        raise ValueError("Legacy record has no authoritative resolution; supply an explicit reviewed resolution.")
    expected=repr(get_transforms(resolution,augment=False,normalize_imagenet=True))
    return {"actual_resolution":resolution,"source":source,"legacy":True,
            "validation_transform":expected,
            "legacy_inconsistency_acknowledged":raw.get("image_size")!=[resolution,resolution]
                or raw.get("validation_transform")!=expected,
            "stored_preprocessing":copy.deepcopy(raw)}

def checkpoint_inference_transform(info, *, legacy_policy=None, legacy_resolution=None):
    """Build the actual nonaugmented RGB tensor transform from resolved provenance."""
    prior=info.get("inference_preprocessing",{})
    if legacy_policy is None and prior.get("source")=="legacy_recorded_epoch_resolution":
        legacy_policy="recorded_epoch_resolution"
    elif legacy_policy is None and prior.get("source")=="legacy_explicit_resolution":
        legacy_policy="explicit_resolution";legacy_resolution=prior["actual_resolution"]
    resolved=resolve_checkpoint_preprocessing(info,legacy_policy=legacy_policy,
                                              legacy_resolution=legacy_resolution)
    return get_transforms(resolved["actual_resolution"],augment=False,normalize_imagenet=True)


In [ ]:
def fitting_sampling_plan(split, config):
    """Use ONLY split.fit; integer tickets give an exact finite-pool distribution.

    Ideal per-image weight is min(sqrt(max_disease_count / count_d), cap).
    Effective tickets are floor(weight), bounded from1 to cap. Draw N of T
    tickets without replacement: P(disease)=count_d*tickets_d/T exactly in
    expectation, and any image can appear at most cap times in an epoch.
    """
    mode=config.get("fitting_sampler","uniform_shuffle")
    cap=config.get("sampler_weight_cap",3)
    if mode not in ("uniform_shuffle","capped_disease_tickets"):
        raise ValueError("Unknown fitting sampler.")
    if type(cap) is not int or not 1<=cap<=4:
        raise ValueError("sampler_weight_cap must be an integer from1 to4.")
    fit=split.fit
    if fit.empty or fit.image_name.duplicated().any():
        raise ValueError("Fitting rows must be nonempty with unique image names.")
    labels=fit.disease_id.to_numpy()
    if any(not isinstance(x,Integral) or not 0<=int(x)<175 for x in labels):
        raise ValueError("Fitting disease IDs must retain the original0..174labels.")
    counts=fit.groupby("disease_id").size()
    maximum=int(counts.max())
    rows=[];tickets_by_disease={}
    for disease,count in counts.items():
        ideal=min(math.sqrt(maximum/int(count)),cap)
        tickets=max(1,min(cap,math.floor(ideal))) if mode=="capped_disease_tickets" else 1
        tickets_by_disease[int(disease)]=tickets
        rows.append({"disease_id":int(disease),"fitting_count":int(count),
                     "capped_inverse_sqrt_weight":ideal if mode=="capped_disease_tickets" else 1.,
                     "tickets_per_image":tickets})
    tickets=[tickets_by_disease[int(d)] for d in labels]
    pool=sum(tickets);n=len(fit)
    for row in rows:
        mass=row["fitting_count"]*row["tickets_per_image"]
        row["effective_disease_probability"]=mass/pool
        row["expected_epoch_draws"]=n*mass/pool
        row["ratio_to_control_probability"]=n*row["tickets_per_image"]/pool
        row["per_image_inclusion_expected_repeats"]=n*row["tickets_per_image"]/pool
    identity=[[str(row.image_name),int(row.crop_id),int(row.disease_id),int(row.composition_id)]
              for row in fit.itertuples(index=False)]
    return {"mode":mode,"formula":"tickets=floor(min(sqrt(max_fitting_disease_count/n_d),cap))",
            "cap":cap,"reference_disease_count":maximum,"samples_per_epoch":n,"pool_size":pool,"tickets_per_row":tickets,
            "maximum_image_repeats":max(tickets),"diseases":rows,
            "fitting_rows_sha256":hashlib.sha256(json.dumps(identity,separators=(",",":")).encode()).hexdigest(),
            "sampling_seed":int(config["seed"]),"generator":"shared fitting DataLoader generator",
            "source":"fitting rows only; no validation/test statistics",
            "draw":"uniform tickets without replacement" if mode=="capped_disease_tickets" else
                   "unchanged RandomSampler without replacement"}

class CappedDiseaseTicketSampler(torch.utils.data.Sampler):
    """Finite integer-ticket sampler with a hard per-image repetition limit."""
    def __init__(self, plan, *, generator):
        if plan["mode"]!="capped_disease_tickets":
            raise ValueError("Use the ordinary shuffled loader for control.")
        self.plan=copy.deepcopy(plan)
        self.generator=generator
        counts=torch.tensor(plan["tickets_per_row"],dtype=torch.long)
        if (counts<1).any() or (counts>plan["cap"]).any() or not 1<=plan["cap"]<=4:
            raise ValueError("Ticket multiplicities violate the configured cap.")
        self.pool=torch.repeat_interleave(torch.arange(len(counts)),counts)
        if len(self.pool)!=plan["pool_size"] or len(counts)!=plan["samples_per_epoch"]:
            raise ValueError("Sampler plan is inconsistent.")
        self.last_indices=None
    def __len__(self):
        return self.plan["samples_per_epoch"]
    def __iter__(self):
        selected=self.pool[torch.randperm(len(self.pool),generator=self.generator)[:len(self)]]
        self.last_indices=selected.clone()
        return iter(selected.tolist())

def fitting_epoch_sampling_record(loader, fitting):
    plan=loader.fitting_sampling_plan
    if isinstance(loader.sampler,CappedDiseaseTicketSampler):
        draw=loader.sampler.last_indices
        if draw is None:raise ValueError("No completed sampler draw.")
        multiplicity=torch.bincount(draw,minlength=len(fitting))
        selected=fitting.iloc[draw.tolist()]
        maximum=int(multiplicity.max());unique=int((multiplicity>0).sum())
        draw_hash=hashlib.sha256(draw.numpy().tobytes()).hexdigest()
    else:
        if not isinstance(loader.sampler,torch.utils.data.RandomSampler) or loader.sampler.replacement:
            raise ValueError("Control must retain the original nonreplacement RandomSampler.")
        selected=fitting;maximum=1;unique=len(fitting);draw_hash=None
    counts={int(i):int(n) for i,n in selected.groupby("disease_id").size().items()}
    return {"mode":plan["mode"],"draw_count":len(selected),"distinct_fitting_images":unique,
            "maximum_image_repeats":maximum,"disease_draw_counts":counts,
            "draw_indices_sha256":draw_hash,"sampling_seed":plan["sampling_seed"]}


In [ ]:
def make_m5e_pilot_plan(base_config):
    """Exactly two fresh Base runs; the fitting sampler is the only intervention."""
    common={**copy.deepcopy(base_config),"architecture":"multitask_convnext_base",
        "initialization":"local_pretrained",
        "local_weight_path":"/home/user1/COS30082_pretrained_weights/convnext_base-6075fbad.pth",
        "expected_weight_sha256":"6075fbad31c688f9ed013922a59daf0fc2f4dc12e6e43bc01fa840c7bd1ca2cd",
        "warmup_epochs":2,"finetune_epochs":15,"epochs":17,
        "start_resolution":224,"final_resolution":320,"progressive_epochs":5,
        "min_finetune_epochs":15,"early_stopping_patience":20,
        "resume_from":None,"run_id":None,"experiment_kind":"m5e_pilot",
        "early_stopping_reference":"global_best_h1","sampler_weight_cap":3,
        "output_directory":"/home/user1/COS30082_experiments/wp2_m5e_pilot"}
    return [{**copy.deepcopy(common),"pilot_arm":"control","fitting_sampler":"uniform_shuffle"},
            {**copy.deepcopy(common),"pilot_arm":"balanced","fitting_sampler":"capped_disease_tickets"}]

def validate_m5e_pilot_plan(plan):
    if not isinstance(plan,(list,tuple)) or len(plan)!=2:
        raise ValueError("M5E requires exactly control then intervention.")
    expected=(("control","uniform_shuffle"),("balanced","capped_disease_tickets"))
    locked={"architecture":"multitask_convnext_base","initialization":"local_pretrained",
            "warmup_epochs":2,"finetune_epochs":15,"epochs":17,
            "start_resolution":224,"final_resolution":320,"progressive_epochs":5,
            "min_finetune_epochs":15,"early_stopping_patience":20,
            "early_stopping_reference":"global_best_h1","experiment_kind":"m5e_pilot",
            "optimizer":"AdamW","warmup_head_lr":.001,"backbone_lr":.00001,"head_lr":.0001,
            "learning_rate":.0001,"weight_decay":.01,"crop_weight":.5,"disease_weight":.5,
            "augment_fit":True,"effective_batch_size":64,"warmup_batch_size":64,
            "start_batch_size":32,"batch_size":16,"accumulation_steps":4,"seed":42,
            "precision":"bf16","channels_last":True,"gradient_checkpointing":False,
            "preprocessing_policy":"common_rgb_imagenet_progressive_v1"}
    for cfg,(arm,sampler) in zip(plan,expected):
        validate_config(cfg)
        if cfg.get("pilot_arm")!=arm or cfg.get("fitting_sampler")!=sampler:
            raise ValueError("Run the unchanged control before the capped sampler.")
        if cfg["resume_from"] is not None:
            raise ValueError("M5E pilot arms must start fresh, never historical recovery states.")
        for key,value in locked.items():
            if cfg.get(key)!=value:
                raise ValueError("M5E isolates sampling; unexpected setting: "+key)
        if cfg["local_weight_path"]!="/home/user1/COS30082_pretrained_weights/convnext_base-6075fbad.pth":
            raise ValueError("Use the verified fresh ImageNet artifact.")
        if cfg["expected_weight_sha256"]!="6075fbad31c688f9ed013922a59daf0fc2f4dc12e6e43bc01fa840c7bd1ca2cd":
            raise ValueError("Unexpected ImageNet artifact identity.")
        if not 1<=cfg["sampler_weight_cap"]<=4:
            raise ValueError("Use a bounded ticket cap.")
    ignored={"pilot_arm","fitting_sampler","run_id"}
    if ({k:v for k,v in plan[0].items() if k not in ignored}
            !={k:v for k,v in plan[1].items() if k not in ignored}):
        raise ValueError("Paired settings differ beyond the declared sampler intervention.")
    return copy.deepcopy(plan)

def pilot_epoch_diagnostics(seen, unseen, split, *, fitting_loader=None):
    """Visible component/pair diagnostics; canonical composition scoring is reused."""
    spec={"ids":split.candidate_ids,"pairs":split.candidate_pairs,"sha256":split.candidate_sha256}
    ids,pairs,_=validate_candidate_spec(spec)
    fitting_ids=set(map(int,split.fit.composition_id))
    healthy=[int(i) for i,name in split.mappings["disease_id_to_name"].items()
             if str(name).strip().casefold()=="healthy"]
    if len(healthy)!=1:
        raise ValueError("Resolve the dataset's exact healthy disease ID before pilot diagnostics.")
    healthy_id=healthy[0];folds={}
    for name,result,frame in (("seen",seen,split.val_seen),("unseen",unseen,split.val_unseen)):
        n=len(frame)
        truths=torch.tensor(frame.composition_id.to_numpy(),dtype=torch.long)
        if result["sample_count"]!=n or not torch.equal(result["composition_ids"].cpu(),truths):
            raise ValueError("Pilot validation order/count changed.")
        crop=result["crop_logits"].float().cpu();disease=result["disease_logits"].float().cpu()
        scores=composition_scores(crop,disease,spec)
        rank=torch.argsort(scores,dim=1,descending=True,stable=True)
        top_ids=torch.tensor(ids)[rank[:,:5]]
        crop_rank=torch.argsort(crop,dim=1,descending=True,stable=True)
        disease_rank=torch.argsort(disease,dim=1,descending=True,stable=True)
        ctrue=torch.tensor(frame.crop_id.to_numpy(),dtype=torch.long)
        dtrue=torch.tensor(frame.disease_id.to_numpy(),dtype=torch.long)
        comp1=top_ids[:,0]==truths;comp5=(top_ids==truths[:,None]).any(1)
        stats={"sample_count":n}
        for task,ranking,labels in (("crop",crop_rank,ctrue),("disease",disease_rank,dtrue)):
            for k in (1,5):
                hit=(ranking[:,:k]==labels[:,None]).any(1)
                stats[f"{task}_top{k}_hits"]=int(hit.sum());stats[f"{task}_top{k}"]=float(hit.double().mean())
        for k,hits in ((1,comp1),(5,comp5)):
            stats[f"composition_top{k}_hits"]=int(hits.sum())
            stats[f"composition_top{k}"]=composition_topk(scores,truths,spec,k)
        fitting_hits=sum(int(i) in fitting_ids for i in top_ids[:,0])
        stats.update(healthy_disease_id=healthy_id,
                     healthy_disease_prediction_count=int((disease_rank[:,0]==healthy_id).sum()),
                     healthy_disease_truth_count=int((dtrue==healthy_id).sum()),
                     fitting_composition_prediction_count=fitting_hits,
                     fitting_composition_prediction_fraction=fitting_hits/n,
                     healthy_disease_prediction_fraction=float((disease_rank[:,0]==healthy_id).double().mean()))
        if name=="unseen":
            stats["pairs"]=[]
            for global_id in sorted(set(map(int,truths))):
                mask=truths==global_id
                row=frame[frame.composition_id==global_id].iloc[0]
                stats["pairs"].append({"composition_id":global_id,"crop":str(row.crop_name),
                    "disease":str(row.disease_name),"sample_count":int(mask.sum()),
                    "top1_hits":int(comp1[mask].sum()),"top5_hits":int(comp5[mask].sum()),
                    "top1_accuracy":float(comp1[mask].double().mean()),"top5_accuracy":float(comp5[mask].double().mean()),
                    "disease_top1_hits":int((disease_rank[mask,0]==dtrue[mask]).sum())})
        folds[name]=stats
    output={"folds":folds,"metrics":evaluate_development(seen,unseen,spec)}
    if fitting_loader is not None:
        output["observed_fitting_sampling"]=fitting_epoch_sampling_record(fitting_loader,split.fit)
    return output

def validate_m5e_actual_budgets(summaries):
    if len(summaries)!=2:raise ValueError("Need exactly two completed pilot summaries.")
    validate_m5e_pilot_plan([s["configuration"] for s in summaries])
    common=None
    for summary in summaries:
        if summary["execution_status"]!="completed" or summary["completed_epochs"]!=17 or summary.get("selected") is None:
            raise ValueError("Do not compare partial or unequal pilot budgets.")
        history=summary["history"]
        if [r["epoch"] for r in history]!=list(range(1,18)):
            raise ValueError("Incomplete actual epoch history.")
        signature=[(r["train_samples"],r["optimizer_steps"]) for r in history]
        if any(samples!=4868 or steps!=77 for samples,steps in signature):
            raise ValueError("Actual sample/update budget changed.")
        if common is None:common=signature
        elif signature!=common:raise ValueError("Actual processed budgets differ.")
        if not summary.get("initialization_verified"):
            raise ValueError("Initialization was not verified.")
    if (not summaries[0].get("initial_model_sha256") or
            summaries[0]["initial_model_sha256"]!=summaries[1].get("initial_model_sha256")):
        raise ValueError("Fresh initial models/heads were not identical.")
    return True

def compare_m5e_pilot_results(summaries):
    validate_m5e_actual_budgets(summaries)
    rows=[]
    for summary in summaries:
        diag=summary["selected_pilot_diagnostics"]
        unseen=diag["folds"]["unseen"];seen=diag["folds"]["seen"]
        rows.append({"arm":summary["configuration"]["pilot_arm"],"selected_epoch":summary["selected"]["epoch"],
            **summary["selected"]["metrics"],
            "seen_crop_top1":seen["crop_top1"],"seen_disease_top1":seen["disease_top1"],
            "unseen_crop_top1":unseen["crop_top1"],"unseen_disease_top1":unseen["disease_top1"],
            "unseen_healthy_prediction_fraction":unseen["healthy_disease_prediction_fraction"],
            "unseen_fitting_composition_prediction_fraction":unseen["fitting_composition_prediction_fraction"],
            "completed_epochs":summary["completed_epochs"],"duration_seconds":summary["duration_seconds"],
            "peak_gpu_allocated_mib":summary["peak_gpu_allocated_mib"],"checkpoint":summary["selected"]["path"]})
    a,b=rows
    before={p["composition_id"]:p for p in summaries[0]["selected_pilot_diagnostics"]["folds"]["unseen"]["pairs"]}
    after={p["composition_id"]:p for p in summaries[1]["selected_pilot_diagnostics"]["folds"]["unseen"]["pairs"]}
    if set(before)!=set(after):raise ValueError("Unseen pair universe changed.")
    pair_rows=[]
    for i in sorted(before):
        x,y=before[i],after[i]
        if x["sample_count"]!=y["sample_count"]:raise ValueError("Per-pair validation budget changed.")
        pair_rows.append({"composition_id":i,"crop":x["crop"],"disease":x["disease"],
                          "sample_count":x["sample_count"],"control_top1_hits":x["top1_hits"],
                          "balanced_top1_hits":y["top1_hits"],"control_top5_hits":x["top5_hits"],
                          "control_top1_accuracy":x["top1_hits"]/x["sample_count"],
                          "balanced_top1_accuracy":y["top1_hits"]/y["sample_count"],
                          "balanced_top5_hits":y["top5_hits"]})
    supported=(b["top1_harmonic_mean"]>a["top1_harmonic_mean"]
               and b["unseen_disease_top1"]>a["unseen_disease_top1"]
               and b["unseen_top1"]>a["unseen_top1"])
    return {"comparison":rows,"per_pair":pair_rows,
        "assessment":"promising_development_signal_requires_review" if supported else
                     "no_joint_evidence_of_improvement",
        "success_claimed":False,
        "interpretation":"One seed and a short development pilot cannot establish robust success; inspect pair changes and repeat after review."}



In [ ]:
def m5e_bounded_synthetic_smoke(plan, *, device="cuda:0"):
    """Two fresh models, one synthetic head update each; never iterate a dataset epoch."""
    plan=validate_m5e_pilot_plan(plan)
    state,provenance=verified_backbone_state(plan[0])
    inputs=torch.randn(2,3,224,224,generator=torch.Generator().manual_seed(901))
    targets={"crop":torch.tensor([0,1]),"disease":torch.tensor([9,1])}
    records=[]
    for cfg in plan:
        seed_everything(cfg["seed"])
        model=create_multitask_model(cfg["architecture"],backbone_state_dict=state)
        initial=d_state_digest(model.state_dict())
        active=torch.device(device);model.to(active)
        if cfg["channels_last"]:model.to(memory_format=torch.channels_last)
        optimizer,_=configure_training_stage(model,cfg,"warmup")
        scaler=make_scaler(active,cfg["precision"])
        model.train();model.backbone.eval()
        if active.type=="cuda":torch.cuda.reset_peak_memory_stats(active)
        images=inputs.to(active).contiguous(memory_format=torch.channels_last)
        labels={k:v.to(active) for k,v in targets.items()}
        model.zero_grad(set_to_none=True)
        with amp_context(active,cfg["precision"]):
            output=model(images);losses=compute_multitask_loss(output,labels)
        scaler.scale(losses["total"]).backward()
        gradients={name:bool(parameter.grad is not None and torch.isfinite(parameter.grad).all()
                             and torch.count_nonzero(parameter.grad)>0)
                   for name,parameter in model.named_parameters() if name.startswith(("crop_head","disease_head"))}
        optimizer_update(model,optimizer,scaler,cfg["clip_grad_norm"])
        if active.type=="cuda":torch.cuda.synchronize(active)
        records.append({"arm":cfg["pilot_arm"],"initial_model_sha256":initial,
            "pretrained_provenance":provenance,"synthetic_images":2,"optimizer_updates":1,"real_image_epochs":0,
            "losses":{k:float(v.detach().cpu()) for k,v in losses.items()},
            "crop_shape":list(output["crop_logits"].shape),"disease_shape":list(output["disease_logits"].shape),
            "finite_nonzero_head_gradients":gradients,
            "backbone_has_no_gradients":all(p.grad is None for p in model.backbone.parameters()),
            "precision":cfg["precision"],"peak_allocated_mib":torch.cuda.max_memory_allocated(active)/2**20 if active.type=="cuda" else None})
        del model,optimizer,scaler,output,losses
        gc.collect()
        if active.type=="cuda":torch.cuda.empty_cache()
    if records[0]["initial_model_sha256"]!=records[1]["initial_model_sha256"]:
        raise RuntimeError("Fresh paired initialization did not match.")
    return {"records":records,"identical_fresh_initialization":True,"pilot_epochs_executed":0}


In [ ]:
def m5e_sampling_evidence(split, plan, *, seeds=tuple(range(42,74))):
    """Index-only sampling simulations: no image decoding or training."""
    validate_m5e_pilot_plan(plan)
    control=fitting_sampling_plan(split,plan[0])
    balanced=fitting_sampling_plan(split,plan[1])
    observed=[];totals=Counter();coverages=[]
    for seed in seeds:
        sampler=CappedDiseaseTicketSampler(balanced,generator=torch.Generator().manual_seed(seed))
        draw=torch.tensor(list(sampler),dtype=torch.long)
        multiplicity=torch.bincount(draw,minlength=len(split.fit))
        maximum=int(multiplicity.max())
        if maximum>balanced["cap"]:raise RuntimeError("Hard repeat cap violated.")
        counts={int(i):int(n) for i,n in split.fit.iloc[draw.tolist()].groupby("disease_id").size().items()}
        totals.update(counts);coverage=int((multiplicity>0).sum());coverages.append(coverage)
        observed.append({"seed":int(seed),"draws":len(draw),"distinct_images":coverage,
                         "maximum_image_repeats":maximum,"disease_counts":counts})
    comparison=[]
    for row in balanced["diseases"]:
        disease=row["disease_id"];expected=row["effective_disease_probability"]
        empirical=totals[disease]/(len(seeds)*len(split.fit))
        comparison.append({**row,"control_disease_probability":row["fitting_count"]/len(split.fit),
                           "empirical_disease_probability":empirical,
                           "absolute_probability_error":abs(empirical-expected)})
    return {"control":control,"balanced":balanced,"index_only_simulations":observed,
        "distribution_comparison":comparison,"mean_distinct_images":sum(coverages)/len(coverages),
        "maximum_probability_error":max(x["absolute_probability_error"] for x in comparison),
        "maximum_probability_ratio_to_control":max(x["ratio_to_control_probability"] for x in comparison),
        "real_image_decodes":0,"training_steps":0}


In [ ]:
RUN_M5E_PILOT = False
M5E_PILOT_PLAN = make_m5e_pilot_plan(CONFIG)
validate_m5e_pilot_plan(M5E_PILOT_PLAN)
print("M5E paired pilot is prepared, not executed. Historical launches are disabled.")
print(pd.DataFrame([{k:c[k] for k in ("pilot_arm","architecture","fitting_sampler","sampler_weight_cap","warmup_epochs","finetune_epochs","effective_batch_size")} for c in M5E_PILOT_PLAN]).to_string(index=False))
M5E_PILOT_SUMMARIES = []
if RUN_M5E_PILOT:
    pilot_split = build_development_split(ROOT / "DPD_Small_train.csv", seed=42)
    M5E_PILOT_SUMMARIES = run_m5e_pilot_sequence(M5E_PILOT_PLAN,pilot_split,IMAGE_DIRECTORY,ROOT)
else:
    print("SKIPPED: owner must explicitly enable RUN_M5E_PILOT in VS Code.")

RUN_M5E_SMOKE = False
if RUN_M5E_SMOKE:
    M5E_SMOKE_RESULTS = m5e_bounded_synthetic_smoke(M5E_PILOT_PLAN,device=CONFIG["device"])
    print(json.dumps(M5E_SMOKE_RESULTS,indent=2))
